# Current-video full diarization test

Attach a Kaggle dataset containing `uAtiEviUzGA.mp4`. This notebook runs the evidence-based baseline, finds repeated presentations, and evaluates uncertain overlap intervals with target-conditioned extraction plus conditional stereo-channel review. Supplemental stages never overwrite the baseline.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, time, zipfile

VIDEO_URL = 'https://www.youtube.com/watch?v=uAtiEviUzGA'
NOTEBOOK_REVISION = 'stable-promotion-calibration-v15'
RUN_FULL_VIDEO = True
RUN_TARGETED_REVIEW = True
RUN_OVERLAP_EXTRACTION = True
REVIEW_WEAK_CONFIDENCE = 0.35
REVIEW_SHORT_SECONDS = 1.0
BATCH_SIZE = 4

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    probe = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
    if probe is None or probe.returncode or 'GPU ' not in probe.stdout:
        raise RuntimeError('Enable a GPU accelerator in Kaggle Settings, then rerun this cell.')
    print(probe.stdout)
    BASE = Path('/kaggle/working')
else:
    BASE = Path.cwd()/'diarization-run'
BASE.mkdir(parents=True, exist_ok=True)
WORK = BASE/'diarization'; WORK.mkdir(exist_ok=True)
RESULTS = BASE/'results'; RESULTS.mkdir(exist_ok=True)
CACHE = BASE/'stage-cache'
VENV = BASE/'diarization-venv'
PYTHON = str(VENV/('Scripts/python.exe' if os.name == 'nt' else 'bin/python'))
VIDEO = WORK/'video-h264-v7.mp4'
REFERENCE = WORK/'target-reference'; REFERENCE.mkdir(exist_ok=True)
print('Video URL:', VIDEO_URL)
print('Notebook revision:', NOTEBOOK_REVISION)
print('Results folder:', RESULTS)
print('Setup will use:', PYTHON)


## Install and verify

Enable Internet and a GPU before running. The setup uses an isolated environment and verifies CUDA before the full video starts.


In [ ]:
import base64
EMBEDDED_FILES = {'chainofrules.py': '"""Evidence-based refactor of the recovered runnable chainofrules.py.\n'
                    'Run from the existing project directory with HF_TOKEN set in the '
                    'environment.\n'
                    'Accepts any video and matching target embedding files.\n'
                    'All rows in each embedding file are reference samples of the same target.\n'
                    'Defaults retain short.mp4, large-v2, ECAPA and buffalo_l.\n'
                    'Confidence values are heuristic evidence strengths, not calibrated '
                    'probabilities.\n'
                    '"""\n'
                    'import os\n'
                    'import argparse\n'
                    'import json\n'
                    'import re\n'
                    'import gc\n'
                    'from pathlib import Path\n'
                    'from importlib.metadata import version\n'
                    'import pandas as pd\n'
                    'import onnxruntime as ort\n'
                    'from cloud_runtime import StageCache, file_digest, create_face_analyzer, '
                    'create_full_audio_vad\n'
                    'from dataclasses import dataclass, field, asdict\n'
                    'import cv2\n'
                    'import numpy as np\n'
                    'import torch\n'
                    'import torchaudio\n'
                    'import whisperx\n'
                    'from whisperx.diarize import DiarizationPipeline\n'
                    'from speechbrain.inference.speaker import SpeakerRecognition\n'
                    'from insightface.app import FaceAnalysis\n'
                    'from collections import defaultdict\n'
                    'from repeat_evidence import (find_repeat_groups, build_repeat_proposals,\n'
                    '                             repeat_target_corroboration,\n'
                    '                             resolve_repeat_target_corroboration)\n'
                    '\n'
                    '\n'
                    '@dataclass(frozen=True)\n'
                    'class Baseline:\n'
                    '    raw_speaker_track: str\n'
                    '    speaker: str\n'
                    '\n'
                    '@dataclass(frozen=True)\n'
                    'class Evidence:\n'
                    '    source: str\n'
                    '    target_score: float\n'
                    '    confidence: float\n'
                    '    details: dict = field(default_factory=dict)\n'
                    '\n'
                    '@dataclass\n'
                    'class TimelineSegment:\n'
                    '    start: float\n'
                    '    end: float\n'
                    '    text: str\n'
                    '    baseline: Baseline\n'
                    '    words: list = field(default_factory=list)\n'
                    '    evidence: list = field(default_factory=list)\n'
                    '    final_speaker: str = "Uncertain"\n'
                    '    final_confidence: float = 0.0\n'
                    '    reasons: list = field(default_factory=list)\n'
                    '\n'
                    '\n'
                    'def normalize_vector(value):\n'
                    '    value = np.asarray(value, dtype=np.float32).reshape(-1)\n'
                    '    norm = np.linalg.norm(value)\n'
                    '    if not np.all(np.isfinite(value)) or norm <= 0:\n'
                    '        raise ValueError("Embedding must be finite and nonzero")\n'
                    '    return value / norm\n'
                    '\n'
                    '\n'
                    'def short_voice_crop(segment, previous=None, following=None, '
                    'media_duration=None):\n'
                    '    """Recover small timing gaps without including neighboring '
                    'utterances."""\n'
                    '    start, end = segment.start, segment.end\n'
                    '    if end - start < 0.4:\n'
                    '        lower = previous.end if previous is not None else 0.0\n'
                    '        upper = following.start if following is not None else media_duration\n'
                    '        start = max(lower, start - 0.15)\n'
                    '        end = min(end + 0.15, upper) if upper is not None else end\n'
                    '        # Overlapping transcript boundaries are not safe padding '
                    'opportunities.\n'
                    '        if start > segment.start or end < segment.end:\n'
                    '            return segment.start, segment.end\n'
                    '    return start, end\n'
                    '\n'
                    '\n'
                    'def add_question_response_evidence(segment, previous, tracks, target_track, '
                    'mapping_confidence):\n'
                    '    """A conversational hypothesis, never a police-specific identity '
                    'rule."""\n'
                    '    if previous is None or segment.end - segment.start > 1.0:\n'
                    '        return\n'
                    '    gap = segment.start - previous.end\n'
                    '    if not 0.0 <= gap <= 0.6 or mapping_confidence < 0.75:\n'
                    '        return\n'
                    '    if previous.final_speaker in ("Uncertain", "NonTarget_Unknown", '
                    '"Unknown_Speaker"):\n'
                    '        return\n'
                    '    if previous.final_confidence < 0.65:\n'
                    '        return\n'
                    '    question = previous.text.strip().lower()\n'
                    '    # Narrow to addressed yes/no questions; punctuation alone is '
                    'insufficient.\n'
                    '    direct_question = re.match(\n'
                    '        r"^(?:(?:ok|okay|all right)[.,]?\\s+)?"\n'
                    '        r"(?:do you|did you|have you|are you|were you|can you|could you|"\n'
                    '        r"would you|will you|don\'t you|didn\'t you|haven\'t you|aren\'t '
                    'you)\\b", question)\n'
                    '    if not question.endswith("?") or direct_question is None:\n'
                    '        return\n'
                    '    answer = re.sub(r"[^a-z\' ]", " ", segment.text.lower()).split()\n'
                    '    if not answer or len(answer) > 4 or answer[0] not in ("yes", "no", '
                    '"yeah", "yep", "nope", "nah"):\n'
                    '        return\n'
                    '    question_track = target_track if previous.final_speaker == '
                    '"Target_Speaker" else previous.final_speaker\n'
                    '    if question_track not in tracks:\n'
                    '        return\n'
                    '    candidates = sorted(track for track in tracks if track != '
                    'question_track)\n'
                    '    candidate = candidates[0] if len(candidates) == 1 else None\n'
                    '    segment.evidence.append(Evidence("question_response", 0.0, 0.20,\n'
                    '        {"question_start": previous.start, "question_track": question_track,\n'
                    '         "candidate_tracks": candidates, "candidate_track": candidate,\n'
                    '         "gap": gap, "assumption": "Immediate brief answer may be a different '
                    'speaker; not voice-verified."}))\n'
                    '\n'
                    '\n'
                    '\n'
                    '\n'
                    'def add_brief_exchange_evidence(segment, previous, tracks, target_track, '
                    'mapping_confidence):\n'
                    '    """Tentative acknowledgement or confirmation, with independent voice '
                    'agreement."""\n'
                    '    if previous is None or mapping_confidence < .75 or segment.end - '
                    'segment.start >= .4:\n'
                    '        return\n'
                    '    if not 0 <= segment.start - previous.end <= .6:\n'
                    '        return\n'
                    '    words = re.findall(r"[a-z\']+", segment.text.lower())\n'
                    '    preceding = re.findall(r"[a-z\']+", previous.text.lower())\n'
                    '    acknowledgement = words in (["okay"], ["ok"], ["oh", "okay"], ["oh", '
                    '"ok"])\n'
                    '    confirmation = (preceding in (["really"], ["seriously"]) and '
                    'previous.text.strip().endswith("?")\n'
                    '                    and words in (["yes"], ["yeah"], ["yep"], ["no"], '
                    '["nope"]))\n'
                    '    if not acknowledgement and not confirmation:\n'
                    '        return\n'
                    '    previous_track = target_track if previous.final_speaker == '
                    '"Target_Speaker" else previous.final_speaker\n'
                    '    candidates = sorted(set(tracks) - {previous_track})\n'
                    '    if previous_track not in tracks or len(candidates) != 1:\n'
                    '        return\n'
                    '    if acknowledgement and (previous.final_confidence < .65 or '
                    'previous.text.strip().endswith("?")):\n'
                    '        return\n'
                    '    if confirmation:\n'
                    '        # A weak question is usable only when its baseline agrees, a target '
                    'face is\n'
                    '        # tracked, and it was not itself attributed through conversational '
                    'inference.\n'
                    '        face = next((e for e in previous.evidence if e.source == '
                    '"target_face_visible"), None)\n'
                    '        if (previous.final_confidence < .25 or '
                    'previous.baseline.raw_speaker_track != previous_track\n'
                    '            or previous_track != target_track or face is None\n'
                    '            or not face.details.get("target_visible_hint", False)\n'
                    '            or any("inference" in reason for reason in previous.reasons)):\n'
                    '            return\n'
                    '    voice = next((e for e in segment.evidence if e.source == "local_voice"), '
                    'None)\n'
                    '    profiles = voice.details.get("track_similarities", {}) if voice is not '
                    'None else {}\n'
                    '    if (voice is None or voice.confidence > .30 or len(profiles) < 2\n'
                    '        or max(profiles.values()) >= .30 or voice.details.get("best_track") '
                    '!= candidates[0]):\n'
                    '        return\n'
                    '    segment.evidence.append(Evidence("question_response", 0, .20,\n'
                    '        {"candidate_track": candidates[0], "previous_track": previous_track,\n'
                    '         "assumption": "Brief acknowledgement or confirmation may change '
                    'speaker; weak voice agrees, not verified."}))\n'
                    '\n'
                    '\n'
                    'def add_echo_question_evidence(segment, previous, tracks, target_track, '
                    'mapping_confidence):\n'
                    '    """A brief quoted question can suggest another speaker, never establish '
                    'one."""\n'
                    '    if previous is None or not segment.text.strip().endswith("?"):\n'
                    '        return\n'
                    '    tokens = lambda text: re.findall(r"[a-z\']+", text.lower())\n'
                    '    phrase, statement = tokens(segment.text), tokens(previous.text)\n'
                    '    if not 2 <= len(phrase) <= 5 or statement[-len(phrase):] != phrase:\n'
                    '        return\n'
                    '    if not 0 <= segment.start - previous.end <= 0.8 or segment.end - '
                    'segment.start > 1.2:\n'
                    '        return\n'
                    '    if previous.final_confidence < 0.65 or mapping_confidence < 0.75:\n'
                    '        return\n'
                    '    previous_track = target_track if previous.final_speaker == '
                    '"Target_Speaker" else previous.final_speaker\n'
                    '    candidates = sorted(set(tracks) - {previous_track})\n'
                    '    if previous_track not in tracks or len(candidates) != 1:\n'
                    '        return\n'
                    '    segment.evidence.append(Evidence("echo_question", 0, 0.20,\n'
                    '        {"candidate_track": candidates[0], "previous_track": previous_track,\n'
                    '         "assumption": "Brief repeated question may come from the listener; '
                    'not voice-verified."}))\n'
                    '\n'
                    '\n'
                    'def bbox_iou(left, right):\n'
                    '    x1, y1 = max(left[0], right[0]), max(left[1], right[1])\n'
                    '    x2, y2 = min(left[2], right[2]), min(left[3], right[3])\n'
                    '    intersection = max(0.0, x2-x1) * max(0.0, y2-y1)\n'
                    '    left_area = max(0.0, left[2]-left[0]) * max(0.0, left[3]-left[1])\n'
                    '    right_area = max(0.0, right[2]-right[0]) * max(0.0, right[3]-right[1])\n'
                    '    return intersection / max(left_area + right_area - intersection, 1e-9)\n'
                    '\n'
                    '\n'
                    'def collect_visual_evidence(segment, cap, fps, face_analyzer, '
                    'target_face_centroid):\n'
                    '    """Track a recently recognized face through head turns; mouth motion is '
                    'only a hint."""\n'
                    '    if not np.isfinite(fps) or fps <= 0:\n'
                    '        segment.evidence.append(Evidence("visual_context", 0, 0, {"reason": '
                    '"invalid_fps"}))\n'
                    '        return\n'
                    '    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n'
                    '    # Lead-in frames establish identity; only frames inside speech measure '
                    'mouth motion.\n'
                    '    times = np.arange(max(0.0, segment.start - 0.5), segment.end, 0.125)\n'
                    '    indices = np.unique(np.rint(times * fps).astype(int))\n'
                    '    best, anchor_best, direct_matches, frames_read = None, None, 0, 0\n'
                    '    anchor = None\n'
                    '    observations, apertures, target_frames = [], [], []\n'
                    '    for index in indices:\n'
                    '        if frame_count > 0 and index >= frame_count:\n'
                    '            continue\n'
                    '        cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))\n'
                    '        ok, frame = cap.read()\n'
                    '        if not ok:\n'
                    '            continue\n'
                    '        frames_read += 1\n'
                    '        time = index / fps\n'
                    '        candidates = []\n'
                    '        for face in face_analyzer.get(frame):\n'
                    '            embedding = getattr(face, "embedding", None)\n'
                    '            if embedding is None:\n'
                    '                continue\n'
                    '            embedding = normalize_vector(embedding)\n'
                    '            similarity = float(np.dot(target_face_centroid, embedding))\n'
                    '            if segment.start <= time <= segment.end:\n'
                    '                best = similarity if best is None else max(best, similarity)\n'
                    '            candidates.append((similarity, face, embedding))\n'
                    '        recognized = [item for item in candidates if item[0] >= 0.40]\n'
                    '        selected, identity_source = None, None\n'
                    '        if recognized:\n'
                    '            selected = max(recognized, key=lambda item: item[0])\n'
                    '            direct_matches += 1\n'
                    '            anchor_best = selected[0] if anchor_best is None else '
                    'max(anchor_best, selected[0])\n'
                    '            identity_source = "reference_match"\n'
                    '        elif anchor is not None and time - anchor[2] <= 0.30:\n'
                    '            linked = [item for item in candidates\n'
                    '                      if bbox_iou(item[1].bbox, anchor[0]) >= 0.20\n'
                    '                      and float(np.dot(item[2], anchor[1])) >= 0.45]\n'
                    '            if linked:\n'
                    '                selected = max(linked, key=lambda item: float(np.dot(item[2], '
                    'anchor[1])))\n'
                    '                identity_source = "face_continuity"\n'
                    '        for similarity, face, embedding in candidates:\n'
                    '            observations.append({"frame": int(index), "similarity": '
                    'similarity,\n'
                    '                                 "bbox": face.bbox.tolist()})\n'
                    '        if selected is None:\n'
                    '            continue\n'
                    '        similarity, face, embedding = selected\n'
                    '        anchor = (face.bbox.copy(), embedding, time)\n'
                    '        if not segment.start <= time <= segment.end:\n'
                    '            continue\n'
                    '        target_frames.append({"frame": int(index), "identity_source": '
                    'identity_source,\n'
                    '                              "similarity": similarity})\n'
                    '        landmarks = getattr(face, "landmark_3d_68", None)\n'
                    '        if landmarks is not None and np.all(np.isfinite(landmarks)):\n'
                    '            # Standard 68-point inner mouth: aperture / width in 3D landmark '
                    'coordinates.\n'
                    '            width = float(np.linalg.norm(landmarks[60] - landmarks[64]))\n'
                    '            if width > 1e-6:\n'
                    '                apertures.append(float(np.linalg.norm(landmarks[62] - '
                    'landmarks[66]) / width))\n'
                    '    spread = float(np.percentile(apertures, 90) - np.percentile(apertures, '
                    '10)) if len(apertures) >= 5 else 0.0\n'
                    '    motion_hint = direct_matches >= 2 and len(apertures) >= 5 and spread >= '
                    '0.03\n'
                    '    segment.evidence.append(Evidence("target_face_visible", 0, 0,\n'
                    '        {"best_similarity": best, "identity_anchor_similarity": anchor_best,\n'
                    '         "target_visible_hint": bool(target_frames), "tracked_target_frames": '
                    'target_frames,\n'
                    '         "active_speaker_verified": False}))\n'
                    '    segment.evidence.append(Evidence("visual_context", 0, 0,\n'
                    '        {"frames_read": frames_read, "observations": observations,\n'
                    '         "note": "Face identity and visibility do not identify police or '
                    'prove speech."}))\n'
                    '    segment.evidence.append(Evidence("target_mouth_motion", 1.0 if '
                    'motion_hint else 0.0,\n'
                    '        0.20 if motion_hint else 0.0,\n'
                    '        {"direct_identity_matches": direct_matches, "mouth_samples": '
                    'len(apertures),\n'
                    '         "aperture_spread": spread, "active_speaker_verified": False,\n'
                    '         "note": "Weak landmark motion hint; no lipreading or audio-visual '
                    'synchronization model."}))\n'
                    '\n'
                    '\n'
                    'def resolve_segment(segment, target_track, mapping_confidence,\n'
                    '                    target_like_tracks=()):\n'
                    '    """Only the resolver assigns final identity; visibility alone cannot flip '
                    'it."""\n'
                    '    raw = segment.baseline.raw_speaker_track\n'
                    '    known = raw != "Unknown_Speaker"\n'
                    '    prior_weight = 0.55 * mapping_confidence if known else 0.0\n'
                    '    prior = 1.0 if raw == target_track else -1.0\n'
                    '    score, weight = prior * prior_weight, prior_weight\n'
                    '    reasons = [f"baseline={raw}; mapping strength={mapping_confidence:.3f}"]\n'
                    '    voice = None\n'
                    '    response = None\n'
                    '    mouth_motion = None\n'
                    '    echo = None\n'
                    '    visible = None\n'
                    '    overlap = None\n'
                    '    for item in segment.evidence:\n'
                    '        # Presence/context describes the scene, not the active speaker.\n'
                    '        if item.source in ("target_face_visible", "visual_context"):\n'
                    '            if item.source == "target_face_visible":\n'
                    '                visible = item\n'
                    '            continue\n'
                    '        if item.source == "echo_question":\n'
                    '            echo = item\n'
                    '            continue\n'
                    '        if item.source == "target_mouth_motion":\n'
                    '            mouth_motion = item\n'
                    '            continue\n'
                    '        if item.source == "overlapping_speakers":\n'
                    '            overlap = item\n'
                    '            continue\n'
                    '        if item.source == "question_response":\n'
                    '            response = item\n'
                    '            continue\n'
                    '        contribution = item.target_score * item.confidence\n'
                    '        score += contribution\n'
                    '        weight += item.confidence\n'
                    '        if item.source == "local_voice":\n'
                    '            voice = item\n'
                    '        if item.confidence:\n'
                    '            reasons.append(f"{item.source}: {contribution:+.3f}")\n'
                    '    normalized = score / max(weight, 1e-9)\n'
                    '    # Role phrases cannot establish or contradict identity without acoustic '
                    'support.\n'
                    '    acoustic_support = prior_weight > 0.08 or (voice is not None and '
                    'voice.confidence > 0.2)\n'
                    '    strong_conflict = (voice is not None and voice.confidence >= 0.5\n'
                    '                      and voice.target_score * prior < -0.4 and known)\n'
                    '    # A strong reference match plus an independent track match can correct '
                    'diarization.\n'
                    '    details = voice.details if voice is not None else {}\n'
                    '    matched_track = details.get("best_track")\n'
                    '    verified_correction = (strong_conflict and voice.confidence >= 0.5\n'
                    '                          and abs(voice.target_score) >= 0.4\n'
                    '                          and details.get("track_margin", 0.0) >= 0.10\n'
                    '                          and matched_track is not None\n'
                    '                          and details.get("track_similarities", '
                    '{}).get(matched_track, -1.0) >= 0.30\n'
                    '                          and ((voice.target_score > 0 and matched_track == '
                    'target_track)\n'
                    '                               or (voice.target_score < 0 and matched_track '
                    '!= target_track)))\n'
                    '    insufficient_short_audio = (segment.end - segment.start < 0.4\n'
                    '                                and (voice is None or voice.confidence == '
                    '0))\n'
                    '    response_track = response.details.get("candidate_track") if response is '
                    'not None else None\n'
                    '    profiles = details.get("track_similarities", {})\n'
                    '    weak_padded_voice = (segment.end - segment.start < 0.4 and voice is not '
                    'None\n'
                    '                         and voice.confidence <= 0.30 and len(profiles) >= 2\n'
                    '                         and max(profiles.values()) < 0.30)\n'
                    '    response_inference = ((insufficient_short_audio or weak_padded_voice) and '
                    'response_track is not None\n'
                    '                          and response.confidence > 0)\n'
                    '    visual_inference = (mouth_motion is not None and mouth_motion.confidence '
                    '> 0\n'
                    '                        and voice is not None and voice.confidence > 0\n'
                    '                        and mapping_confidence >= 0.75\n'
                    '                        and len(details.get("track_similarities", {})) >= 2\n'
                    '                        and details.get("track_margin", 1.0) < 0.05\n'
                    '                        and max(details["track_similarities"].values()) < '
                    '0.35)\n'
                    '    # Pyannote may split one person across tracks over a long recording. '
                    'Recover\n'
                    '    # only a segment with three agreeing signals: a strong direct reference\n'
                    "    # match, a recognized target face, and motion of that target's mouth.\n"
                    '    local_similarity = details.get("similarity")\n'
                    '    competitor_mean = details.get("competitor_mean")\n'
                    '    reference_margin = details.get("reference_margin")\n'
                    '    if reference_margin is None and local_similarity is not None \\\n'
                    '            and competitor_mean is not None:\n'
                    '        reference_margin = local_similarity - competitor_mean\n'
                    '    # Reference promotion can raise both the target and competitor means. '
                    'Use\n'
                    '    # their local margin for recovery gates so adding valid reference '
                    'samples\n'
                    '    # does not silently make an already-matching secondary-track segment '
                    'fail.\n'
                    '    stable_reference_support = (\n'
                    '        reference_margin >= 0.08 if reference_margin is not None\n'
                    '        else voice is not None and voice.target_score >= 0.15\n'
                    '    )\n'
                    '    audiovisual_target_recovery = (\n'
                    '        raw != target_track\n'
                    '        and mapping_confidence >= 0.75\n'
                    '        and voice is not None and voice.confidence >= 0.5\n'
                    '        and local_similarity is not None and local_similarity >= 0.35\n'
                    '        and visible is not None and '
                    'visible.details.get("target_visible_hint", False)\n'
                    '        and mouth_motion is not None and mouth_motion.confidence > 0\n'
                    '        and mouth_motion.target_score > 0\n'
                    '    )\n'
                    '    # Long recordings can split the supplied target across multiple '
                    'diarization\n'
                    '    # tracks. A globally target-like secondary track is only a candidate; '
                    'promote\n'
                    '    # an individual, non-overlapping segment when its direct reference match '
                    'is\n'
                    '    # independently strong. This is deliberately stricter than ordinary '
                    'target\n'
                    '    # assignment and leaves marginal segments on their original track for '
                    'review.\n'
                    '    direct_secondary_target_recovery = (\n'
                    '        raw != target_track\n'
                    '        and raw in set(target_like_tracks)\n'
                    '        and mapping_confidence >= 0.75\n'
                    '        and segment.end - segment.start >= 0.6\n'
                    '        and voice is not None and voice.confidence >= 0.5\n'
                    '        and local_similarity is not None and local_similarity >= 0.35\n'
                    '        and stable_reference_support\n'
                    '    )\n'
                    '    # A brief interruption should not erase a well-supported dominant '
                    'speaker\n'
                    '    # from the whole ASR segment. Keep the target attribution only when '
                    'direct\n'
                    '    # target-voice evidence is strong and simultaneous activity occupies a\n'
                    '    # minority of the segment. The overlap intervals remain in the evidence '
                    'so\n'
                    '    # downstream output can show that the secondary speech is unresolved.\n'
                    '    overlap_fraction = (overlap.details.get("overlap_fraction", 1.0)\n'
                    '                        if overlap is not None else 0.0)\n'
                    '    localized_target_overlap = (\n'
                    '        overlap is not None\n'
                    '        and overlap.details.get("target_and_non_target", False)\n'
                    '        and overlap_fraction <= 0.35\n'
                    '        and mapping_confidence >= 0.75\n'
                    '        and segment.end - segment.start >= 0.6\n'
                    '        and voice is not None and voice.confidence >= 0.5\n'
                    '        and local_similarity is not None and local_similarity >= 0.35\n'
                    '        and stable_reference_support\n'
                    '        and (raw == target_track\n'
                    '             or audiovisual_target_recovery\n'
                    '             or direct_secondary_target_recovery)\n'
                    '    )\n'
                    '    echo_inference = (echo is not None and echo.details["candidate_track"] == '
                    'target_track\n'
                    '                      and visible is not None and '
                    'visible.details.get("target_visible_hint", False)\n'
                    '                      and len(profiles) >= 2 and max(profiles.values()) < '
                    '0.30\n'
                    '                      and matched_track == target_track and voice.confidence '
                    '< 0.5)\n'
                    '    if overlap is not None and overlap.details.get("target_and_non_target", '
                    'False) \\\n'
                    '            and not localized_target_overlap:\n'
                    '        final = "Overlapping_Speakers"\n'
                    '        reasons.append("target and non-target diarization tracks overlap; '
                    'text speaker is unresolved")\n'
                    '    elif localized_target_overlap:\n'
                    '        final = "Target_Speaker"\n'
                    '        reasons.append(\n'
                    '            "strong target voice dominates the segment; concurrent speech is '
                    '"\n'
                    '            "localized in overlapping_speakers evidence and remains '
                    'unresolved"\n'
                    '        )\n'
                    '    elif audiovisual_target_recovery:\n'
                    '        final = "Target_Speaker"\n'
                    '        reasons.append("strong local target voice agrees with recognized '
                    'target mouth motion")\n'
                    '    elif direct_secondary_target_recovery:\n'
                    '        final = "Target_Speaker"\n'
                    '        reasons.append("strong direct target voice recovers a segment from a '
                    'globally target-like secondary track")\n'
                    '    elif echo_inference:\n'
                    '        final = "Target_Speaker"\n'
                    '        reasons.append("weak repeated-question inference with face continuity '
                    'and weak supporting voice profile; not voice-verified")\n'
                    '    elif visual_inference:\n'
                    '        final = "Target_Speaker"\n'
                    '        reasons.append("weak visible-target mouth-motion inference; '
                    'independent voice profiles are ambiguous")\n'
                    '    elif response_inference:\n'
                    '        final = "Target_Speaker" if response_track == target_track else '
                    'response_track\n'
                    '        reasons.append(f"weak question/answer inference to {response_track}; '
                    'not voice-verified")\n'
                    '    elif verified_correction:\n'
                    '        final = "Target_Speaker" if matched_track == target_track else '
                    'matched_track\n'
                    '        reasons.append(f"independent voice profile supports correction to '
                    '{matched_track}")\n'
                    '    elif insufficient_short_audio or not acoustic_support or abs(normalized) '
                    '<= 0.18 or strong_conflict:\n'
                    '        final = "Uncertain"\n'
                    '        reasons.append("weak, balanced, or conflicting acoustic evidence")\n'
                    '    elif normalized > 0:\n'
                    '        final = "Target_Speaker"\n'
                    '    else:\n'
                    '        final = raw if known and raw != target_track else '
                    '"NonTarget_Unknown"\n'
                    '    segment.final_speaker = final\n'
                    '    # Avoid baseline-only certainty and account for weak global separation.\n'
                    '    segment.final_confidence = float(min(abs(normalized), weight / 1.55))\n'
                    '    if overlap is not None and overlap.details.get("target_and_non_target", '
                    'False) \\\n'
                    '            and not localized_target_overlap:\n'
                    '        segment.final_confidence = 0.0\n'
                    '    elif localized_target_overlap:\n'
                    '        segment.final_confidence = float(\n'
                    '            min(0.65, local_similarity, voice.confidence) *\n'
                    '            (1.0 - 0.5 * overlap_fraction)\n'
                    '        )\n'
                    '    elif audiovisual_target_recovery:\n'
                    '        segment.final_confidence = float(min(0.65, local_similarity, '
                    'voice.confidence))\n'
                    '    elif direct_secondary_target_recovery:\n'
                    '        segment.final_confidence = float(min(local_similarity, '
                    'voice.confidence))\n'
                    '    elif verified_correction:\n'
                    '        segment.final_confidence = float(min(voice.confidence, '
                    'abs(voice.target_score),\n'
                    '                                             details["track_margin"] / '
                    '0.20))\n'
                    '    if echo_inference:\n'
                    '        segment.final_confidence = 0.20\n'
                    '    elif visual_inference:\n'
                    '        segment.final_confidence = 0.25\n'
                    '    elif response_inference:\n'
                    '        segment.final_confidence = min(0.25, response.confidence)\n'
                    '    elif insufficient_short_audio:\n'
                    '        segment.final_confidence = 0.0\n'
                    '        reasons.append("short utterance has no usable local voice evidence")\n'
                    '    if segment.end - segment.start < 0.4:\n'
                    '        segment.final_confidence = min(segment.final_confidence, 0.30)\n'
                    '    segment.reasons = reasons\n'
                    '\n'
                    '\n'
                    'def voice_mapping_confidence(ranked, means, sample_counts):\n'
                    '    """Return mapping strength without inventing a competing speaker.\n'
                    '\n'
                    '    A well-sampled lone track can be mapped by absolute reference affinity.\n'
                    '    The 0.18 floor leaves weak or borderline single-track clips uncertain;\n'
                    '    0.33 reaches full strength. Multi-track clips continue to use '
                    'separation.\n'
                    '    """\n'
                    '    if not ranked:\n'
                    '        return 0.0\n'
                    '    if len(ranked) == 1:\n'
                    '        track = ranked[0]\n'
                    '        if sample_counts.get(track, 0) < 3:\n'
                    '            return 0.0\n'
                    '        return min(1.0, max(0.0, (means[track] - 0.18) / 0.15))\n'
                    '    separation = means[ranked[0]] - means[ranked[1]]\n'
                    '    return min(1.0, max(0.0, separation / 0.15))\n'
                    '\n'
                    '\n'
                    'def add_overlap_evidence(segment, diarization_rows, target_track, '
                    'minimum_seconds=0.15):\n'
                    '    """Mark simultaneous target/non-target activity without assigning the '
                    'words."""\n'
                    '    rows = []\n'
                    '    for row in diarization_rows:\n'
                    '        start = max(segment.start, float(row["start"]))\n'
                    '        end = min(segment.end, float(row["end"]))\n'
                    '        if end > start:\n'
                    '            rows.append((start, end, str(row["speaker"])))\n'
                    '    intervals, pairs = [], set()\n'
                    '    for index, first in enumerate(rows):\n'
                    '        for second in rows[index + 1:]:\n'
                    '            if first[2] == second[2]:\n'
                    '                continue\n'
                    '            start, end = max(first[0], second[0]), min(first[1], second[1])\n'
                    '            if end > start:\n'
                    '                intervals.append((start, end))\n'
                    '                pairs.add(tuple(sorted((first[2], second[2]))))\n'
                    '    if not intervals:\n'
                    '        return\n'
                    '    merged = []\n'
                    '    for start, end in sorted(intervals):\n'
                    '        if merged and start <= merged[-1][1]:\n'
                    '            merged[-1] = (merged[-1][0], max(merged[-1][1], end))\n'
                    '        else:\n'
                    '            merged.append((start, end))\n'
                    '    duration = sum(end - start for start, end in merged)\n'
                    '    if duration < minimum_seconds:\n'
                    '        return\n'
                    '    tracks = sorted({track for _, _, track in rows})\n'
                    '    segment.evidence.append(Evidence("overlapping_speakers", 0.0, 0.0, {\n'
                    '        "overlap_seconds": duration,\n'
                    '        "overlap_fraction": duration / max(segment.end - segment.start, '
                    '1e-9),\n'
                    '        "intervals": [{"start": start, "end": end} for start, end in '
                    'merged],\n'
                    '        "tracks": tracks,\n'
                    '        "track_pairs": [list(pair) for pair in sorted(pairs)],\n'
                    '        "target_and_non_target": target_track in tracks and any(t != '
                    'target_track for t in tracks),\n'
                    '        "note": "Simultaneous diarization tracks do not identify which '
                    'speaker produced the ASR text.",\n'
                    '    }))\n'
                    '\n'
                    '\n'
                    'def main():\n'
                    '    parser = argparse.ArgumentParser(description="Resolve a supplied target '
                    'voice in any video.")\n'
                    '    parser.add_argument("video", nargs="?", default="short.mp4")\n'
                    '    parser.add_argument("--voice-priors", default="voice_embeddings.npy")\n'
                    '    parser.add_argument("--face-priors", default="face_embeddings.npy")\n'
                    '    parser.add_argument("--output", default="diarization_evidence.json")\n'
                    '    parser.add_argument("--batch-size", type=int, default=16)\n'
                    '    parser.add_argument("--cache-dir", help="Reuse completed stages for '
                    'matching inputs/code/runtime")\n'
                    '    parser.add_argument("--transcription-coverage", choices=("vad", "full"), '
                    'default="vad",\n'
                    '                        help="Experimental full coverage includes '
                    'noise/silence; review for hallucinations")\n'
                    '    args = parser.parse_args()\n'
                    '    if args.batch_size < 1:\n'
                    '        parser.error("--batch-size must be positive")\n'
                    '    HF_TOKEN = os.environ.get("HF_TOKEN") or '
                    'os.environ.get("HUGGINGFACE_TOKEN")\n'
                    '    if not HF_TOKEN:\n'
                    '        raise RuntimeError("Set HF_TOKEN (or HUGGINGFACE_TOKEN) before '
                    'running.")\n'
                    '    device = "cuda" if torch.cuda.is_available() else "cpu"\n'
                    '    compute_type = "float16" if torch.cuda.is_available() else "int8"\n'
                    '\n'
                    '    # =====================================================================\n'
                    '    # 1. CORE PIPELINE INITIALIZATION\n'
                    '    # =====================================================================\n'
                    '    print("⏳ Initializing Core Tracking Engines...")\n'
                    '    def release_gpu():\n'
                    '        gc.collect()\n'
                    '        if device == "cuda":\n'
                    '            torch.cuda.empty_cache()\n'
                    '\n'
                    '    fingerprint = {"inputs": {name: file_digest(path) for name, path in\n'
                    '        (("video", args.video), ("voice", args.voice_priors), ("face", '
                    'args.face_priors))},\n'
                    '        "code": file_digest(__file__), "runtime_helper": '
                    'file_digest(Path(__file__).with_name("cloud_runtime.py")),\n'
                    '        "device": device, "batch_size": args.batch_size,\n'
                    '        "versions": {name: version(name) for name in\n'
                    '            ("torch", "torchaudio", "whisperx", "speechbrain", "insightface", '
                    '"numpy")},\n'
                    '        "onnxruntime": ort.__version__, "providers": '
                    'ort.get_available_providers()}\n'
                    '    cache = StageCache(args.cache_dir, fingerprint)\n'
                    '    print(f"WhisperX/SpeechBrain device: {device}")\n'
                    '\n'
                    '    # Load Priors Matrix\n'
                    '    voice_priors = np.load(args.voice_priors, allow_pickle=False)\n'
                    '    face_priors = np.load(args.face_priors, allow_pickle=False)\n'
                    '    if voice_priors.ndim == 1: voice_priors = np.expand_dims(voice_priors, '
                    'axis=0)\n'
                    '    if face_priors.ndim == 1: face_priors = np.expand_dims(face_priors, '
                    'axis=0)\n'
                    '\n'
                    '    def reference_centroid(samples, label):\n'
                    '        if samples.ndim != 2:\n'
                    '            raise ValueError(f"{label}: expected a vector or matrix of target '
                    'samples")\n'
                    '        norms = np.linalg.norm(samples, axis=1)\n'
                    '        valid = np.all(np.isfinite(samples), axis=1) & (norms > 0)\n'
                    '        if not np.any(valid):\n'
                    '            raise ValueError(f"{label}: no valid target samples")\n'
                    '        normalized = samples[valid] / norms[valid, None]\n'
                    '        print(f"{label}: using {len(normalized)} of {len(samples)} target '
                    'reference samples")\n'
                    '        return normalize_vector(np.mean(normalized, axis=0))\n'
                    '\n'
                    '    target_voice_vector = reference_centroid(voice_priors, "Voice '
                    'references")\n'
                    '    target_face_centroid = reference_centroid(face_priors, "Face '
                    'references")\n'
                    '\n'
                    '    # =====================================================================\n'
                    '    # 2. AUDIO & VIDEO DATA PREP\n'
                    '    # =====================================================================\n'
                    '    print("⏳ Preparing media data tracks...")\n'
                    '    video_path = args.video\n'
                    '    audio_loaded = whisperx.load_audio(video_path)\n'
                    '    cap = cv2.VideoCapture(video_path)\n'
                    '    fps = cap.get(cv2.CAP_PROP_FPS)\n'
                    '\n'
                    '    waveform, sample_rate = torchaudio.load(video_path)\n'
                    '    if sample_rate != 16000:\n'
                    '        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, '
                    'new_freq=16000)\n'
                    '        waveform = resampler(waveform)\n'
                    '    waveform = torch.mean(waveform, dim=0, keepdim=True)\n'
                    '    total_samples = waveform.shape[1]\n'
                    '\n'
                    '    # =====================================================================\n'
                    '    # 3. GENERAL TRANSCRIPTION & LAYERING\n'
                    '    # =====================================================================\n'
                    '    print("⏳ Processing WhisperX Text Script...")\n'
                    '    def transcribe():\n'
                    '        options = {"vad_model": create_full_audio_vad()} if '
                    'args.transcription_coverage == "full" else {}\n'
                    '        model = whisperx.load_model("large-v2", device, '
                    'compute_type=compute_type, **options)\n'
                    '        try:\n'
                    '            return model.transcribe(audio_loaded, '
                    'batch_size=args.batch_size)\n'
                    '        finally:\n'
                    '            del model\n'
                    '            release_gpu()\n'
                    '\n'
                    '    asr_result = cache.get("transcription_" + args.transcription_coverage, '
                    'transcribe)\n'
                    '\n'
                    '    def align():\n'
                    '        model, metadata = '
                    'whisperx.load_align_model(language_code=asr_result["language"], '
                    'device=device)\n'
                    '        try:\n'
                    '            return whisperx.align(asr_result["segments"], model, metadata, '
                    'audio_loaded,\n'
                    '                                  device, return_char_alignments=False)\n'
                    '        finally:\n'
                    '            del model\n'
                    '            release_gpu()\n'
                    '\n'
                    '    aligned_result = cache.get("alignment_" + args.transcription_coverage, '
                    'align)\n'
                    '    print("⏳ Generating Unsupervised Voice Tracks...")\n'
                    '\n'
                    '    def diarize():\n'
                    '        model = DiarizationPipeline(token=HF_TOKEN, device=device)\n'
                    '        try:\n'
                    '            # Preserve original whole-video track IDs; no independent chunk '
                    'clustering.\n'
                    '            return model(audio_loaded)[["start", "end", '
                    '"speaker"]].to_dict(orient="records")\n'
                    '        finally:\n'
                    '            del model\n'
                    '            release_gpu()\n'
                    '\n'
                    '    diarize_segments = pd.DataFrame(cache.get("diarization", diarize))\n'
                    '    embedding_model = SpeakerRecognition.from_hparams(\n'
                    '        source="speechbrain/spkrec-ecapa-voxceleb", '
                    'savedir="pretrained_models/spkrec-ecapa-voxceleb",\n'
                    '        run_opts={"device": device})\n'
                    '    face_analyzer, face_providers = create_face_analyzer(device, ort, '
                    'FaceAnalysis)\n'
                    '\n'
                    '    target_voice_vector = normalize_vector(target_voice_vector)\n'
                    '    target_face_centroid = normalize_vector(target_face_centroid)\n'
                    '    embedding_cache = {}\n'
                    '\n'
                    '    def audio_embedding(start, end):\n'
                    '        key = (float(start), float(end))\n'
                    '        if key in embedding_cache:\n'
                    '            return embedding_cache[key]\n'
                    '        left = max(0, int(start * 16000))\n'
                    '        right = min(total_samples, int(end * 16000))\n'
                    '        checkpoint_name = f"voice_{left}_{right}"\n'
                    '        saved = cache.read(checkpoint_name)\n'
                    '        if saved is not None:\n'
                    '            result = np.asarray(saved, dtype=np.float32)\n'
                    '            embedding_cache[key] = result\n'
                    '            return result\n'
                    '        result = None\n'
                    '        if right - left >= 6400:\n'
                    '            try:\n'
                    '                with torch.no_grad():\n'
                    '                    result = normalize_vector(embedding_model.encode_batch(\n'
                    '                        waveform[:, '
                    'left:right].to(device)).flatten().cpu().numpy())\n'
                    '                if result.shape != target_voice_vector.shape:\n'
                    '                    raise ValueError("Voice prior dimensions do not match '
                    'ECAPA output")\n'
                    '            except ValueError:\n'
                    '                raise\n'
                    '            except Exception as exc:\n'
                    '                print(f"Voice embedding unavailable at {start:.2f}-{end:.2f}: '
                    '{exc}")\n'
                    '        if result is not None:\n'
                    '            cache.write(checkpoint_name, result.tolist())\n'
                    '        embedding_cache[key] = result\n'
                    '        return result\n'
                    '\n'
                    '    cluster_scores = defaultdict(list)\n'
                    '    cluster_embeddings = defaultdict(list)\n'
                    '    for _, row in diarize_segments.iterrows():\n'
                    '        start, end = float(row["start"]), float(row["end"])\n'
                    '        if end - start < 0.6:\n'
                    '            continue\n'
                    '        emb = audio_embedding(start, end)\n'
                    '        if emb is not None:\n'
                    '            track = str(row["speaker"])\n'
                    '            cluster_scores[track].append(float(np.dot(target_voice_vector, '
                    'emb)))\n'
                    '            cluster_embeddings[track].append((start, end, emb))\n'
                    '    means = {track: float(np.mean(scores)) for track, scores in '
                    'cluster_scores.items()}\n'
                    '    ranked = sorted(means, key=means.get, reverse=True)\n'
                    '    target_track = ranked[0] if ranked else None\n'
                    '    target_mean = means[target_track] if ranked else 0.0\n'
                    '    # No invented competitor when only one cluster has usable speech.\n'
                    '    other_mean = means[ranked[1]] if len(ranked) > 1 else None\n'
                    '    separation = target_mean - other_mean if other_mean is not None else 0.0\n'
                    '    mapping_confidence = voice_mapping_confidence(\n'
                    '        ranked, means, {track: len(scores) for track, scores in '
                    'cluster_scores.items()})\n'
                    '    target_like_tracks = {\n'
                    '        track for track, mean in means.items()\n'
                    '        if track != target_track and mean >= 0.18 and target_mean - mean <= '
                    '0.20\n'
                    '    }\n'
                    '    print("\\n--- Baseline voice affinity ---")\n'
                    '    for track in ranked:\n'
                    '        print(f"{track}: {means[track]:.3f} ({len(cluster_scores[track])} '
                    'samples)")\n'
                    '    print(f"Target candidate: {target_track}; mapping '
                    'strength={mapping_confidence:.3f}")\n'
                    '    if target_like_tracks:\n'
                    '        print(f"Target-like secondary tracks (segment review only): '
                    '{sorted(target_like_tracks)}")\n'
                    '\n'
                    '    assigned = whisperx.assign_word_speakers(diarize_segments, '
                    'aligned_result)\n'
                    '    timeline = []\n'
                    '    for source in assigned["segments"]:\n'
                    '        raw = str(source.get("speaker") or "Unknown_Speaker")\n'
                    '        base = "Target_Speaker" if raw == target_track else ("Unknown" if raw '
                    '== "Unknown_Speaker" else raw)\n'
                    '        timeline.append(TimelineSegment(float(source["start"]), '
                    'float(source["end"]),\n'
                    '                        source["text"].strip(), Baseline(raw, base), '
                    'source.get("words", [])))\n'
                    '\n'
                    '    def collect_voice(segment, previous=None, following=None):\n'
                    '        crop_start, crop_end = short_voice_crop(segment, previous, following, '
                    'total_samples / 16000)\n'
                    '        emb = audio_embedding(crop_start, crop_end)\n'
                    '        similarity = float(np.dot(target_voice_vector, emb)) if emb is not '
                    'None else None\n'
                    '        strength = min(1.0, max(0.0, (segment.end - segment.start) / 1.2)) * '
                    'mapping_confidence\n'
                    '        if segment.end - segment.start < 0.4:\n'
                    '            strength = min(strength, 0.30)\n'
                    '        target_score = 0.0\n'
                    '        if similarity is not None and separation >= 0.03:\n'
                    '            target_score = float(np.clip((similarity - (target_mean + '
                    'other_mean) / 2) / separation, -1, 1))\n'
                    '        else:\n'
                    '            strength = 0.0\n'
                    '        # Exclude intersecting speech from profiles so a segment cannot '
                    'validate itself.\n'
                    '        track_similarities = {}\n'
                    '        profile_counts = {}\n'
                    '        if emb is not None:\n'
                    '            for track, samples in cluster_embeddings.items():\n'
                    '                independent = [vector for start, end, vector in samples\n'
                    '                               if end <= crop_start or start >= crop_end]\n'
                    '                if len(independent) < 2:\n'
                    '                    continue\n'
                    '                centroid = normalize_vector(np.mean(independent, axis=0))\n'
                    '                track_similarities[track] = float(np.dot(emb, centroid))\n'
                    '                profile_counts[track] = len(independent)\n'
                    '        candidates = sorted(track_similarities, key=track_similarities.get, '
                    'reverse=True)\n'
                    '        best_track = candidates[0] if len(candidates) >= 2 else None\n'
                    '        margin = (track_similarities[candidates[0]] - '
                    'track_similarities[candidates[1]]\n'
                    '                  if len(candidates) >= 2 else 0.0)\n'
                    '        segment.evidence.append(Evidence("local_voice", target_score, '
                    'strength,\n'
                    '            {"similarity": similarity, "crop_start": crop_start, "crop_end": '
                    'crop_end,\n'
                    '             "target_mean": target_mean, "competitor_mean": other_mean,\n'
                    '             "reference_margin": (similarity - other_mean\n'
                    '                                  if similarity is not None and other_mean is '
                    'not None\n'
                    '                                  else None),\n'
                    '             "track_similarities": track_similarities, '
                    '"independent_profile_counts": profile_counts,\n'
                    '             "best_track": best_track, "track_margin": margin}))\n'
                    '\n'
                    '    def collect_visual(segment):\n'
                    '        name = f"visual_{segment.start:.6f}_{segment.end:.6f}"\n'
                    '        saved = cache.read(name)\n'
                    '        if saved is not None:\n'
                    '            segment.evidence.extend(Evidence(**item) for item in saved)\n'
                    '            return\n'
                    '        offset = len(segment.evidence)\n'
                    '        collect_visual_evidence(segment, cap, fps, face_analyzer, '
                    'target_face_centroid)\n'
                    '        cache.write(name, [asdict(item) for item in '
                    'segment.evidence[offset:]])\n'
                    '\n'
                    '    def collect_semantic(segment):\n'
                    '        # Without an explicit role-to-identity mapping, words cannot identify '
                    'a person.\n'
                    '        # Keep semantic/context observations neutral for arbitrary videos and '
                    'targets.\n'
                    '        segment.evidence.append(Evidence("semantic_context", 0.0, 0.0,\n'
                    '            {"identity_mapping": None,\n'
                    '             "note": "No role or phrase is assumed to identify the supplied '
                    'target."}))\n'
                    '\n'
                    '    try:\n'
                    '        all_tracks = {str(track) for track in '
                    'diarize_segments["speaker"].dropna().unique()}\n'
                    '        diarization_rows = diarize_segments.to_dict("records")\n'
                    '        for index, segment in enumerate(timeline):\n'
                    '            previous = timeline[index - 1] if index else None\n'
                    '            following = timeline[index + 1] if index + 1 < len(timeline) else '
                    'None\n'
                    '            collect_voice(segment, previous, following)\n'
                    '            collect_visual(segment)\n'
                    '            collect_semantic(segment)\n'
                    '            add_overlap_evidence(segment, diarization_rows, target_track)\n'
                    '            add_question_response_evidence(segment, previous, all_tracks, '
                    'target_track, mapping_confidence)\n'
                    '            add_brief_exchange_evidence(segment, previous, all_tracks, '
                    'target_track, mapping_confidence)\n'
                    '            add_echo_question_evidence(segment, previous, all_tracks, '
                    'target_track, mapping_confidence)\n'
                    '            resolve_segment(segment, target_track, mapping_confidence,\n'
                    '                            target_like_tracks)\n'
                    '            print(f"Resolved segment {index + 1}/{len(timeline)} at '
                    '{segment.end:.1f}s", flush=True)\n'
                    '        repeat_groups = find_repeat_groups(timeline)\n'
                    '        repeat_proposals = build_repeat_proposals(timeline, repeat_groups)\n'
                    '        for index, proposals in repeat_proposals.items():\n'
                    '            for details in proposals:\n'
                    '                timeline[index].evidence.append(Evidence(\n'
                    '                    "repeated_presentation", 0.0, 0.0, details\n'
                    '                ))\n'
                    '            corroboration = repeat_target_corroboration(timeline[index], '
                    'proposals)\n'
                    '            if corroboration is not None:\n'
                    '                timeline[index].evidence.append(Evidence(\n'
                    '                    "repeat_target_corroboration", 1.0,\n'
                    '                    min(0.55, corroboration["alignment_confidence"]),\n'
                    '                    corroboration,\n'
                    '                ))\n'
                    '                resolve_repeat_target_corroboration(\n'
                    '                    timeline[index], corroboration\n'
                    '                )\n'
                    '        print("\\n--- Evidence-Based Speaker Resolution ---")\n'
                    '        for segment in timeline:\n'
                    '            print(f"[{segment.start:.2f}s - {segment.end:.2f}s] '
                    '{segment.final_speaker} "\n'
                    '                  f"(strength={segment.final_confidence:.2f}): '
                    '{segment.text}")\n'
                    '            print(f"    baseline={segment.baseline.speaker}; '
                    'raw={segment.baseline.raw_speaker_track}")\n'
                    '            for item in segment.evidence:\n'
                    '                print(f"    {item.source}: score={item.target_score:+.2f}, '
                    'strength={item.confidence:.2f}, {item.details}")\n'
                    '        with open(args.output, "w", encoding="utf-8") as output:\n'
                    '            json.dump({"target_candidate": target_track, '
                    '"cluster_voice_means": means,\n'
                    '                       "mapping_strength": mapping_confidence,\n'
                    '                       "runtime": {"device": device, "face_providers": '
                    'face_providers,\n'
                    '                                   "transcription_coverage": '
                    'args.transcription_coverage},\n'
                    '                       "confidence_is_calibrated": False,\n'
                    '                       "repeated_presentations": repeat_groups,\n'
                    '                       "segments": [asdict(segment) for segment in '
                    'timeline]}, output, indent=2, ensure_ascii=False)\n'
                    '    finally:\n'
                    '        cap.release()\n'
                    '\n'
                    '\n'
                    'if __name__ == "__main__":\n'
                    '    main()\n',
 'cloud_runtime.py': '"""Small runtime helpers; attribution rules do not live here."""\n'
                     'import hashlib\n'
                     'import json\n'
                     'import os\n'
                     'from pathlib import Path\n'
                     '\n'
                     '\n'
                     'def file_digest(path):\n'
                     '    digest = hashlib.sha256()\n'
                     '    with open(path, "rb") as stream:\n'
                     '        for block in iter(lambda: stream.read(1024 * 1024), b""):\n'
                     '            digest.update(block)\n'
                     '    return digest.hexdigest()\n'
                     '\n'
                     '\n'
                     'class StageCache:\n'
                     '    """Atomic, JSON-only checkpoints, isolated by inputs/code/runtime '
                     'fingerprint."""\n'
                     '    def __init__(self, directory, fingerprint):\n'
                     '        self.root = None\n'
                     '        if directory:\n'
                     '            key = hashlib.sha256(json.dumps(fingerprint, '
                     'sort_keys=True).encode()).hexdigest()\n'
                     '            self.root = Path(directory) / key\n'
                     '            self.root.mkdir(parents=True, exist_ok=True)\n'
                     '            self.write("manifest", fingerprint)\n'
                     '\n'
                     '    def read(self, name):\n'
                     '        if self.root is None:\n'
                     '            return None\n'
                     '        path = self.root / (name + ".json")\n'
                     '        if not path.exists():\n'
                     '            return None\n'
                     '        return json.loads(path.read_text())\n'
                     '\n'
                     '    def write(self, name, value):\n'
                     '        if self.root is None:\n'
                     '            return\n'
                     '        path = self.root / (name + ".json")\n'
                     '        temporary = path.with_suffix(".tmp")\n'
                     '        temporary.write_text(json.dumps(value, ensure_ascii=False))\n'
                     '        os.replace(temporary, path)\n'
                     '\n'
                     '    def get(self, name, compute):\n'
                     '        value = self.read(name)\n'
                     '        if value is not None:\n'
                     '            print(f"Reusing checkpoint: {name}")\n'
                     '            return value\n'
                     '        value = compute()\n'
                     '        self.write(name, value)\n'
                     '        return value\n'
                     '\n'
                     '\n'
                     'def create_face_analyzer(device, ort, factory):\n'
                     '    if device == "cuda" and hasattr(ort, "preload_dlls"):\n'
                     '        ort.preload_dlls()\n'
                     '    use_cuda = device == "cuda" and "CUDAExecutionProvider" in '
                     'ort.get_available_providers()\n'
                     '    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if '
                     'use_cuda else ["CPUExecutionProvider"]\n'
                     '    analyzer = factory(name="buffalo_l", providers=providers)\n'
                     '    analyzer.prepare(ctx_id=0 if use_cuda else -1, det_size=(640, 640))\n'
                     '    actual = {name: model.session.get_providers() for name, model in '
                     'analyzer.models.items()\n'
                     '              if getattr(model, "session", None) is not None}\n'
                     '    print(f"Face analysis actual providers: {actual}")\n'
                     '    if device == "cuda" and (not actual or any("CUDAExecutionProvider" not '
                     'in value for value in actual.values())):\n'
                     '        print("WARNING: one or more face models are using CPU; check '
                     'onnxruntime-gpu/CUDA libraries.")\n'
                     '    return analyzer, actual\n'
                     '\n'
                     '\n'
                     'def full_audio_chunks(sample_count, sample_rate, chunk_size=30):\n'
                     '    """Cover every sample with bounded windows; do not infer whether it is '
                     'speech."""\n'
                     '    if sample_count <= 0 or sample_rate <= 0 or chunk_size <= 0:\n'
                     '        raise ValueError("Audio length, sample rate and chunk size must be '
                     'positive")\n'
                     '    step = max(1, int(sample_rate * chunk_size))\n'
                     '    return [{"start": left / sample_rate, "end": min(left + step, '
                     'sample_count) / sample_rate}\n'
                     '            for left in range(0, sample_count, step)]\n'
                     '\n'
                     '\n'
                     'def create_full_audio_vad():\n'
                     '    """WhisperX coverage adapter for controlled experiments, not a speech '
                     'detector."""\n'
                     '    from whisperx.vads.vad import Vad\n'
                     '\n'
                     '    class FullAudio(Vad):\n'
                     '        @staticmethod\n'
                     '        def preprocess_audio(audio):\n'
                     '            return audio\n'
                     '\n'
                     '        def __call__(self, inputs):\n'
                     '            return {"sample_count": len(inputs["waveform"]), "sample_rate": '
                     'inputs["sample_rate"]}\n'
                     '\n'
                     '        @staticmethod\n'
                     '        def merge_chunks(segments, chunk_size, onset, offset):\n'
                     '            return full_audio_chunks(segments["sample_count"], '
                     'segments["sample_rate"], chunk_size)\n'
                     '\n'
                     '    return FullAudio(.5)\n',
 'export_confident_transcript.py': '"""Export a readable transcript while retaining every omitted '
                                   'row for review."""\n'
                                   '\n'
                                   'import argparse\n'
                                   'import json\n'
                                   'from pathlib import Path\n'
                                   '\n'
                                   '\n'
                                   'def timestamp(seconds):\n'
                                   '    milliseconds = round(float(seconds) * 1000)\n'
                                   '    hours, remainder = divmod(milliseconds, 3_600_000)\n'
                                   '    minutes, remainder = divmod(remainder, 60_000)\n'
                                   '    secs = remainder / 1000\n'
                                   '    return f"{hours:02d}:{minutes:02d}:{secs:06.3f}"\n'
                                   '\n'
                                   '\n'
                                   'def classify(segment, target_minimum=0.35, '
                                   'other_minimum=0.65,\n'
                                   '             ambiguous_target_tracks=()):\n'
                                   '    speaker = segment.get("final_speaker", "Uncertain")\n'
                                   '    confidence = float(segment.get("final_confidence", 0.0))\n'
                                   '    if speaker == "Overlapping_Speakers":\n'
                                   '        return False, "overlapping_speakers"\n'
                                   '    if speaker in ("Uncertain", "Unknown_Speaker", '
                                   '"NonTarget_Unknown"):\n'
                                   '        return False, "uncertain_identity"\n'
                                   '    if speaker in ambiguous_target_tracks:\n'
                                   '        return False, "ambiguous_target_like_track"\n'
                                   '    threshold = target_minimum if speaker == "Target_Speaker" '
                                   'else other_minimum\n'
                                   '    if confidence < threshold:\n'
                                   '        return False, "below_confidence_threshold"\n'
                                   '    if not str(segment.get("text", "")).strip():\n'
                                   '        return False, "empty_text"\n'
                                   '    return True, "included"\n'
                                   '\n'
                                   '\n'
                                   'def export_transcript(payload, output_dir, '
                                   'target_minimum=0.35,\n'
                                   '                      other_minimum=0.65):\n'
                                   '    output_dir = Path(output_dir)\n'
                                   '    output_dir.mkdir(parents=True, exist_ok=True)\n'
                                   '    target_track = payload.get("target_candidate")\n'
                                   '    cluster_means = payload.get("cluster_voice_means", {})\n'
                                   '    target_mean = cluster_means.get(target_track)\n'
                                   '    ambiguous_target_tracks = set()\n'
                                   '    if target_mean is not None:\n'
                                   '        ambiguous_target_tracks = {\n'
                                   '            track for track, mean in cluster_means.items()\n'
                                   '            if track != target_track and mean >= 0.18\n'
                                   '            and target_mean - mean <= 0.20\n'
                                   '        }\n'
                                   '    included, review = [], []\n'
                                   '    for index, segment in enumerate(payload.get("segments", '
                                   '[])):\n'
                                   '        keep, reason = classify(\n'
                                   '            segment, target_minimum, other_minimum, '
                                   'ambiguous_target_tracks\n'
                                   '        )\n'
                                   '        overlap = next((item for item in '
                                   'segment.get("evidence", [])\n'
                                   '                        if item.get("source") == '
                                   '"overlapping_speakers"), None)\n'
                                   '        overlap_details = overlap.get("details", {}) if '
                                   'overlap else {}\n'
                                   '        row = {\n'
                                   '            "baseline_index": index,\n'
                                   '            "start": segment["start"],\n'
                                   '            "end": segment["end"],\n'
                                   '            "speaker": segment.get("final_speaker", '
                                   '"Uncertain"),\n'
                                   '            "confidence": '
                                   'float(segment.get("final_confidence", 0.0)),\n'
                                   '            "text": segment.get("text", "").strip(),\n'
                                   '            "disposition": reason,\n'
                                   '            "unresolved_overlap": ({\n'
                                   '                "seconds": '
                                   'float(overlap_details.get("overlap_seconds", 0.0)),\n'
                                   '                "fraction": '
                                   'float(overlap_details.get("overlap_fraction", 0.0)),\n'
                                   '                "intervals": overlap_details.get("intervals", '
                                   '[]),\n'
                                   '            } if overlap and segment.get("final_speaker") != '
                                   '"Overlapping_Speakers"\n'
                                   '              else None),\n'
                                   '        }\n'
                                   '        (included if keep else review).append(row)\n'
                                   '\n'
                                   '    def render(rows, show_reason=False):\n'
                                   '        values = []\n'
                                   '        for row in rows:\n'
                                   '            suffix = f" [{row[\'disposition\']}]" if '
                                   'show_reason else ""\n'
                                   '            if row.get("unresolved_overlap"):\n'
                                   '                overlap = row["unresolved_overlap"]\n'
                                   '                suffix += (f" [unresolved overlap: '
                                   '{overlap[\'seconds\']:.2f}s, "\n'
                                   '                           f"{overlap[\'fraction\']:.0%} of '
                                   'segment]")\n'
                                   '            values.append(\n'
                                   '                '
                                   'f"[{timestamp(row[\'start\'])}–{timestamp(row[\'end\'])}] "\n'
                                   '                f"{row[\'speaker\']} '
                                   '({row[\'confidence\']:.2f}){suffix}: "\n'
                                   '                f"{row[\'text\']}"\n'
                                   '            )\n'
                                   '        return "\\n".join(values) + ("\\n" if values else "")\n'
                                   '\n'
                                   '    (output_dir / '
                                   '"confident_transcript.txt").write_text(render(included))\n'
                                   '    (output_dir / '
                                   '"review_transcript.txt").write_text(render(review, True))\n'
                                   '    (output_dir / "review_segments.json").write_text(\n'
                                   '        json.dumps(review, indent=2, ensure_ascii=False) + '
                                   '"\\n"\n'
                                   '    )\n'
                                   '    counts = {}\n'
                                   '    for row in review:\n'
                                   '        counts[row["disposition"]] = '
                                   'counts.get(row["disposition"], 0) + 1\n'
                                   '    summary = {\n'
                                   '        "baseline_modified": False,\n'
                                   '        "target_minimum": target_minimum,\n'
                                   '        "other_minimum": other_minimum,\n'
                                   '        "ambiguous_target_tracks": '
                                   'sorted(ambiguous_target_tracks),\n'
                                   '        "total_segments": len(included) + len(review),\n'
                                   '        "included_segments": len(included),\n'
                                   '        "review_segments": len(review),\n'
                                   '        "review_reasons": counts,\n'
                                   '        "included_duration_seconds": sum(\n'
                                   '            row["end"] - row["start"] for row in included\n'
                                   '        ),\n'
                                   '        "review_duration_seconds": sum(\n'
                                   '            row["end"] - row["start"] for row in review\n'
                                   '        ),\n'
                                   '    }\n'
                                   '    (output_dir / '
                                   '"summary.json").write_text(json.dumps(summary, indent=2) + '
                                   '"\\n")\n'
                                   '    return summary\n'
                                   '\n'
                                   '\n'
                                   'def main():\n'
                                   '    parser = argparse.ArgumentParser()\n'
                                   '    parser.add_argument("baseline", type=Path)\n'
                                   '    parser.add_argument("--output-dir", type=Path, '
                                   'required=True)\n'
                                   '    parser.add_argument("--target-minimum", type=float, '
                                   'default=0.35)\n'
                                   '    parser.add_argument("--other-minimum", type=float, '
                                   'default=0.65)\n'
                                   '    args = parser.parse_args()\n'
                                   '    payload = json.loads(args.baseline.read_text())\n'
                                   '    summary = export_transcript(\n'
                                   '        payload, args.output_dir, args.target_minimum, '
                                   'args.other_minimum\n'
                                   '    )\n'
                                   '    print(json.dumps(summary, indent=2))\n'
                                   '\n'
                                   '\n'
                                   'if __name__ == "__main__":\n'
                                   '    main()\n',
 'recover_transcript_gaps.py': '"""Recover review candidates only inside uncovered transcript '
                               'intervals.\n'
                               '\n'
                               'Existing segments are copied unchanged. Candidates are separate, '
                               'have no speaker\n'
                               'identity, and require review. Detection of a transcript gap does '
                               'not prove speech.\n'
                               '"""\n'
                               'import argparse\n'
                               'import copy\n'
                               'import hashlib\n'
                               'import json\n'
                               'import math\n'
                               'from pathlib import Path\n'
                               'import re\n'
                               'import subprocess\n'
                               'import numpy as np\n'
                               '\n'
                               '\n'
                               'def uncovered_intervals(segments, duration, minimum_gap=2.0):\n'
                               '    cursor=0.0; gaps=[]\n'
                               "    for segment in sorted(segments,key=lambda s:s['start']):\n"
                               "        start=max(0.0,min(duration,float(segment['start'])))\n"
                               "        end=max(start,min(duration,float(segment['end'])))\n"
                               '        if start-cursor>=minimum_gap: gaps.append((cursor,start))\n'
                               '        cursor=max(cursor,end)\n'
                               '    if '
                               'duration-cursor>=minimum_gap:gaps.append((cursor,duration))\n'
                               '    return gaps\n'
                               '\n'
                               '\n'
                               'def '
                               'recovery_windows(gap,duration,size=25.0,overlap=12.0,context=.5):\n'
                               '    if not all(math.isfinite(x) for x in (size,overlap)) or '
                               "size<=0 or not 0<=overlap<size:raise ValueError('Invalid window "
                               "size/overlap')\n"
                               '    if not math.isfinite(context) or context<0:raise '
                               "ValueError('Context must be finite and nonnegative')\n"
                               '    left=max(0.,gap[0]-context); '
                               'right=min(duration,gap[1]+context)\n'
                               '    if right-left<=size:return [(left,right)]\n'
                               '    starts=[];start=left\n'
                               '    while start+size<right:\n'
                               '        starts.append(start); start+=size-overlap\n'
                               '    final=max(left,right-size)\n'
                               '    if not starts or '
                               'abs(final-starts[-1])>1e-6:starts.append(final)\n'
                               '    return [(s,min(s+size,right)) for s in starts]\n'
                               '\n'
                               '\n'
                               "def word_key(text):return re.sub(r'[^\\w]+','',text.casefold())\n"
                               '\n'
                               '\n'
                               'def collect_candidates(observations,gap):\n'
                               '    # Retain all eligible words for review; repeated words need '
                               'distinct windows.\n'
                               '    clusters=[]\n'
                               '    for word in sorted(observations,key=lambda '
                               "w:(w['start'],w['window_index'])):\n"
                               "        if word['start']<gap[0] or word['end']>gap[1] or "
                               "word['end']<word['start']:continue\n"
                               "        key=word_key(word['word'])\n"
                               '        if not key:continue\n'
                               "        matches=[c for c in clusters if c['key']==key and "
                               "abs(c['anchor']-(word['start']+word['end'])/2)<=.6 and "
                               "word['window_index'] not in {x['window_index'] for x in "
                               "c['observations']}]\n"
                               '        if matches:\n'
                               '            closest=min(matches,key=lambda '
                               "c:abs(c['anchor']-(word['start']+word['end'])/2));closest['observations'].append(word)\n"
                               '        '
                               "else:clusters.append({'key':key,'anchor':(word['start']+word['end'])/2,'observations':[word]})\n"
                               '    # Keep words from one decoder window together; never splice '
                               'hypotheses.\n'
                               '    support={}\n'
                               '    for cluster in clusters:\n'
                               "        indices=sorted({w['window_index'] for w in "
                               "cluster['observations']})\n"
                               "        for w in cluster['observations']:\n"
                               '            '
                               "support[(w['window_index'],w['start'],w['end'],w['word'])]=indices\n"
                               '    hypotheses=[]\n'
                               "    for index in sorted({w['window_index'] for w in "
                               'observations}):\n'
                               '        runs=[]\n'
                               '        for w in sorted((w for w in observations if '
                               "w['window_index']==index and w['start']>=gap[0] and "
                               "w['end']<=gap[1] and w['end']>=w['start'] and "
                               "word_key(w['word'])),key=lambda w:w['start']):\n"
                               '            '
                               "word=dict(w,supporting_windows=support.get((index,w['start'],w['end'],w['word']),[index]))\n"
                               "            if runs and word['start']-runs[-1]['end']<=.65:\n"
                               '                '
                               "runs[-1]['words'].append(word);runs[-1]['end']=max(runs[-1]['end'],word['end'])\n"
                               '            '
                               "else:runs.append({'start':word['start'],'end':word['end'],'words':[word],'window_index':index})\n"
                               '        for run in runs:\n'
                               "            if run['end']<=run['start']:continue\n"
                               "            repeated=sum(len(w['supporting_windows'])>=2 for w in "
                               "run['words'])\n"
                               "            run.update(text=''.join(w['word'] for w in "
                               "run['words']).strip(),supported_word_count=repeated,supported_word_fraction=repeated/len(run['words']),repeated_in_overlapping_windows=repeated/len(run['words'])>=.5,review_required=True,speaker='Uncertain')\n"
                               '            hypotheses.append(run)\n'
                               '    selected=[]\n'
                               '    for run in sorted(hypotheses,key=lambda '
                               "r:(r['supported_word_count'],sum(w['probability'] for w in "
                               "r['words'])/len(r['words']),r['end']-r['start']),reverse=True):\n"
                               '        if '
                               "any(min(run['end'],chosen['end'])-max(run['start'],chosen['start'])>.15 "
                               'for chosen in selected):continue\n'
                               '        selected.append(run)\n'
                               "    return sorted(selected,key=lambda r:r['start'])\n"
                               '\n'
                               '\n'
                               'def preserve_baseline(baseline,candidates):\n'
                               '    result=copy.deepcopy(baseline)\n'
                               "    result['gap_recovery_candidates']=copy.deepcopy(candidates)\n"
                               "    result['gap_recovery_note']='Existing segments unchanged; "
                               'candidates require review and have no attributed speaker. Repeated '
                               "decoding is not ground truth.'\n"
                               '    return result\n'
                               '\n'
                               '\n'
                               'def main():\n'
                               '    parser=argparse.ArgumentParser(description=__doc__)\n'
                               "    parser.add_argument('video',type=Path)\n"
                               "    parser.add_argument('--baseline',required=True,type=Path)\n"
                               "    parser.add_argument('--output-dir',required=True,type=Path)\n"
                               "    parser.add_argument('--minimum-gap',type=float,default=2.)\n"
                               '    '
                               "parser.add_argument('--window-seconds',type=float,default=25.)\n"
                               '    '
                               "parser.add_argument('--overlap-seconds',type=float,default=12.)\n"
                               '    '
                               "parser.add_argument('--context-seconds',type=float,default=.5,help='Audio "
                               "context on each side; words outside the gap are never added')\n"
                               "    parser.add_argument('--language',default='en',help='Language "
                               "of the working transcript')\n"
                               '    args=parser.parse_args()\n'
                               "    if args.output_dir.exists():parser.error('Choose a new output "
                               "directory; existing results are never overwritten')\n"
                               '    if not math.isfinite(args.minimum_gap) or '
                               "args.minimum_gap<=0:parser.error('Minimum gap must be positive')\n"
                               '    '
                               'try:recovery_windows((0.,1.),1.,args.window_seconds,args.overlap_seconds,args.context_seconds)\n'
                               '    except ValueError as error:parser.error(str(error))\n'
                               '    baseline=json.loads(args.baseline.read_text())\n'
                               '    '
                               "raw=subprocess.check_output(['ffmpeg','-nostdin','-hide_banner','-loglevel','error','-i',str(args.video),'-vn','-ar','16000','-ac','1','-f','f32le','pipe:1'])\n"
                               '    '
                               "audio=np.frombuffer(raw,dtype='<f4').copy();duration=len(audio)/16000\n"
                               '    '
                               "gaps=uncovered_intervals(baseline['segments'],duration,args.minimum_gap)\n"
                               '    args.output_dir.mkdir(parents=True)\n'
                               '    candidates=[];decodes=[];model=None;device=None\n'
                               '    if gaps:\n'
                               '        import torch\n'
                               '        from faster_whisper import WhisperModel\n'
                               "        device='cuda' if torch.cuda.is_available() else 'cpu'\n"
                               '        '
                               "model=WhisperModel('large-v2',device=device,compute_type='float16' "
                               "if device=='cuda' else 'int8',cpu_threads=4)\n"
                               '    for gap_index,gap in enumerate(gaps):\n'
                               '        observations=[]\n'
                               '        for window_index,(left,right) in '
                               'enumerate(recovery_windows(gap,duration,args.window_seconds,args.overlap_seconds,args.context_seconds)):\n'
                               '            '
                               'segments,info=model.transcribe(audio[int(left*16000):int(right*16000)],language=args.language,vad_filter=False,beam_size=5,condition_on_previous_text=False,word_timestamps=True)\n'
                               '            rows=[]\n'
                               '            for segment in segments:\n'
                               '                eligible=bool(segment.avg_logprob>=-1.0 and '
                               'segment.no_speech_prob<=.6 and segment.compression_ratio<=2.4)\n'
                               '                '
                               "row={'start':left+segment.start,'end':left+segment.end,'text':segment.text,'avg_logprob':float(segment.avg_logprob),'no_speech_prob':float(segment.no_speech_prob),'compression_ratio':float(segment.compression_ratio),'quality_filter_passed':eligible,'words':[]}\n"
                               '                for word in segment.words or []:\n'
                               '                    '
                               "item={'start':left+word.start,'end':left+word.end,'word':word.word,'probability':float(word.probability),'window_index':window_index}\n"
                               "                    row['words'].append(item)\n"
                               '                    '
                               "item['low_confidence']=bool(word.probability<.4)\n"
                               '                    if eligible:observations.append(item)\n'
                               '                rows.append(row)\n'
                               '            '
                               "decodes.append({'gap_index':gap_index,'window_index':window_index,'window_start':left,'window_end':right,'segments':rows})\n"
                               '            '
                               "(args.output_dir/'window_decodes.json').write_text(json.dumps(decodes,indent=2)+'\\n')\n"
                               "            print('Decoded "
                               "gap',gap_index+1,'window',window_index+1,f'{left:.2f}-{right:.2f}',flush=True)\n"
                               '        for candidate in collect_candidates(observations,gap):\n'
                               '            '
                               "candidate['gap_index']=gap_index;candidates.append(candidate)\n"
                               '    output=preserve_baseline(baseline,candidates)\n'
                               "    assert output['segments']==baseline['segments'],'Existing "
                               "transcript changed'\n"
                               "    offset=float(baseline.get('source_offset_seconds',0))\n"
                               '    '
                               "output['gap_recovery_settings']={'model':'large-v2','device':device,'vad_filter':False,'condition_on_previous_text':False,'window_seconds':args.window_seconds,'overlap_seconds':args.overlap_seconds,'context_seconds':args.context_seconds,'minimum_gap':args.minimum_gap,'gaps':gaps,'baseline_sha256':hashlib.sha256(args.baseline.read_bytes()).hexdigest(),'video_sha256':hashlib.sha256(args.video.read_bytes()).hexdigest()}\n"
                               '    '
                               "(args.output_dir/'transcript_with_candidates.json').write_text(json.dumps(output,indent=2)+'\\n')\n"
                               '    lines=[]\n'
                               '    for s in '
                               'baseline[\'segments\']:lines.append((s[\'start\'],f"[{s[\'start\']+offset:.2f}-{s[\'end\']+offset:.2f}] '
                               "{s.get('final_speaker',s.get('speaker','Unknown'))}: "
                               '{s[\'text\']}"))\n'
                               '    for s in candidates:\n'
                               '        support=f"overlap support '
                               '{s[\'supported_word_count\']}/{len(s[\'words\'])} words" if '
                               "s['supported_word_count'] else 'single decode'\n"
                               '        '
                               'lines.append((s[\'start\'],f"[{s[\'start\']+offset:.2f}-{s[\'end\']+offset:.2f}] '
                               'REVIEW ({support}; speaker unknown): {s[\'text\']}"))\n'
                               '    '
                               "(args.output_dir/'review_transcript.txt').write_text('\\n'.join(text "
                               "for _,text in sorted(lines))+'\\n')\n"
                               "    print('Completed; "
                               "preserved',len(baseline['segments']),'existing "
                               "segments;',len(candidates),'review candidates',flush=True)\n"
                               '\n'
                               "if __name__=='__main__':main()\n",
 'reference_promotion.py': '"""Export and promote high-confidence post-run target reference '
                           'candidates.\n'
                           '\n'
                           'Export never changes a reference. Promotion requires an explicit '
                           'approvals file,\n'
                           'creates a new directory, records provenance, and leaves the parent '
                           'untouched.\n'
                           '"""\n'
                           '\n'
                           'import argparse\n'
                           'import hashlib\n'
                           'import json\n'
                           'from pathlib import Path\n'
                           'import shutil\n'
                           'import subprocess\n'
                           '\n'
                           'import numpy as np\n'
                           '\n'
                           '\n'
                           'DEFAULT_CRITERIA = {\n'
                           '    "minimum_final_confidence": 0.80,\n'
                           '    "minimum_duration_seconds": 1.50,\n'
                           '    "minimum_reference_similarity": 0.50,\n'
                           '    "minimum_local_voice_strength": 0.50,\n'
                           '    "requires_original_target_track": True,\n'
                           '    "requires_no_overlap": True,\n'
                           '}\n'
                           '\n'
                           '\n'
                           'def unit(value):\n'
                           '    value = np.asarray(value, dtype=np.float32).reshape(-1)\n'
                           '    norm = float(np.linalg.norm(value))\n'
                           '    if not np.isfinite(value).all() or norm <= 0:\n'
                           '        raise ValueError("Embedding must be finite and nonzero")\n'
                           '    return value / norm\n'
                           '\n'
                           '\n'
                           'def file_hash(path):\n'
                           '    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n'
                           '\n'
                           '\n'
                           'def source_key(video, start, end):\n'
                           '    return str(video), round(float(start), 3), round(float(end), 3)\n'
                           '\n'
                           '\n'
                           'def promoted_source_keys(metadata):\n'
                           '    """Collect promoted source intervals through the recorded ancestry '
                           'chain."""\n'
                           '    keys = set()\n'
                           '    if not isinstance(metadata, dict):\n'
                           '        return keys\n'
                           '    review = metadata.get("promotion_review", {})\n'
                           '    for row in review.get("promoted_candidates", []):\n'
                           '        source = row.get("source", {})\n'
                           '        if all(name in source for name in ("video", "start", "end")):\n'
                           '            keys.add(source_key(source["video"], source["start"], '
                           'source["end"]))\n'
                           '    parent = metadata.get("parent_reference", {}).get("metadata")\n'
                           '    keys.update(promoted_source_keys(parent))\n'
                           '    return keys\n'
                           '\n'
                           '\n'
                           'def select_candidates(payload, criteria=None, *, source_video=None,\n'
                           '                      excluded_sources=()):\n'
                           '    criteria = dict(DEFAULT_CRITERIA if criteria is None else '
                           'criteria)\n'
                           '    target_track = payload.get("target_candidate")\n'
                           '    selected = []\n'
                           '    for index, segment in enumerate(payload.get("segments", [])):\n'
                           '        duration = float(segment["end"]) - float(segment["start"])\n'
                           '        voice = next((item for item in segment.get("evidence", [])\n'
                           '                      if item.get("source") == "local_voice"), None)\n'
                           '        similarity = (voice or {}).get("details", '
                           '{}).get("similarity")\n'
                           '        has_overlap = any(item.get("source") == '
                           '"overlapping_speakers"\n'
                           '                          for item in segment.get("evidence", []))\n'
                           '        raw_track = segment.get("baseline", '
                           '{}).get("raw_speaker_track")\n'
                           '        if segment.get("final_speaker") != "Target_Speaker":\n'
                           '            continue\n'
                           '        if float(segment.get("final_confidence", 0)) < '
                           'criteria["minimum_final_confidence"]:\n'
                           '            continue\n'
                           '        if duration < criteria["minimum_duration_seconds"]:\n'
                           '            continue\n'
                           '        if criteria["requires_original_target_track"] and raw_track != '
                           'target_track:\n'
                           '            continue\n'
                           '        if criteria["requires_no_overlap"] and has_overlap:\n'
                           '            continue\n'
                           '        if voice is None or float(voice.get("confidence", 0)) < '
                           'criteria["minimum_local_voice_strength"]:\n'
                           '            continue\n'
                           '        if similarity is None or float(similarity) < '
                           'criteria["minimum_reference_similarity"]:\n'
                           '            continue\n'
                           '        if source_video is not None and source_key(\n'
                           '                source_video, segment["start"], segment["end"]) in '
                           'excluded_sources:\n'
                           '            continue\n'
                           '        selected.append({\n'
                           '            "baseline_index": index,\n'
                           '            "start": float(segment["start"]),\n'
                           '            "end": float(segment["end"]),\n'
                           '            "duration": duration,\n'
                           '            "text": str(segment.get("text", "")).strip(),\n'
                           '            "raw_speaker_track": raw_track,\n'
                           '            "final_confidence": float(segment["final_confidence"]),\n'
                           '            "reference_similarity": float(similarity),\n'
                           '            "local_voice_strength": float(voice["confidence"]),\n'
                           '            "review_required": True,\n'
                           '        })\n'
                           '    return selected\n'
                           '\n'
                           '\n'
                           'def evaluate_reference_update(parent_rows, candidate_rows,\n'
                           '                              minimum_parent_similarity=0.50,\n'
                           '                              minimum_centroid_similarity=0.995,\n'
                           '                              maximum_existing_median_drop=0.01):\n'
                           '    parent = np.stack([unit(row) for row in np.asarray(parent_rows)])\n'
                           '    candidates = np.stack([unit(row) for row in '
                           'np.asarray(candidate_rows)])\n'
                           '    old_centroid = unit(parent.mean(axis=0))\n'
                           '    new_centroid = unit(np.concatenate([parent, '
                           'candidates]).mean(axis=0))\n'
                           '    candidate_scores = candidates @ old_centroid\n'
                           '    centroid_similarity = float(np.dot(old_centroid, new_centroid))\n'
                           '    old_existing = parent @ old_centroid\n'
                           '    new_existing = parent @ new_centroid\n'
                           '    median_drop = float(np.median(old_existing) - '
                           'np.median(new_existing))\n'
                           '    checks = {\n'
                           '        "all_candidates_match_parent": bool(np.min(candidate_scores) '
                           '>= minimum_parent_similarity),\n'
                           '        "centroid_shift_is_bounded": bool(centroid_similarity >= '
                           'minimum_centroid_similarity),\n'
                           '        "existing_reference_affinity_is_preserved": bool(median_drop '
                           '<= maximum_existing_median_drop),\n'
                           '    }\n'
                           '    return {\n'
                           '        "passed": all(checks.values()),\n'
                           '        "checks": checks,\n'
                           '        "thresholds": {\n'
                           '            "minimum_parent_similarity": minimum_parent_similarity,\n'
                           '            "minimum_centroid_similarity": '
                           'minimum_centroid_similarity,\n'
                           '            "maximum_existing_median_drop": '
                           'maximum_existing_median_drop,\n'
                           '        },\n'
                           '        "candidate_similarity_to_parent_centroid": '
                           'candidate_scores.tolist(),\n'
                           '        "old_to_new_centroid_similarity": centroid_similarity,\n'
                           '        "existing_reference_median_affinity_drop": median_drop,\n'
                           '    }\n'
                           '\n'
                           '\n'
                           'def export_review(args):\n'
                           '    if args.output_dir.exists():\n'
                           '        raise ValueError("Output directory already exists")\n'
                           '    payload = json.loads(args.evidence.read_text())\n'
                           '    source_video = args.source_url or str(args.video.resolve())\n'
                           '    excluded_sources = set()\n'
                           '    if args.reference_metadata is not None:\n'
                           '        excluded_sources = promoted_source_keys(\n'
                           '            json.loads(args.reference_metadata.read_text())\n'
                           '        )\n'
                           '    candidates = select_candidates(\n'
                           '        payload, source_video=source_video,\n'
                           '        excluded_sources=excluded_sources,\n'
                           '    )\n'
                           '    args.output_dir.mkdir(parents=True)\n'
                           '    audio_dir = args.output_dir / "audio"\n'
                           '    audio_dir.mkdir()\n'
                           '    for row in candidates:\n'
                           '        name = (f"candidate-{row[\'baseline_index\']:04d}-"\n'
                           '                f"{row[\'start\']:.3f}-{row[\'end\']:.3f}.wav")\n'
                           '        destination = audio_dir / name\n'
                           '        subprocess.run([\n'
                           '            "ffmpeg", "-nostdin", "-hide_banner", "-loglevel", '
                           '"error", "-y",\n'
                           '            "-ss", str(row["start"]), "-i", str(args.video),\n'
                           '            "-t", str(row["duration"]), "-vn", "-ar", "16000", "-ac", '
                           '"1",\n'
                           '            str(destination),\n'
                           '        ], check=True)\n'
                           '        row["audio"] = f"audio/{name}"\n'
                           '        row["audio_sha256"] = file_hash(destination)\n'
                           '        row["source"] = {\n'
                           '            "video": source_video,\n'
                           '            "start": row["start"], "end": row["end"],\n'
                           '        }\n'
                           '    if candidates:\n'
                           '        import soundfile as sf\n'
                           '        parts = []\n'
                           '        silence = np.zeros(round(0.45 * 16000), dtype=np.float32)\n'
                           '        for row in candidates:\n'
                           '            wave, sample_rate = sf.read(\n'
                           '                args.output_dir / row["audio"], dtype="float32"\n'
                           '            )\n'
                           '            if sample_rate != 16000 or wave.ndim != 1:\n'
                           '                raise ValueError("Exported candidate audio must be '
                           'mono 16 kHz")\n'
                           '            peak = float(np.max(np.abs(wave))) if len(wave) else 0.0\n'
                           '            parts.extend([0.8 * wave / peak if peak else wave, '
                           'silence])\n'
                           '        sf.write(args.output_dir / "review-montage.wav",\n'
                           '                 np.concatenate(parts[:-1]), 16000)\n'
                           '    manifest = {\n'
                           '        "schema_version": 1,\n'
                           '        "purpose": "post-run target reference promotion review",\n'
                           '        "permanent_reference_modified": False,\n'
                           '        "source_evidence_sha256": file_hash(args.evidence),\n'
                           '        "criteria": DEFAULT_CRITERIA,\n'
                           '        "previously_promoted_sources_excluded": '
                           'len(excluded_sources),\n'
                           '        "candidates": candidates,\n'
                           '    }\n'
                           '    (args.output_dir / "manifest.json").write_text(\n'
                           '        json.dumps(manifest, indent=2, ensure_ascii=False) + "\\n"\n'
                           '    )\n'
                           '    approvals = {\n'
                           '        "manifest_sha256": file_hash(args.output_dir / '
                           '"manifest.json"),\n'
                           '        "approved_baseline_indices": [],\n'
                           '        "note": "Add only personally reviewed target-only clips.",\n'
                           '    }\n'
                           '    (args.output_dir / "approvals.json").write_text(\n'
                           '        json.dumps(approvals, indent=2) + "\\n"\n'
                           '    )\n'
                           '    (args.output_dir / "review.txt").write_text("\\n".join(\n'
                           '        f"{row[\'baseline_index\']}: '
                           '[{row[\'start\']:.3f}-{row[\'end\']:.3f}] "\n'
                           '        f"similarity={row[\'reference_similarity\']:.3f} '
                           '{row[\'text\']}"\n'
                           '        for row in candidates\n'
                           '    ) + ("\\n" if candidates else ""))\n'
                           '    print(json.dumps({"candidates": len(candidates),\n'
                           '                      "review": str(args.output_dir)}, indent=2))\n'
                           '\n'
                           '\n'
                           'def promote(args):\n'
                           '    if args.output_dir.exists():\n'
                           '        raise ValueError("Output reference directory already exists")\n'
                           '    manifest_path = args.review_dir / "manifest.json"\n'
                           '    manifest = json.loads(manifest_path.read_text())\n'
                           '    approvals = json.loads(args.approvals.read_text())\n'
                           '    if approvals.get("manifest_sha256") != file_hash(manifest_path):\n'
                           '        raise ValueError("Approvals do not match this review '
                           'manifest")\n'
                           '    approved = set(int(value) for value in '
                           'approvals.get("approved_baseline_indices", []))\n'
                           '    available = {int(row["baseline_index"]): row for row in '
                           'manifest["candidates"]}\n'
                           '    if not approved:\n'
                           '        raise ValueError("No candidates were approved")\n'
                           '    if not approved <= set(available):\n'
                           '        raise ValueError("Approvals contain an unknown baseline '
                           'index")\n'
                           '\n'
                           '    import torch\n'
                           '    from speechbrain.inference.speaker import SpeakerRecognition\n'
                           '    import soundfile as sf\n'
                           '\n'
                           '    parent_voice_path = args.parent_reference / '
                           '"voice_embeddings.npy"\n'
                           '    parent_rows = np.load(parent_voice_path, allow_pickle=False)\n'
                           '    device = args.device\n'
                           '    if device == "auto":\n'
                           '        device = "cuda" if torch.cuda.is_available() else "cpu"\n'
                           '    model = SpeakerRecognition.from_hparams(\n'
                           '        source="speechbrain/spkrec-ecapa-voxceleb",\n'
                           '        savedir=str(args.model_dir), run_opts={"device": device},\n'
                           '    )\n'
                           '    vectors, promoted = [], []\n'
                           '    for index in sorted(approved):\n'
                           '        row = dict(available[index])\n'
                           '        audio_path = args.review_dir / row["audio"]\n'
                           '        if file_hash(audio_path) != row["audio_sha256"]:\n'
                           '            raise ValueError(f"Candidate audio hash changed: '
                           '{index}")\n'
                           '        wave, sample_rate = sf.read(audio_path, dtype="float32")\n'
                           '        if sample_rate != 16000 or wave.ndim != 1:\n'
                           '            raise ValueError(f"Candidate audio must be mono 16 kHz: '
                           '{index}")\n'
                           '        with torch.no_grad():\n'
                           '            vector = model.encode_batch(\n'
                           '                torch.from_numpy(wave).unsqueeze(0).to(device)\n'
                           '            ).flatten().cpu().numpy()\n'
                           '        vectors.append(unit(vector))\n'
                           '        promoted.append(row)\n'
                           '    vectors = np.stack(vectors)\n'
                           '    validation = evaluate_reference_update(parent_rows, vectors)\n'
                           '    if not validation["passed"]:\n'
                           '        raise ValueError("Reference update failed safety checks: " + '
                           'json.dumps(validation))\n'
                           '\n'
                           '    shutil.copytree(args.parent_reference, args.output_dir)\n'
                           '    combined = np.concatenate([\n'
                           '        np.stack([unit(row) for row in parent_rows]), vectors\n'
                           '    ]).astype(np.float32)\n'
                           '    np.save(args.output_dir / "voice_embeddings.npy", combined)\n'
                           '    np.save(args.output_dir / "voice_embedding.npy", '
                           'unit(combined.mean(axis=0)))\n'
                           '    # Keep target-conditioned extraction aligned with the promoted '
                           'embedding\n'
                           '    # profile. The parent enrollment remains first and approved '
                           'samples are\n'
                           '    # appended with short silences; the parent directory is never '
                           'modified.\n'
                           '    enrollment_path = args.output_dir / "auditor_enrollment.wav"\n'
                           '    enrollment_parts = []\n'
                           '    if enrollment_path.exists():\n'
                           '        parent_enrollment, enrollment_rate = sf.read(\n'
                           '            enrollment_path, dtype="float32"\n'
                           '        )\n'
                           '        if enrollment_rate != 16000 or parent_enrollment.ndim != 1:\n'
                           '            raise ValueError("Parent enrollment must be mono 16 kHz")\n'
                           '        enrollment_parts.append(parent_enrollment)\n'
                           '    silence = np.zeros(round(0.25 * 16000), dtype=np.float32)\n'
                           '    for row in promoted:\n'
                           '        wave, _ = sf.read(args.review_dir / row["audio"], '
                           'dtype="float32")\n'
                           '        if enrollment_parts:\n'
                           '            enrollment_parts.append(silence)\n'
                           '        enrollment_parts.append(wave)\n'
                           '    sf.write(enrollment_path, np.concatenate(enrollment_parts), '
                           '16000)\n'
                           '\n'
                           '    parent_metadata_path = args.parent_reference / "reference.json"\n'
                           '    parent_metadata = (json.loads(parent_metadata_path.read_text())\n'
                           '                       if parent_metadata_path.exists() else None)\n'
                           '    metadata = {\n'
                           '        "schema_version": 1,\n'
                           '        "method": "reviewed post-run promotion; versioned and '
                           'reversible",\n'
                           '        "parent_reference": {\n'
                           '            "path": str(args.parent_reference),\n'
                           '            "voice_embeddings_sha256": file_hash(parent_voice_path),\n'
                           '            "reference_json_sha256": (file_hash(parent_metadata_path)\n'
                           '                                      if parent_metadata_path.exists() '
                           'else None),\n'
                           '            "metadata": parent_metadata,\n'
                           '        },\n'
                           '        "promotion_review": {\n'
                           '            "manifest_sha256": file_hash(manifest_path),\n'
                           '            "approvals_sha256": file_hash(args.approvals),\n'
                           '            "promoted_candidates": promoted,\n'
                           '        },\n'
                           '        "validation": validation,\n'
                           '        "voice": {\n'
                           '            "parent_rows": int(len(parent_rows)),\n'
                           '            "promoted_rows": int(len(vectors)),\n'
                           '            "total_rows": int(len(combined)),\n'
                           '            "enrollment_sha256": file_hash(enrollment_path),\n'
                           '        },\n'
                           '    }\n'
                           '    (args.output_dir / "reference.json").write_text(\n'
                           '        json.dumps(metadata, indent=2, ensure_ascii=False) + "\\n"\n'
                           '    )\n'
                           '    (args.output_dir / "promotion-validation.json").write_text(\n'
                           '        json.dumps(validation, indent=2) + "\\n"\n'
                           '    )\n'
                           '    print(json.dumps({"created": str(args.output_dir),\n'
                           '                      "promoted": sorted(approved),\n'
                           '                      "validation": validation}, indent=2))\n'
                           '\n'
                           '\n'
                           'def main():\n'
                           '    parser = argparse.ArgumentParser(description=__doc__)\n'
                           '    commands = parser.add_subparsers(dest="command", required=True)\n'
                           '    export = commands.add_parser("export")\n'
                           '    export.add_argument("--video", type=Path, required=True)\n'
                           '    export.add_argument("--evidence", type=Path, required=True)\n'
                           '    export.add_argument("--output-dir", type=Path, required=True)\n'
                           '    export.add_argument("--source-url")\n'
                           '    export.add_argument("--reference-metadata", type=Path)\n'
                           '    export.set_defaults(function=export_review)\n'
                           '    accept = commands.add_parser("promote")\n'
                           '    accept.add_argument("--review-dir", type=Path, required=True)\n'
                           '    accept.add_argument("--approvals", type=Path, required=True)\n'
                           '    accept.add_argument("--parent-reference", type=Path, '
                           'required=True)\n'
                           '    accept.add_argument("--output-dir", type=Path, required=True)\n'
                           '    accept.add_argument("--model-dir", type=Path,\n'
                           '                        '
                           'default=Path("pretrained_models/spkrec-ecapa-voxceleb"))\n'
                           '    accept.add_argument("--device", default="auto")\n'
                           '    accept.set_defaults(function=promote)\n'
                           '    args = parser.parse_args()\n'
                           '    args.function(args)\n'
                           '\n'
                           '\n'
                           'if __name__ == "__main__":\n'
                           '    main()\n',
 'repeat_evidence.py': '"""Detect repeated presentations and propose locally aligned transcript '
                       'evidence."""\n'
                       '\n'
                       'from bisect import bisect_left\n'
                       'from difflib import SequenceMatcher\n'
                       'import re\n'
                       'from statistics import median\n'
                       '\n'
                       '\n'
                       'STOP_WORDS = {\n'
                       '    "a", "an", "and", "are", "at", "be", "been", "but", "can", "did",\n'
                       '    "do", "does", "for", "from", "had", "has", "have", "he", "her", '
                       '"here",\n'
                       '    "him", "his", "how", "i", "if", "in", "is", "it", "just", "me", "my",\n'
                       '    "no", "not", "of", "on", "or", "our", "she", "so", "that", "the",\n'
                       '    "their", "them", "there", "they", "this", "to", "was", "we", "were",\n'
                       '    "what", "when", "where", "which", "who", "why", "will", "with", '
                       '"would",\n'
                       '    "you", "your",\n'
                       '}\n'
                       '\n'
                       '\n'
                       'def tokens(text):\n'
                       '    return re.findall(r"[a-z\']+", text.lower())\n'
                       '\n'
                       '\n'
                       'def phrase_similarity(left, right):\n'
                       '    """Favor ordered wording while requiring shared meaningful words."""\n'
                       '    left_tokens, right_tokens = tokens(left), tokens(right)\n'
                       '    left_content = set(left_tokens) - STOP_WORDS\n'
                       '    right_content = set(right_tokens) - STOP_WORDS\n'
                       '    shared = left_content & right_content\n'
                       '    if len(shared) < 2:\n'
                       '        return 0.0\n'
                       '    sequence = SequenceMatcher(None, left_tokens, right_tokens).ratio()\n'
                       '    jaccard = len(shared) / max(len(left_content | right_content), 1)\n'
                       '    return 0.65 * sequence + 0.35 * jaccard\n'
                       '\n'
                       '\n'
                       'def _ordered_unique(candidates):\n'
                       '    """Select a high-scoring, one-to-one monotonic anchor chain."""\n'
                       '    selected, used_left, used_right = [], set(), set()\n'
                       '    for candidate in sorted(candidates, key=lambda item: item["score"], '
                       'reverse=True):\n'
                       '        if candidate["left"] in used_left or candidate["right"] in '
                       'used_right:\n'
                       '            continue\n'
                       '        selected.append(candidate)\n'
                       '        used_left.add(candidate["left"])\n'
                       '        used_right.add(candidate["right"])\n'
                       '    selected.sort(key=lambda item: item["left"])\n'
                       '    chain = []\n'
                       '    for candidate in selected:\n'
                       '        position = bisect_left([item["right"] for item in chain], '
                       'candidate["right"])\n'
                       '        if position == len(chain):\n'
                       '            chain.append(candidate)\n'
                       '        elif candidate["score"] > chain[position]["score"]:\n'
                       '            chain[position] = candidate\n'
                       '    return chain\n'
                       '\n'
                       '\n'
                       'def find_repeat_groups(segments, minimum_separation=45.0, '
                       'minimum_score=0.55,\n'
                       '                       offset_tolerance=4.0, minimum_anchors=4,\n'
                       '                       minimum_span=20.0):\n'
                       '    candidates = []\n'
                       '    for left, first in enumerate(segments):\n'
                       '        for right in range(left + 1, len(segments)):\n'
                       '            second = segments[right]\n'
                       '            separation = float(second.start - first.start)\n'
                       '            if separation < minimum_separation:\n'
                       '                continue\n'
                       '            score = phrase_similarity(first.text, second.text)\n'
                       '            if score >= minimum_score:\n'
                       '                candidates.append({\n'
                       '                    "left": left,\n'
                       '                    "right": right,\n'
                       '                    "score": score,\n'
                       '                    "offset": separation,\n'
                       '                })\n'
                       '\n'
                       '    groups, remaining = [], candidates[:]\n'
                       '    while remaining:\n'
                       '        # Start with the offset having the largest local support, then '
                       'refine by median.\n'
                       '        seed = max(\n'
                       '            remaining,\n'
                       '            key=lambda item: sum(\n'
                       '                abs(other["offset"] - item["offset"]) <= offset_tolerance\n'
                       '                for other in remaining\n'
                       '            ),\n'
                       '        )\n'
                       '        nearby = [item for item in remaining\n'
                       '                  if abs(item["offset"] - seed["offset"]) <= '
                       'offset_tolerance]\n'
                       '        center = median(item["offset"] for item in nearby)\n'
                       '        nearby = [item for item in remaining\n'
                       '                  if abs(item["offset"] - center) <= offset_tolerance]\n'
                       '        chain = _ordered_unique(nearby)\n'
                       '        if len(chain) >= minimum_anchors:\n'
                       '            left_span = segments[chain[-1]["left"]].start - '
                       'segments[chain[0]["left"]].start\n'
                       '            right_span = segments[chain[-1]["right"]].start - '
                       'segments[chain[0]["right"]].start\n'
                       '            if left_span >= minimum_span and right_span >= minimum_span:\n'
                       '                group_id = f"repeat_{len(groups) + 1:02d}"\n'
                       '                groups.append({\n'
                       '                    "id": group_id,\n'
                       '                    "offset_seconds": median(item["offset"] for item in '
                       'chain),\n'
                       '                    "left_start": segments[chain[0]["left"]].start,\n'
                       '                    "left_end": segments[chain[-1]["left"]].end,\n'
                       '                    "right_start": segments[chain[0]["right"]].start,\n'
                       '                    "right_end": segments[chain[-1]["right"]].end,\n'
                       '                    "anchors": chain,\n'
                       '                })\n'
                       '        consumed = set((item["left"], item["right"]) for item in nearby)\n'
                       '        remaining = [item for item in remaining\n'
                       '                     if (item["left"], item["right"]) not in consumed]\n'
                       '    return groups\n'
                       '\n'
                       '\n'
                       'def _interpolate(value, source, destination):\n'
                       '    if value <= source[0]:\n'
                       '        return destination[0] + value - source[0]\n'
                       '    if value >= source[-1]:\n'
                       '        return destination[-1] + value - source[-1]\n'
                       '    for index in range(1, len(source)):\n'
                       '        if value <= source[index]:\n'
                       '            fraction = (value - source[index - 1]) / max(\n'
                       '                source[index] - source[index - 1], 1e-9\n'
                       '            )\n'
                       '            return destination[index - 1] + fraction * (\n'
                       '                destination[index] - destination[index - 1]\n'
                       '            )\n'
                       '    return destination[-1]\n'
                       '\n'
                       '\n'
                       'def build_repeat_proposals(segments, groups, maximum_timing_error=3.0):\n'
                       '    """Return donor candidates without changing text or speaker '
                       'identity."""\n'
                       '    proposals = {index: [] for index in range(len(segments))}\n'
                       '    for group in groups:\n'
                       '        anchors = group["anchors"]\n'
                       '        left_times = [segments[item["left"]].start for item in anchors]\n'
                       '        right_times = [segments[item["right"]].start for item in anchors]\n'
                       '        directions = (\n'
                       '            ("left", "right", left_times, right_times),\n'
                       '            ("right", "left", right_times, left_times),\n'
                       '        )\n'
                       '        for recipient_side, donor_side, source_times, donor_times in '
                       'directions:\n'
                       '            recipient_indices = [item[recipient_side] for item in '
                       'anchors]\n'
                       '            lower, upper = min(recipient_indices), max(recipient_indices)\n'
                       '            if lower > 0 and segments[lower].start - segments[lower - '
                       '1].end <= 5.0:\n'
                       '                lower -= 1\n'
                       '            if upper + 1 < len(segments) and segments[upper + 1].start - '
                       'segments[upper].end <= 5.0:\n'
                       '                upper += 1\n'
                       '            donor_pool = sorted({item[donor_side] for item in anchors})\n'
                       '            # Include segments between donor anchors so corrupted or '
                       'skipped anchor text\n'
                       '            # can still be proposed through the local time map.\n'
                       '            donor_lower, donor_upper = min(donor_pool), max(donor_pool)\n'
                       '            if donor_lower > 0 and segments[donor_lower].start - '
                       'segments[donor_lower - 1].end <= 5.0:\n'
                       '                donor_lower -= 1\n'
                       '            if (donor_upper + 1 < len(segments)\n'
                       '                    and segments[donor_upper + 1].start - '
                       'segments[donor_upper].end <= 5.0):\n'
                       '                donor_upper += 1\n'
                       '            donor_pool = list(range(donor_lower, donor_upper + 1))\n'
                       '            for recipient in range(lower, upper + 1):\n'
                       '                midpoint = (segments[recipient].start + '
                       'segments[recipient].end) / 2\n'
                       '                predicted = _interpolate(midpoint, source_times, '
                       'donor_times)\n'
                       '                donor = min(\n'
                       '                    donor_pool,\n'
                       '                    key=lambda index: abs(\n'
                       '                        (segments[index].start + segments[index].end) / 2 '
                       '- predicted\n'
                       '                    ),\n'
                       '                )\n'
                       '                donor_midpoint = (segments[donor].start + '
                       'segments[donor].end) / 2\n'
                       '                timing_error = abs(donor_midpoint - predicted)\n'
                       '                if timing_error > maximum_timing_error:\n'
                       '                    continue\n'
                       '                anchor_scores = [item["score"] for item in anchors]\n'
                       '                alignment = max(0.0, min(1.0,\n'
                       '                    median(anchor_scores) * (1.0 - timing_error / '
                       '(maximum_timing_error * 2))))\n'
                       '                proposals[recipient].append({\n'
                       '                    "group_id": group["id"],\n'
                       '                    "donor_start": segments[donor].start,\n'
                       '                    "donor_end": segments[donor].end,\n'
                       '                    "donor_text": segments[donor].text,\n'
                       '                    "donor_final_speaker": segments[donor].final_speaker,\n'
                       '                    "donor_final_confidence": '
                       'segments[donor].final_confidence,\n'
                       '                    "alignment_confidence": alignment,\n'
                       '                    "timing_error_seconds": timing_error,\n'
                       '                    "note": "Corroboration candidate from an aligned '
                       'repeated presentation; original text is preserved.",\n'
                       '                })\n'
                       '    return {index: rows for index, rows in proposals.items() if rows}\n'
                       '\n'
                       '\n'
                       'def repeat_target_corroboration(segment, proposals, '
                       'minimum_donor_confidence=0.75,\n'
                       '                                minimum_alignment=0.72,\n'
                       '                                minimum_text_similarity=0.85,\n'
                       '                                minimum_local_similarity=0.05):\n'
                       '    """Return one strict target corroboration candidate, without changing '
                       'text.\n'
                       '\n'
                       '    This intentionally handles only nearly identical repeated speech. '
                       'Partial\n'
                       '    wording, overlap, weak donors, and local acoustic contradictions '
                       'remain\n'
                       '    review-only.\n'
                       '    """\n'
                       '    if segment.final_speaker in ("Target_Speaker", '
                       '"Overlapping_Speakers"):\n'
                       '        return None\n'
                       '    voice = next(\n'
                       '        (item for item in segment.evidence if item.source == '
                       '"local_voice"), None\n'
                       '    )\n'
                       '    local_similarity = (\n'
                       '        voice.details.get("similarity") if voice is not None else None\n'
                       '    )\n'
                       '    if local_similarity is None or local_similarity < '
                       'minimum_local_similarity:\n'
                       '        return None\n'
                       '    candidates = []\n'
                       '    for proposal in proposals:\n'
                       '        similarity = phrase_similarity(segment.text, '
                       'proposal["donor_text"])\n'
                       '        if (proposal["donor_final_speaker"] == "Target_Speaker"\n'
                       '                and proposal["donor_final_confidence"] >= '
                       'minimum_donor_confidence\n'
                       '                and proposal["alignment_confidence"] >= minimum_alignment\n'
                       '                and similarity >= minimum_text_similarity):\n'
                       '            candidates.append({\n'
                       '                **proposal,\n'
                       '                "recipient_text_similarity": similarity,\n'
                       '                "recipient_local_target_similarity": local_similarity,\n'
                       '                "note": "Strict target corroboration from a nearly '
                       'identical aligned repeated presentation; text is unchanged.",\n'
                       '            })\n'
                       '    if not candidates:\n'
                       '        return None\n'
                       '    return max(candidates, key=lambda item: (\n'
                       '        item["recipient_text_similarity"],\n'
                       '        item["alignment_confidence"],\n'
                       '        item["donor_final_confidence"],\n'
                       '    ))\n'
                       '\n'
                       '\n'
                       'def resolve_repeat_target_corroboration(segment, details):\n'
                       '    """Second-pass resolver for a candidate produced by the strict '
                       'gate."""\n'
                       '    if details is None:\n'
                       '        return False\n'
                       '    segment.final_speaker = "Target_Speaker"\n'
                       '    segment.final_confidence = float(min(\n'
                       '        0.55,\n'
                       '        details["donor_final_confidence"],\n'
                       '        details["alignment_confidence"],\n'
                       '        details["recipient_text_similarity"],\n'
                       '    ))\n'
                       '    segment.reasons.append(\n'
                       '        "nearly identical repeated presentation corroborates target '
                       'identity"\n'
                       '    )\n'
                       '    return True\n',
 'review_audio_window.py': '"""Optional local audio-window review using the project\'s existing '
                           'models.\n'
                           '\n'
                           'Writes independent ASR/alignment/voice hypotheses, never edits a '
                           'baseline.\n'
                           'No expected transcript text, named-video rules, clothing rules, or HF '
                           'token.\n'
                           '"""\n'
                           'import argparse\n'
                           'import gc\n'
                           'import hashlib\n'
                           'import json\n'
                           'import math\n'
                           'from pathlib import Path\n'
                           'import subprocess\n'
                           'import time\n'
                           '\n'
                           '\n'
                           'def baseline_evidence(segments, start, end, offset=0):\n'
                           '    tracks = {}\n'
                           '    sources = []\n'
                           '    for index, s in enumerate(segments):\n'
                           '        '
                           "overlap=max(0,min(end,float(s['end'])+offset)-max(start,float(s['start'])+offset))\n"
                           '        if not overlap: continue\n'
                           '        '
                           "track=s.get('raw_speaker_track',s.get('baseline',{}).get('raw_speaker_track',s.get('base_track',s.get('speaker'))))\n"
                           '        '
                           "sources.append({'segment_index':index,'raw_speaker_track':track,\n"
                           '            '
                           "'baseline_speaker':s.get('final_speaker',s.get('speaker')),\n"
                           "            'overlap_seconds':overlap})\n"
                           '        if track is not None: '
                           'tracks[track]=tracks.get(track,0)+overlap\n'
                           '    return '
                           "{'source':'baseline_overlap','raw_track_overlap_seconds':tracks,\n"
                           "            'segments':sources,'identity_verified':False}\n"
                           '\n'
                           '\n'
                           'def decoder_sentence_bounds(decoded_segments, aligned_segments):\n'
                           '    """Link generated sentence text to its own decoder words, in '
                           'sequence.\n'
                           '\n'
                           '    This matches two representations of the same ASR hypothesis, not '
                           'supplied\n'
                           '    expected dialogue. Missing links yield None rather than invented '
                           'timings.\n'
                           '    """\n'
                           '    text_parts=[];word_spans=[];base=0\n'
                           '    for segment in decoded_segments:\n'
                           "        body=' '.join(segment['text'].split());cursor=0\n"
                           "        for word in segment.get('words',[]):\n"
                           "            token=' '.join(word.get('word','').split())\n"
                           '            location=body.find(token,cursor) if token else -1\n'
                           '            if location<0:continue\n'
                           '            '
                           "word_spans.append((base+location,base+location+len(token),float(word['start']),float(word['end'])))\n"
                           '            cursor=location+len(token)\n'
                           '        text_parts.append(body);base+=len(body)+1\n'
                           "    text=' '.join(text_parts);cursor=0;bounds=[]\n"
                           '    for segment in aligned_segments:\n'
                           "        sentence=' "
                           "'.join(segment['text'].split());left=text.find(sentence,cursor) if "
                           'sentence else -1\n'
                           '        if left<0:\n'
                           '            bounds.append(None);continue\n'
                           '        right=left+len(sentence);cursor=right\n'
                           '        words=[w for w in word_spans if w[0]<right and w[1]>left]\n'
                           '        if not words or max(w[3] for w in words)<=min(w[2] for w in '
                           'words):\n'
                           '            bounds.append(None)\n'
                           '        else:bounds.append((min(w[2] for w in words),max(w[3] for w in '
                           'words)))\n'
                           '    return bounds\n'
                           '\n'
                           '\n'
                           'def resolve_voice(scores, min_similarity=.25, min_margin=.08):\n'
                           '    """Conservative review hypothesis; thresholds are not calibrated '
                           'confidence."""\n'
                           "    if not scores: return 'Uncertain'\n"
                           '    ranked=sorted(scores,key=scores.get,reverse=True)\n'
                           '    # A margin requires at least one competing profile. With only a '
                           'target\n'
                           '    # reference, use similarity alone but keep the explicit review '
                           'requirement.\n'
                           '    runner=scores[ranked[1]] if len(ranked)>1 else None\n'
                           "    if scores[ranked[0]] < min_similarity: return 'Uncertain'\n"
                           '    if runner is not None and scores[ranked[0]]-runner < min_margin: '
                           "return 'Uncertain'\n"
                           '    return ranked[0]\n'
                           '\n'
                           '\n'
                           'def resolve_timing_evidence(variants, min_similarity=.25, '
                           'min_margin=.08):\n'
                           '    """Use one evidence family: agree on the leading voice, with '
                           'qualified support.\n'
                           '\n'
                           '    Alternate crops are not independent votes and do not raise '
                           'confidence.\n'
                           '    Conflicting leading identities abstain, even if one crop matches '
                           'strongly.\n'
                           '    """\n'
                           '    usable=[scores for scores in variants if scores]\n'
                           "    if not usable:return 'Uncertain'\n"
                           '    leaders={max(scores,key=scores.get) for scores in usable}\n'
                           "    if len(leaders)!=1:return 'Uncertain'\n"
                           '    leader=next(iter(leaders))\n'
                           '    return leader if '
                           'any(resolve_voice(scores,min_similarity,min_margin)==leader for scores '
                           "in usable) else 'Uncertain'\n"
                           '\n'
                           '\n'
                           'def decoder_quality_flags(segments, start, end):\n'
                           '    """Keep decoder warnings as evidence; word probability is not '
                           'accuracy."""\n'
                           "    observed=[s for s in segments if s['start']<end and "
                           "s['end']>start]\n"
                           '    flags=[]\n'
                           "    if any(s.get('no_speech_prob',0)>.6 for s in observed):\n"
                           "        flags.append('Decoder marks this passage as possible "
                           "non-speech')\n"
                           "    if any(s.get('avg_logprob',0)<-1 for s in observed):\n"
                           "        flags.append('Low decoder support for wording')\n"
                           "    if any(s.get('compression_ratio',0)>2.4 for s in observed):\n"
                           "        flags.append('Decoder wording may be repetitive')\n"
                           "    return flags, [{'start':s['start'],'end':s['end'],**{k:s[k] for k "
                           "in ('avg_logprob','no_speech_prob','compression_ratio') if k in s}} "
                           'for s in observed]\n'
                           '\n'
                           '\n'
                           'def sha(path):\n'
                           '    h=hashlib.sha256()\n'
                           "    with path.open('rb') as f:\n"
                           "        for chunk in iter(lambda:f.read(1024*1024),b''): "
                           'h.update(chunk)\n'
                           '    return h.hexdigest()\n'
                           '\n'
                           '\n'
                           'def main():\n'
                           '    p=argparse.ArgumentParser(description=__doc__)\n'
                           "    p.add_argument('--video',type=Path,required=True)\n"
                           "    p.add_argument('--start',type=float,required=True)\n"
                           "    p.add_argument('--duration',type=float,required=True)\n"
                           "    p.add_argument('--target-reference',type=Path,required=True)\n"
                           '    '
                           "p.add_argument('--other-reference',action='append',default=[],metavar='NAME=PATH')\n"
                           "    p.add_argument('--baseline',type=Path)\n"
                           "    p.add_argument('--baseline-offset',type=float,default=0,\n"
                           "                   help='Add this offset to baseline times; default "
                           "assumes absolute video times')\n"
                           "    p.add_argument('--output-dir',type=Path,required=True)\n"
                           "    p.add_argument('--model',default='large-v2')\n"
                           '    '
                           "p.add_argument('--asr-engine',choices=['native','bounded-whisperx'],default='native')\n"
                           '    '
                           "p.add_argument('--timing-source',choices=['consensus','decoder','alignment'],default='consensus')\n"
                           "    p.add_argument('--language',default='en')\n"
                           '    '
                           "p.add_argument('--device',choices=['auto','cpu','cuda'],default='auto')\n"
                           "    p.add_argument('--threads',type=int,default=4)\n"
                           "    p.add_argument('--chunk-seconds',type=float,default=20)\n"
                           "    p.add_argument('--context-seconds',type=float,default=.5)\n"
                           "    p.add_argument('--min-similarity',type=float,default=.25)\n"
                           "    p.add_argument('--min-margin',type=float,default=.08)\n"
                           "    p.add_argument('--speechbrain-cache',type=Path)\n"
                           '    a=p.parse_args()\n'
                           '    if not math.isfinite(a.start) or a.start<0 or not '
                           'math.isfinite(a.duration) or a.duration<=0:\n'
                           "        p.error('Provide a nonnegative start and positive duration')\n"
                           '    if not math.isfinite(a.baseline_offset) or a.threads<1:\n'
                           "        p.error('Invalid offset or thread count')\n"
                           '    if not math.isfinite(a.chunk_seconds) or not '
                           '1<=a.chunk_seconds<=30:\n'
                           "        p.error('Chunk length must be between 1 and 30 seconds')\n"
                           '    if not -1<=a.min_similarity<=1 or not 0<=a.min_margin<=2:\n'
                           "        p.error('Invalid voice gates')\n"
                           "    references={'Target_Speaker':a.target_reference}\n"
                           '    for item in a.other_reference:\n'
                           "        if '=' not in item:p.error('Other reference must be "
                           "NAME=PATH')\n"
                           "        name,path=item.split('=',1)\n"
                           '        if not name or name in references or name in '
                           "('Unknown','Uncertain'):p.error('Reference name must be unique')\n"
                           '        references[name]=Path(path)\n'
                           '    paths=[a.video,*references.values()]+([a.baseline] if a.baseline '
                           'else [])\n'
                           '    for path in paths:\n'
                           "        if not path.is_file():p.error(f'File not found: {path}')\n"
                           '    planned=[a.output_dir/x for x in '
                           "('asr.json','alignment.json','review_hypotheses.json','review_transcript.txt')]\n"
                           '    if any(path.resolve() in [x.resolve() for x in paths] for path in '
                           'planned):\n'
                           "        p.error('Outputs must be separate from input files')\n"
                           '    if any(path.exists() for path in planned):\n'
                           "        p.error('Choose an empty output directory to preserve earlier "
                           "experiments')\n"
                           '    import numpy as np\n'
                           '    import torch\n'
                           '    import whisperx\n'
                           '    from bounded_silero_vad import make_vad\n'
                           '    torch.set_num_threads(a.threads)\n'
                           "    device=('cuda' if torch.cuda.is_available() else 'cpu') if "
                           "a.device=='auto' else a.device\n"
                           "    if device=='cuda' and not torch.cuda.is_available():p.error('CUDA "
                           "is unavailable')\n"
                           '    a.output_dir.mkdir(parents=True,exist_ok=True)\n'
                           '    started=time.monotonic()\n'
                           '    '
                           "raw=subprocess.check_output(['ffmpeg','-nostdin','-hide_banner','-loglevel','error',\n"
                           '        '
                           "'-ss',str(a.start),'-i',str(a.video),'-t',str(a.duration),'-vn',\n"
                           "        '-ar','16000','-ac','1','-f','f32le','pipe:1'])\n"
                           "    audio=np.frombuffer(raw,dtype='<f4').copy()\n"
                           "    if not len(audio) or not np.isfinite(audio).all():p.error('No "
                           "valid audio decoded')\n"
                           "    if a.asr_engine=='native':\n"
                           '        from faster_whisper import WhisperModel\n'
                           "        asr=WhisperModel(a.model,device=device,compute_type='float16' "
                           "if device=='cuda' else 'int8',cpu_threads=a.threads)\n"
                           '        '
                           'decoded,_=asr.transcribe(audio,language=a.language,beam_size=5,\n'
                           '            '
                           'vad_filter=False,condition_on_previous_text=False,word_timestamps=True)\n'
                           '        '
                           "result={'language':a.language,'segments':[{'start':float(s.start),'end':float(s.end),'text':s.text,\n"
                           '            '
                           "'avg_logprob':float(s.avg_logprob),'no_speech_prob':float(s.no_speech_prob),\n"
                           "            'compression_ratio':float(s.compression_ratio),\n"
                           '            '
                           "'words':[{'start':float(w.start),'end':float(w.end),'word':w.word,'probability':float(w.probability)} "
                           'for w in s.words or []]} for s in decoded]}\n'
                           '    else:\n'
                           "        asr=whisperx.load_model(a.model,device,compute_type='float16' "
                           "if device=='cuda' else 'int8',\n"
                           '            '
                           'language=a.language,vad_model=make_vad(context=a.context_seconds))\n'
                           '        '
                           'result=asr.transcribe(audio,batch_size=4,chunk_size=a.chunk_seconds,language=a.language)\n'
                           '    '
                           "(a.output_dir/'asr.json').write_text(json.dumps(result,indent=2)+'\\n')\n"
                           '    del asr;gc.collect()\n'
                           "    if device=='cuda':torch.cuda.empty_cache()\n"
                           "    if result['segments']:\n"
                           '        '
                           'aligner,metadata=whisperx.load_align_model(language_code=a.language,device=device)\n'
                           '        '
                           "aligned=whisperx.align(result['segments'],aligner,metadata,audio,device,return_char_alignments=False)\n"
                           '        del aligner;gc.collect()\n'
                           "        if device=='cuda':torch.cuda.empty_cache()\n"
                           "    else:aligned={'segments':[],'word_segments':[]}\n"
                           '    '
                           "(a.output_dir/'alignment.json').write_text(json.dumps(aligned,indent=2)+'\\n')\n"
                           '    from speechbrain.inference.speaker import SpeakerRecognition\n'
                           '    '
                           "options={'source':'speechbrain/spkrec-ecapa-voxceleb','run_opts':{'device':device}}\n"
                           '    if '
                           "a.speechbrain_cache:options['savedir']=str(a.speechbrain_cache)\n"
                           '    encoder=SpeakerRecognition.from_hparams(**options)\n'
                           '    def unit(v):\n'
                           '        v=np.asarray(v,dtype=np.float32).reshape(-1)\n'
                           '        if not np.isfinite(v).all() or np.linalg.norm(v)<1e-8:raise '
                           "ValueError('Invalid embedding')\n"
                           '        return v/np.linalg.norm(v)\n'
                           '    profiles={}\n'
                           '    for name,path in references.items():\n'
                           '        values=np.load(path,allow_pickle=False)\n'
                           '        if values.ndim==1:values=values[None,:]\n'
                           '        if values.ndim!=2 or not len(values):raise '
                           "ValueError('Reference must contain one or more embeddings')\n"
                           '        profiles[name]=unit(np.mean([unit(v) for v in '
                           'values],axis=0))\n'
                           '    baseline=json.loads(a.baseline.read_text()) if a.baseline else '
                           "{'segments':[]}\n"
                           '    rows=[]\n'
                           '    '
                           "decoder_bounds=decoder_sentence_bounds(result['segments'],aligned['segments'])\n"
                           "    for index,s in enumerate(aligned['segments']):\n"
                           '        '
                           "alignment_left,alignment_right=float(s['start']),float(s['end']);scores={};reasons=[]\n"
                           '        bounds=decoder_bounds[index]\n'
                           "        use_decoder=a.timing_source in ('decoder','consensus') and "
                           'bounds is not None\n'
                           '        left,right=bounds if use_decoder else '
                           '(alignment_left,alignment_right)\n'
                           "        if a.timing_source in ('decoder','consensus') and bounds is "
                           "None:reasons.append('Decoder word boundaries unavailable; alignment "
                           "fallback needs review')\n"
                           '        if right-left>=.4:\n'
                           '            '
                           'crop=torch.from_numpy(audio[round(left*16000):round(right*16000)]).unsqueeze(0).to(device)\n'
                           '            with '
                           'torch.no_grad():v=unit(encoder.encode_batch(crop).detach().cpu().numpy())\n'
                           '            for name,profile in profiles.items():\n'
                           "                if v.shape!=profile.shape:raise ValueError('Reference "
                           "embedding model/dimension mismatch')\n"
                           '                scores[name]=float(v@profile)\n'
                           "        else:reasons.append('Voice crop shorter than 0.4 seconds')\n"
                           "        variants=[{'source':'decoder' if use_decoder else "
                           "'alignment','start':a.start+left,'end':a.start+right,'scores':scores}]\n"
                           "        if a.timing_source=='consensus' and use_decoder and "
                           'alignment_right-alignment_left>=.4 and '
                           '(abs(left-alignment_left)>1/16000 or '
                           'abs(right-alignment_right)>1/16000):\n'
                           '            '
                           'crop=torch.from_numpy(audio[round(alignment_left*16000):round(alignment_right*16000)]).unsqueeze(0).to(device)\n'
                           '            with '
                           'torch.no_grad():alternate=unit(encoder.encode_batch(crop).detach().cpu().numpy())\n'
                           '            '
                           "variants.append({'source':'alignment','start':a.start+alignment_left,'end':a.start+alignment_right,'scores':{name:float(alternate@profile) "
                           'for name,profile in profiles.items()}})\n'
                           "        speaker=resolve_timing_evidence([v['scores'] for v in "
                           'variants],a.min_similarity,a.min_margin) if '
                           "a.timing_source=='consensus' else "
                           'resolve_voice(scores,a.min_similarity,a.min_margin)\n'
                           '        '
                           "quality_flags,quality_signals=decoder_quality_flags(result['segments'],*(bounds "
                           'if bounds is not None else (left,right)))\n'
                           '        reasons.extend(quality_flags)\n'
                           '        near_edge=left<=.25 or right>=len(audio)/16000-.25\n'
                           "        if near_edge:reasons.append('Near audio-window boundary; "
                           "wording or timing may be incomplete')\n"
                           "        if speaker=='Uncertain':reasons.append('Insufficient voice "
                           "similarity or separation between profiles')\n"
                           "        if len(profiles)==1:reasons.append('No competing voice "
                           "reference; target-only match needs review')\n"
                           '        absolute_start,absolute_end=a.start+left,a.start+right\n'
                           '        '
                           "rows.append({'index':index,'start':absolute_start,'end':absolute_end,'text':s['text'],\n"
                           '            '
                           "'speaker_hypothesis':speaker,'transcription_status':'decoder_warning' "
                           'if quality_flags else '
                           "'review_hypothesis','review_required':True,'near_window_boundary':near_edge,'reasons':reasons,\n"
                           '            '
                           "'evidence':[{'source':'decoder_support','flags':quality_flags,'signals':quality_signals,'word_probability_is_accuracy':False},{'source':'timing_comparison','selected_source':'decoder' "
                           "if use_decoder else 'alignment',\n"
                           '                '
                           "'alignment_start':a.start+alignment_left,'alignment_end':a.start+alignment_right,\n"
                           "                'decoder_start':a.start+bounds[0] if bounds else "
                           "None,'decoder_end':a.start+bounds[1] if bounds else "
                           "None},baseline_evidence(baseline['segments'],absolute_start,absolute_end,a.baseline_offset),\n"
                           '                '
                           "{'source':'local_voice','similarities':scores,'timing_variants':variants,'variants_are_independent_votes':False,'min_similarity':a.min_similarity,\n"
                           '                 '
                           "'min_margin':a.min_margin,'identity_probability_calibrated':False}],\n"
                           "            'alignment_words':[{**w,**({'start':a.start+w['start']} if "
                           "'start' in w else {}),\n"
                           "                      **({'end':a.start+w['end']} if 'end' in w else "
                           "{})} for w in s.get('words',[])]})\n"
                           '    '
                           "document={'baseline_modified':False,'experimental':True,'segments':rows,\n"
                           '        '
                           "'provenance':{'video':str(a.video.resolve()),'video_sha256':sha(a.video),\n"
                           '            '
                           "'window_start':a.start,'decoded_duration':len(audio)/16000,\n"
                           '            '
                           "'models':{'asr':a.model,'voice':'speechbrain/spkrec-ecapa-voxceleb'},\n"
                           '            '
                           "'device':device,'requested_timing_source':a.timing_source,'asr_engine':a.asr_engine,'vad':'disabled' "
                           "if a.asr_engine=='native' else "
                           "'bounded_silero','chunk_seconds':a.chunk_seconds if "
                           "a.asr_engine=='bounded-whisperx' else None,\n"
                           "            'context_seconds':a.context_seconds if "
                           "a.asr_engine=='bounded-whisperx' else "
                           "None,'references':{k:{'path':str(v.resolve()),'sha256':sha(v)} for k,v "
                           'in references.items()},\n'
                           "            'baseline':str(a.baseline.resolve()) if a.baseline else "
                           'None,\n'
                           "            'baseline_sha256':sha(a.baseline) if a.baseline else "
                           "None,'elapsed_seconds':time.monotonic()-started}}\n"
                           '    '
                           "(a.output_dir/'review_hypotheses.json').write_text(json.dumps(document,indent=2)+'\\n')\n"
                           '    '
                           '(a.output_dir/\'review_transcript.txt\').write_text(\'\\n\'.join(f"[{r[\'start\']:.2f}–{r[\'end\']:.2f}] '
                           "{r['speaker_hypothesis']}{' [decoder warning]' if "
                           "r['transcription_status']=='decoder_warning' else ''}: "
                           '{r[\'text\']}" for r in rows)+\'\\n\')\n'
                           "    print(f'Wrote {len(rows)} review hypotheses to {a.output_dir}; "
                           "baseline preserved.')\n"
                           '\n'
                           "if __name__=='__main__':main()\n",
 'review_overlap_extraction.py': '"""Supplemental target-speaker extraction for baseline overlap '
                                 'intervals.\n'
                                 '\n'
                                 'The baseline file is read-only. Results are review candidates '
                                 'and never replace\n'
                                 'baseline text, timing, evidence, confidence, or speaker '
                                 'identity.\n'
                                 '"""\n'
                                 '\n'
                                 'import argparse\n'
                                 'from collections import Counter\n'
                                 'from difflib import SequenceMatcher\n'
                                 'import json\n'
                                 'from pathlib import Path\n'
                                 'import re\n'
                                 'import subprocess\n'
                                 '\n'
                                 'import numpy as np\n'
                                 '\n'
                                 '\n'
                                 'def select_overlap_segments(baseline):\n'
                                 '    selected = []\n'
                                 '    for index, segment in enumerate(baseline.get("segments", '
                                 '[])):\n'
                                 '        overlap = next(\n'
                                 '            (\n'
                                 '                item\n'
                                 '                for item in segment.get("evidence", [])\n'
                                 '                if item.get("source") == "overlapping_speakers"\n'
                                 '                and item.get("details", '
                                 '{}).get("target_and_non_target", False)\n'
                                 '            ),\n'
                                 '            None,\n'
                                 '        )\n'
                                 '        if overlap is not None:\n'
                                 '            selected.append((index, segment, overlap))\n'
                                 '    return selected\n'
                                 '\n'
                                 '\n'
                                 'def classify_extraction(original_similarity, '
                                 'extracted_similarity, energy_retention,\n'
                                 '                        transcript):\n'
                                 '    """Triage only; every result remains review-required."""\n'
                                 '    similarity_gain = extracted_similarity - '
                                 'original_similarity\n'
                                 '    if energy_retention < 0.10:\n'
                                 '        return "likely_suppressed_residual"\n'
                                 '    if transcript.strip() and energy_retention >= 0.10 and '
                                 'similarity_gain >= 0.10:\n'
                                 '        return "candidate_target_speech"\n'
                                 '    return "unresolved"\n'
                                 '\n'
                                 '\n'
                                 'def unit(vector):\n'
                                 '    vector = np.asarray(vector, dtype=np.float32).reshape(-1)\n'
                                 '    return vector / max(float(np.linalg.norm(vector)), 1e-9)\n'
                                 '\n'
                                 '\n'
                                 'def rms(wave):\n'
                                 '    return float(np.sqrt(np.mean(np.asarray(wave, '
                                 'dtype=np.float32) ** 2)))\n'
                                 '\n'
                                 '\n'
                                 'def stereo_metrics(wave):\n'
                                 '    """Measure whether stereo contains information beyond '
                                 'duplicated mono."""\n'
                                 '    wave = np.asarray(wave, dtype=np.float32)\n'
                                 '    if wave.ndim != 2 or wave.shape[1] != 2 or len(wave) < 2:\n'
                                 '        return {"available": False, "distinct": False}\n'
                                 '    left, right = wave[:, 0], wave[:, 1]\n'
                                 '    middle, side = (left + right) / 2, (left - right) / 2\n'
                                 '    correlation = float(np.corrcoef(left, right)[0, 1])\n'
                                 '    side_to_middle_db = float(20 * np.log10(\n'
                                 '        (rms(side) + 1e-12) / (rms(middle) + 1e-12)\n'
                                 '    ))\n'
                                 '    # Lossy encoders can make duplicated channels differ by tiny '
                                 'amounts. Analyze\n'
                                 '    # channels only when the difference is large enough to carry '
                                 'real content.\n'
                                 '    distinct = bool(np.isfinite(correlation) and correlation < '
                                 '0.98\n'
                                 '                    and side_to_middle_db >= -25.0)\n'
                                 '    return {\n'
                                 '        "available": True,\n'
                                 '        "distinct": distinct,\n'
                                 '        "correlation": correlation,\n'
                                 '        "side_to_middle_db": side_to_middle_db,\n'
                                 '        "left_to_right_level_db": float(20 * np.log10(\n'
                                 '            (rms(left) + 1e-12) / (rms(right) + 1e-12)\n'
                                 '        )),\n'
                                 '    }\n'
                                 '\n'
                                 '\n'
                                 'def stereo_signals(wave):\n'
                                 '    wave = np.asarray(wave, dtype=np.float32)\n'
                                 '    left, right = wave[:, 0], wave[:, 1]\n'
                                 '    return {\n'
                                 '        "left": left,\n'
                                 '        "right": right,\n'
                                 '        "middle": (left + right) / 2,\n'
                                 '        "difference": (left - right) / 2,\n'
                                 '    }\n'
                                 '\n'
                                 '\n'
                                 'def corroborated_novel_words(transcriptions, baseline_text, '
                                 'minimum_views=2):\n'
                                 '    """Return inserted words decoded in independent views.\n'
                                 '\n'
                                 '    Replacement hypotheses such as sue/see or city/scene are '
                                 'transcription\n'
                                 '    disagreements, not recovered concurrent speech, and are '
                                 'intentionally\n'
                                 '    excluded here.\n'
                                 '    """\n'
                                 '    tokens = lambda text: re.findall(r"[a-z0-9\']+", '
                                 'str(text).casefold())\n'
                                 '    baseline_words = tokens(baseline_text)\n'
                                 '    support = Counter()\n'
                                 '    views = {}\n'
                                 '    for name, text in transcriptions.items():\n'
                                 '        candidate_words = tokens(text)\n'
                                 '        inserted = set()\n'
                                 '        for tag, _, _, candidate_start, candidate_end in '
                                 'SequenceMatcher(\n'
                                 '                None, baseline_words, '
                                 'candidate_words).get_opcodes():\n'
                                 '            if tag == "insert":\n'
                                 '                '
                                 'inserted.update(candidate_words[candidate_start:candidate_end])\n'
                                 '        for word in inserted:\n'
                                 '            support[word] += 1\n'
                                 '            views.setdefault(word, []).append(name)\n'
                                 '    return [\n'
                                 '        {"word": word, "support": support[word], "views": '
                                 'sorted(views[word])}\n'
                                 '        for word in sorted(support)\n'
                                 '        if support[word] >= minimum_views\n'
                                 '    ]\n'
                                 '\n'
                                 '\n'
                                 'def transcribe(model, wave):\n'
                                 '    segments, _ = model.transcribe(\n'
                                 '        wave,\n'
                                 '        vad_filter=False,\n'
                                 '        condition_on_previous_text=False,\n'
                                 '        beam_size=5,\n'
                                 '    )\n'
                                 '    rows = list(segments)\n'
                                 '    return {\n'
                                 '        "text": " ".join(row.text.strip() for row in rows if '
                                 'row.text.strip()),\n'
                                 '        "average_log_probability": (\n'
                                 '            float(np.mean([row.avg_logprob for row in rows])) if '
                                 'rows else None\n'
                                 '        ),\n'
                                 '        "maximum_no_speech_probability": (\n'
                                 '            float(max(row.no_speech_prob for row in rows)) if '
                                 'rows else None\n'
                                 '        ),\n'
                                 '    }\n'
                                 '\n'
                                 '\n'
                                 'def main():\n'
                                 '    parser = argparse.ArgumentParser()\n'
                                 '    parser.add_argument("--video", type=Path, required=True)\n'
                                 '    parser.add_argument("--baseline", type=Path, required=True)\n'
                                 '    parser.add_argument("--enrollment", type=Path, '
                                 'required=True)\n'
                                 '    parser.add_argument("--voice-priors", type=Path, '
                                 'required=True)\n'
                                 '    parser.add_argument("--output-dir", type=Path, '
                                 'required=True)\n'
                                 '    parser.add_argument("--device", default="cuda")\n'
                                 '    parser.add_argument("--whisper-model", default="large-v2")\n'
                                 '    parser.add_argument("--wesep-model-dir", type=Path)\n'
                                 '    parser.add_argument("--maximum-segments", type=int)\n'
                                 '    args = parser.parse_args()\n'
                                 '\n'
                                 '    import soundfile as sf\n'
                                 '    import torch\n'
                                 '    import torchaudio\n'
                                 '    import wesep\n'
                                 '    from faster_whisper import WhisperModel\n'
                                 '    from speechbrain.inference.speaker import '
                                 'SpeakerRecognition\n'
                                 '\n'
                                 '    args.output_dir.mkdir(parents=True, exist_ok=True)\n'
                                 '    baseline_bytes = args.baseline.read_bytes()\n'
                                 '    baseline = json.loads(baseline_bytes)\n'
                                 '    selected = select_overlap_segments(baseline)\n'
                                 '    if args.maximum_segments is not None:\n'
                                 '        selected = selected[:args.maximum_segments]\n'
                                 '\n'
                                 '    source_stereo = args.output_dir / "source-stereo-16khz.wav"\n'
                                 '    subprocess.run(\n'
                                 '        [\n'
                                 '            "ffmpeg", "-nostdin", "-hide_banner", "-loglevel", '
                                 '"error", "-y",\n'
                                 '            "-i", str(args.video), "-vn", "-ac", "2", "-ar", '
                                 '"16000",\n'
                                 '            str(source_stereo),\n'
                                 '        ],\n'
                                 '        check=True,\n'
                                 '    )\n'
                                 '    full_stereo, sample_rate = sf.read(\n'
                                 '        source_stereo, dtype="float32", always_2d=True\n'
                                 '    )\n'
                                 '    if sample_rate != 16000:\n'
                                 '        raise RuntimeError(f"Unexpected extracted sample rate: '
                                 '{sample_rate}")\n'
                                 '    full_wave = full_stereo.mean(axis=1)\n'
                                 '    source_audio = args.output_dir / "source-16khz.wav"\n'
                                 '    sf.write(source_audio, full_wave, sample_rate)\n'
                                 '\n'
                                 '    extractor = (\n'
                                 '        wesep.load_model_local(str(args.wesep_model_dir))\n'
                                 '        if args.wesep_model_dir\n'
                                 '        else wesep.load_model("english")\n'
                                 '    )\n'
                                 '    extractor.set_device(args.device)\n'
                                 '    extractor.set_vad(True)\n'
                                 '    # Preserve attenuation so residual noise can be rejected.\n'
                                 '    extractor.set_output_norm(False)\n'
                                 '\n'
                                 '    target_rows = np.load(args.voice_priors)\n'
                                 '    target = unit(np.mean(np.stack([unit(row) for row in '
                                 'target_rows]), axis=0))\n'
                                 '    speaker_dir = '
                                 'Path("pretrained_models/spkrec-ecapa-voxceleb")\n'
                                 '    speaker = SpeakerRecognition.from_hparams(\n'
                                 '        source=str(speaker_dir),\n'
                                 '        savedir=str(speaker_dir),\n'
                                 '        run_opts={"device": args.device},\n'
                                 '    )\n'
                                 '    whisper_device = "cuda" if args.device.startswith("cuda") '
                                 'else "cpu"\n'
                                 '    whisper = WhisperModel(\n'
                                 '        args.whisper_model,\n'
                                 '        device=whisper_device,\n'
                                 '        compute_type="float16" if whisper_device == "cuda" else '
                                 '"int8",\n'
                                 '    )\n'
                                 '\n'
                                 '    extracted_dir = args.output_dir / "audio"\n'
                                 '    extracted_dir.mkdir(exist_ok=True)\n'
                                 '    results = []\n'
                                 '    for number, (baseline_index, segment, overlap) in '
                                 'enumerate(selected, 1):\n'
                                 '        start, end = float(segment["start"]), '
                                 'float(segment["end"])\n'
                                 '        left, right = max(0, round(start * sample_rate)), min(\n'
                                 '            len(full_wave), round(end * sample_rate)\n'
                                 '        )\n'
                                 '        original = full_wave[left:right]\n'
                                 '        original_stereo = full_stereo[left:right]\n'
                                 '        if len(original) < 1:\n'
                                 '            continue\n'
                                 '        original_path = args.output_dir / '
                                 'f"current-{baseline_index:04d}.wav"\n'
                                 '        sf.write(original_path, original, sample_rate)\n'
                                 '        extracted_tensor = extractor.extract_speech(\n'
                                 '            str(original_path), str(args.enrollment)\n'
                                 '        )\n'
                                 '        if extracted_tensor is None:\n'
                                 '            results.append({\n'
                                 '                "baseline_index": baseline_index,\n'
                                 '                "start": start,\n'
                                 '                "end": end,\n'
                                 '                "baseline_text": segment.get("text", ""),\n'
                                 '                "status": "extractor_returned_no_speech",\n'
                                 '                "review_required": True,\n'
                                 '            })\n'
                                 '            continue\n'
                                 '        extracted = extracted_tensor[0].detach().cpu().numpy()\n'
                                 '        extracted_path = extracted_dir / '
                                 'f"overlap-{baseline_index:04d}-target.wav"\n'
                                 '        sf.write(extracted_path, extracted, sample_rate)\n'
                                 '\n'
                                 '        def similarity(wave):\n'
                                 '            tensor = torch.from_numpy(np.asarray(wave, '
                                 'dtype=np.float32)).unsqueeze(0)\n'
                                 '            embedding = unit(\n'
                                 '                '
                                 'speaker.encode_batch(tensor).flatten().detach().cpu().numpy()\n'
                                 '            )\n'
                                 '            return float(np.dot(target, embedding))\n'
                                 '\n'
                                 '        original_similarity = similarity(original)\n'
                                 '        extracted_similarity = similarity(extracted)\n'
                                 '        retention = rms(extracted) / max(rms(original), 1e-9)\n'
                                 '        original_asr = transcribe(whisper, original)\n'
                                 '        extracted_asr = transcribe(whisper, extracted)\n'
                                 '        status = classify_extraction(\n'
                                 '            original_similarity,\n'
                                 '            extracted_similarity,\n'
                                 '            retention,\n'
                                 '            extracted_asr["text"],\n'
                                 '        )\n'
                                 '        channel_metrics = stereo_metrics(original_stereo)\n'
                                 '        stereo_review = {\n'
                                 '            "analyzed": False,\n'
                                 '            "metrics": channel_metrics,\n'
                                 '            "status": "channels_not_distinct",\n'
                                 '            "review_required": True,\n'
                                 '        }\n'
                                 '        if channel_metrics.get("distinct", False):\n'
                                 '            channel_audio = stereo_signals(original_stereo)\n'
                                 '            channel_results = {}\n'
                                 '            for name, wave in channel_audio.items():\n'
                                 '                channel_results[name] = {\n'
                                 '                    "target_similarity": similarity(wave),\n'
                                 '                    "rms": rms(wave),\n'
                                 '                    "transcription": transcribe(whisper, wave),\n'
                                 '                }\n'
                                 '            transcriptions = {\n'
                                 '                name: value["transcription"]["text"]\n'
                                 '                for name, value in channel_results.items()\n'
                                 '            }\n'
                                 '            novel = corroborated_novel_words(\n'
                                 '                transcriptions, segment.get("text", "")\n'
                                 '            )\n'
                                 '            stereo_review = {\n'
                                 '                "analyzed": True,\n'
                                 '                "metrics": channel_metrics,\n'
                                 '                "signals": channel_results,\n'
                                 '                "corroborated_novel_words": novel,\n'
                                 '                "status": ("corroborated_words_for_review" if '
                                 'novel\n'
                                 '                           else '
                                 '"distinct_channels_no_corroborated_new_words"),\n'
                                 '                "speaker": "Uncertain",\n'
                                 '                "review_required": True,\n'
                                 '                "note": (\n'
                                 '                    "Channel decoding is supplemental evidence. '
                                 'Words require "\n'
                                 '                    "speaker review and are never inserted into '
                                 'the baseline."\n'
                                 '                ),\n'
                                 '            }\n'
                                 '        results.append({\n'
                                 '            "baseline_index": baseline_index,\n'
                                 '            "start": start,\n'
                                 '            "end": end,\n'
                                 '            "baseline_text": segment.get("text", ""),\n'
                                 '            "baseline_speaker": segment.get("final_speaker"),\n'
                                 '            "baseline_confidence": '
                                 'segment.get("final_confidence"),\n'
                                 '            "overlap": overlap.get("details", {}),\n'
                                 '            "original": {\n'
                                 '                "target_similarity": original_similarity,\n'
                                 '                "rms": rms(original),\n'
                                 '                "transcription": original_asr,\n'
                                 '            },\n'
                                 '            "extracted": {\n'
                                 '                "target_similarity": extracted_similarity,\n'
                                 '                "similarity_gain": extracted_similarity - '
                                 'original_similarity,\n'
                                 '                "rms": rms(extracted),\n'
                                 '                "energy_retention": retention,\n'
                                 '                "transcription": extracted_asr,\n'
                                 '                "audio": '
                                 'str(extracted_path.relative_to(args.output_dir)),\n'
                                 '            },\n'
                                 '            "stereo": stereo_review,\n'
                                 '            "status": status,\n'
                                 '            "review_required": True,\n'
                                 '            "baseline_modified": False,\n'
                                 '        })\n'
                                 '        print(\n'
                                 '            f"Overlap {number}/{len(selected)} at {start:.2f}s: '
                                 '{status}; "\n'
                                 '            f"retention={retention:.3f}; '
                                 'similarity={original_similarity:.3f}"\n'
                                 '            f"->{extracted_similarity:.3f}; '
                                 '{extracted_asr[\'text\']}",\n'
                                 '            flush=True,\n'
                                 '        )\n'
                                 '\n'
                                 '    counts = {}\n'
                                 '    stereo_counts = {}\n'
                                 '    for row in results:\n'
                                 '        counts[row["status"]] = counts.get(row["status"], 0) + '
                                 '1\n'
                                 '        stereo_status = row.get("stereo", {}).get("status", '
                                 '"not_available")\n'
                                 '        stereo_counts[stereo_status] = '
                                 'stereo_counts.get(stereo_status, 0) + 1\n'
                                 '    report = {\n'
                                 '        "review_required": True,\n'
                                 '        "baseline_modified": False,\n'
                                 '        "selection": "target/non-target diarization overlap '
                                 'evidence",\n'
                                 '        "thresholds_are_provisional": True,\n'
                                 '        "triage_thresholds": {\n'
                                 '            "suppressed_below_energy_retention": 0.10,\n'
                                 '            "candidate_minimum_energy_retention": 0.10,\n'
                                 '            "candidate_minimum_similarity_gain": 0.10,\n'
                                 '            "stereo_maximum_channel_correlation": 0.98,\n'
                                 '            "stereo_minimum_side_to_middle_db": -25.0,\n'
                                 '            "stereo_novel_word_minimum_views": 2,\n'
                                 '        },\n'
                                 '        "summary": {"selected": len(selected), "completed": '
                                 'len(results),\n'
                                 '                    "status": counts, "stereo_status": '
                                 'stereo_counts},\n'
                                 '        "segments": results,\n'
                                 '    }\n'
                                 '    (args.output_dir / '
                                 '"report.json").write_text(json.dumps(report, indent=2) + "\\n")\n'
                                 '    lines = [\n'
                                 '        f"[{row[\'start\']:.2f}-{row[\'end\']:.2f}] '
                                 '{row[\'status\']}: "\n'
                                 '        f"{row.get(\'extracted\', {}).get(\'transcription\', '
                                 '{}).get(\'text\', \'\')}; "\n'
                                 '        f"stereo={row.get(\'stereo\', {}).get(\'status\', '
                                 '\'not_available\')}; "\n'
                                 '        f"novel={\',\'.join(item[\'word\'] for item in '
                                 "row.get('stereo', {}).get('corroborated_novel_words', "
                                 '[]))}"\n'
                                 '        for row in results\n'
                                 '    ]\n'
                                 '    (args.output_dir / '
                                 '"review.txt").write_text("\\n".join(lines) + "\\n")\n'
                                 '    if args.baseline.read_bytes() != baseline_bytes:\n'
                                 '        raise RuntimeError("Baseline changed during supplemental '
                                 'extraction review")\n'
                                 '\n'
                                 '\n'
                                 'if __name__ == "__main__":\n'
                                 '    main()\n',
 'review_transcript_regions.py': '"""Review doubtful transcript regions without modifying the '
                                 'baseline.\n'
                                 '\n'
                                 'This is a batch companion to review_audio_window.py.  It loads '
                                 'each large\n'
                                 'model once, checkpoints transcription/alignment per region, and '
                                 'writes only\n'
                                 'supplemental hypotheses.  Selection uses '
                                 'confidence/duration/gaps, never\n'
                                 'expected dialogue or speaker-specific video rules.\n'
                                 '"""\n'
                                 'from __future__ import annotations\n'
                                 '\n'
                                 'import argparse\n'
                                 'import gc\n'
                                 'import hashlib\n'
                                 'import json\n'
                                 'import math\n'
                                 'from pathlib import Path\n'
                                 'import subprocess\n'
                                 'import time\n'
                                 '\n'
                                 '\n'
                                 'def sha256(path: Path) -> str:\n'
                                 '    digest = hashlib.sha256()\n'
                                 '    with path.open("rb") as source:\n'
                                 '        for block in iter(lambda: source.read(1024 * 1024), '
                                 'b""):\n'
                                 '            digest.update(block)\n'
                                 '    return digest.hexdigest()\n'
                                 '\n'
                                 '\n'
                                 'def media_duration(path: Path) -> float:\n'
                                 '    value = subprocess.check_output([\n'
                                 '        "ffprobe", "-v", "error", "-show_entries", '
                                 '"format=duration",\n'
                                 '        "-of", "default=noprint_wrappers=1:nokey=1", str(path)\n'
                                 '    ], text=True).strip()\n'
                                 '    duration = float(value)\n'
                                 '    if not math.isfinite(duration) or duration <= 0:\n'
                                 '        raise ValueError("Invalid media duration")\n'
                                 '    return duration\n'
                                 '\n'
                                 '\n'
                                 'def select_review_regions(segments, duration, confidence=0.35,\n'
                                 '                          short_seconds=1.0, minimum_gap=5.0,\n'
                                 '                          context=3.0, merge_gap=2.0,\n'
                                 '                          maximum_window=30.0, overlap=4.0,\n'
                                 '                          extra_regions=()):\n'
                                 '    """Select and bound review windows. Extra regions are '
                                 'external controls."""\n'
                                 '    if not (0 <= confidence <= 1 and short_seconds >= 0 and '
                                 'minimum_gap >= 0\n'
                                 '            and context >= 0 and merge_gap >= 0 and '
                                 'maximum_window > 0\n'
                                 '            and 0 <= overlap < maximum_window and duration > '
                                 '0):\n'
                                 '        raise ValueError("Invalid region selection settings")\n'
                                 '    ordered = sorted(segments, key=lambda row: '
                                 '(float(row["start"]), float(row["end"])))\n'
                                 '    candidates = []\n'
                                 '    for index, row in enumerate(ordered):\n'
                                 '        start, end = float(row["start"]), float(row["end"])\n'
                                 '        if not (0 <= start <= end <= duration + 0.5):\n'
                                 '            raise ValueError("Baseline contains invalid segment '
                                 'times")\n'
                                 '        speaker = row.get("final_speaker", row.get("speaker", '
                                 '"Uncertain"))\n'
                                 '        strength = float(row.get("final_confidence", 0.0) or '
                                 '0.0)\n'
                                 '        reasons = []\n'
                                 '        if speaker in ("Uncertain", "Unknown", '
                                 '"Unknown_Speaker", None):\n'
                                 '            reasons.append("uncertain_speaker")\n'
                                 '        if strength < confidence:\n'
                                 '            reasons.append("weak_identity_evidence")\n'
                                 '        if end - start <= short_seconds:\n'
                                 '            reasons.append("short_utterance")\n'
                                 '        if reasons:\n'
                                 '            candidates.append({"start": max(0, start-context),\n'
                                 '                               "end": min(duration, '
                                 'end+context),\n'
                                 '                               "reasons": reasons,\n'
                                 '                               "baseline_indices": [index]})\n'
                                 '    previous = 0.0\n'
                                 '    for index, row in enumerate(ordered):\n'
                                 '        start = float(row["start"])\n'
                                 '        if start - previous >= minimum_gap:\n'
                                 '            candidates.append({"start": max(0, '
                                 'previous-context),\n'
                                 '                               "end": min(duration, '
                                 'start+context),\n'
                                 '                               "reasons": ["transcript_gap"],\n'
                                 '                               "baseline_indices": []})\n'
                                 '        previous = max(previous, float(row["end"]))\n'
                                 '    if duration - previous >= minimum_gap:\n'
                                 '        candidates.append({"start": max(0, previous-context), '
                                 '"end": duration,\n'
                                 '                           "reasons": ["transcript_gap"], '
                                 '"baseline_indices": []})\n'
                                 '    for start, end in extra_regions:\n'
                                 '        if not (0 <= start < end <= duration):\n'
                                 '            raise ValueError("Extra review region is outside the '
                                 'video")\n'
                                 '        candidates.append({"start": start, "end": end,\n'
                                 '                           "reasons": '
                                 '["external_review_control"],\n'
                                 '                           "baseline_indices": []})\n'
                                 '    candidates.sort(key=lambda row: (row["start"], row["end"]))\n'
                                 '    merged = []\n'
                                 '    for item in candidates:\n'
                                 '        if merged and item["start"] <= merged[-1]["end"] + '
                                 'merge_gap:\n'
                                 '            merged[-1]["end"] = max(merged[-1]["end"], '
                                 'item["end"])\n'
                                 '            merged[-1]["reasons"] = '
                                 'sorted(set(merged[-1]["reasons"] + item["reasons"]))\n'
                                 '            merged[-1]["baseline_indices"] = '
                                 'sorted(set(merged[-1]["baseline_indices"] + '
                                 'item["baseline_indices"]))\n'
                                 '        else:\n'
                                 '            merged.append(dict(item))\n'
                                 '    windows = []\n'
                                 '    for item in merged:\n'
                                 '        left = item["start"]\n'
                                 '        while left < item["end"] - 1e-6:\n'
                                 '            right = min(left + maximum_window, item["end"])\n'
                                 '            windows.append({"index": len(windows), "start": '
                                 'left, "end": right,\n'
                                 '                            "reasons": item["reasons"],\n'
                                 '                            "baseline_indices": '
                                 'item["baseline_indices"]})\n'
                                 '            if right >= item["end"]:\n'
                                 '                break\n'
                                 '            left = right - overlap\n'
                                 '    return windows\n'
                                 '\n'
                                 '\n'
                                 'def parse_region(value):\n'
                                 '    try:\n'
                                 '        start, end = (float(part) for part in value.split(":", '
                                 '1))\n'
                                 '    except Exception as error:\n'
                                 '        raise argparse.ArgumentTypeError("Region must be '
                                 'START:END") from error\n'
                                 '    if not (math.isfinite(start) and math.isfinite(end) and 0 <= '
                                 'start < end):\n'
                                 '        raise argparse.ArgumentTypeError("Region must be finite '
                                 'and increasing")\n'
                                 '    return start, end\n'
                                 '\n'
                                 '\n'
                                 'def main():\n'
                                 '    parser = argparse.ArgumentParser(description=__doc__)\n'
                                 '    parser.add_argument("--video", type=Path, required=True)\n'
                                 '    parser.add_argument("--baseline", type=Path, required=True)\n'
                                 '    parser.add_argument("--target-reference", type=Path, '
                                 'required=True)\n'
                                 '    parser.add_argument("--other-reference", action="append", '
                                 'default=[], metavar="NAME=PATH")\n'
                                 '    parser.add_argument("--output-dir", type=Path, '
                                 'required=True)\n'
                                 '    parser.add_argument("--cache-dir", type=Path, '
                                 'required=True)\n'
                                 '    parser.add_argument("--extra-region", action="append", '
                                 'type=parse_region, default=[])\n'
                                 '    parser.add_argument("--weak-confidence", type=float, '
                                 'default=0.35)\n'
                                 '    parser.add_argument("--short-seconds", type=float, '
                                 'default=1.0)\n'
                                 '    parser.add_argument("--minimum-gap", type=float, '
                                 'default=5.0)\n'
                                 '    parser.add_argument("--context-seconds", type=float, '
                                 'default=3.0)\n'
                                 '    parser.add_argument("--maximum-window-seconds", type=float, '
                                 'default=30.0)\n'
                                 '    parser.add_argument("--window-overlap-seconds", type=float, '
                                 'default=4.0)\n'
                                 '    parser.add_argument("--model", default="large-v2")\n'
                                 '    parser.add_argument("--language", default="en")\n'
                                 '    parser.add_argument("--device", choices=("auto", "cpu", '
                                 '"cuda"), default="auto")\n'
                                 '    parser.add_argument("--threads", type=int, default=4)\n'
                                 '    parser.add_argument("--min-similarity", type=float, '
                                 'default=0.25)\n'
                                 '    parser.add_argument("--min-margin", type=float, '
                                 'default=0.08)\n'
                                 '    parser.add_argument("--speechbrain-cache", type=Path)\n'
                                 '    args = parser.parse_args()\n'
                                 '    input_paths = [args.video, args.baseline, '
                                 'args.target_reference]\n'
                                 '    references = {"Target_Speaker": args.target_reference}\n'
                                 '    for item in args.other_reference:\n'
                                 '        if "=" not in item:\n'
                                 '            parser.error("Other reference must be NAME=PATH")\n'
                                 '        name, path = item.split("=", 1)\n'
                                 '        if not name or name in references or name in ("Unknown", '
                                 '"Uncertain"):\n'
                                 '            parser.error("Reference names must be unique")\n'
                                 '        references[name] = Path(path)\n'
                                 '        input_paths.append(Path(path))\n'
                                 '    input_paths.append(args.baseline)\n'
                                 '    for path in input_paths:\n'
                                 '        if not path.is_file():\n'
                                 '            parser.error(f"Missing input: {path}")\n'
                                 '    baseline_bytes = args.baseline.read_bytes()\n'
                                 '    baseline_hash = hashlib.sha256(baseline_bytes).hexdigest()\n'
                                 '    baseline = json.loads(baseline_bytes)\n'
                                 '    if not isinstance(baseline.get("segments"), list):\n'
                                 '        parser.error("Baseline must contain a segments list")\n'
                                 '    duration = media_duration(args.video)\n'
                                 '    windows = select_review_regions(\n'
                                 '        baseline["segments"], duration, args.weak_confidence,\n'
                                 '        args.short_seconds, args.minimum_gap, '
                                 'args.context_seconds, 2.0,\n'
                                 '        args.maximum_window_seconds, '
                                 'args.window_overlap_seconds,\n'
                                 '        args.extra_region)\n'
                                 '    configuration = {\n'
                                 '        "video_sha256": sha256(args.video), "baseline_sha256": '
                                 'baseline_hash,\n'
                                 '        "references": {name: sha256(path) for name, path in '
                                 'references.items()},\n'
                                 '        "model": args.model, "language": args.language,\n'
                                 '        "weak_confidence": args.weak_confidence, '
                                 '"short_seconds": args.short_seconds,\n'
                                 '        "minimum_gap": args.minimum_gap, "context_seconds": '
                                 'args.context_seconds,\n'
                                 '        "maximum_window_seconds": args.maximum_window_seconds,\n'
                                 '        "window_overlap_seconds": args.window_overlap_seconds,\n'
                                 '        "extra_regions": args.extra_region, "windows": windows}\n'
                                 '    fingerprint = hashlib.sha256(json.dumps(configuration, '
                                 'sort_keys=True).encode()).hexdigest()\n'
                                 '    args.output_dir.mkdir(parents=True, exist_ok=True)\n'
                                 '    cache = args.cache_dir / fingerprint\n'
                                 '    cache.mkdir(parents=True, exist_ok=True)\n'
                                 '    (args.output_dir / '
                                 '"selection.json").write_text(json.dumps(configuration, '
                                 'indent=2)+"\\n")\n'
                                 '    print(f"Selected {len(windows)} windows, '
                                 "{sum(w['end']-w['start'] for w in windows)/60:.1f} decoded "
                                 'minutes")\n'
                                 '\n'
                                 '    import numpy as np\n'
                                 '    import torch\n'
                                 '    torch.set_num_threads(args.threads)\n'
                                 '    device = ("cuda" if torch.cuda.is_available() else "cpu") if '
                                 'args.device == "auto" else args.device\n'
                                 '    if device == "cuda" and not torch.cuda.is_available():\n'
                                 '        parser.error("CUDA was requested but is unavailable")\n'
                                 '    raw = subprocess.check_output(["ffmpeg", "-nostdin", '
                                 '"-hide_banner", "-loglevel", "error",\n'
                                 '        "-i", str(args.video), "-vn", "-ar", "16000", "-ac", '
                                 '"1", "-f", "f32le", "pipe:1"])\n'
                                 '    audio = np.frombuffer(raw, dtype="<f4").copy()\n'
                                 '    if not len(audio) or not np.isfinite(audio).all():\n'
                                 '        raise ValueError("Decoded audio is empty or invalid")\n'
                                 '\n'
                                 '    asr_records = {}\n'
                                 '    missing = []\n'
                                 '    for window in windows:\n'
                                 '        path = cache / f"asr-{window[\'index\']:04d}.json"\n'
                                 '        if path.exists():\n'
                                 '            asr_records[window["index"]] = '
                                 'json.loads(path.read_text())\n'
                                 '        else:\n'
                                 '            missing.append(window)\n'
                                 '    if missing:\n'
                                 '        from faster_whisper import WhisperModel\n'
                                 '        model = WhisperModel(args.model, device=device,\n'
                                 '            compute_type="float16" if device == "cuda" else '
                                 '"int8", cpu_threads=args.threads)\n'
                                 '        for count, window in enumerate(missing, 1):\n'
                                 '            left, right = window["start"], window["end"]\n'
                                 '            decoded, _ = '
                                 'model.transcribe(audio[round(left*16000):round(right*16000)],\n'
                                 '                language=args.language, beam_size=5, '
                                 'vad_filter=False,\n'
                                 '                condition_on_previous_text=False, '
                                 'word_timestamps=True)\n'
                                 '            rows = []\n'
                                 '            for segment in decoded:\n'
                                 '                rows.append({"start": float(segment.start), '
                                 '"end": float(segment.end),\n'
                                 '                    "text": segment.text, "avg_logprob": '
                                 'float(segment.avg_logprob),\n'
                                 '                    "no_speech_prob": '
                                 'float(segment.no_speech_prob),\n'
                                 '                    "compression_ratio": '
                                 'float(segment.compression_ratio),\n'
                                 '                    "words": [{"start": float(word.start), '
                                 '"end": float(word.end),\n'
                                 '                               "word": word.word, "probability": '
                                 'float(word.probability)}\n'
                                 '                              for word in segment.words or '
                                 '[]]})\n'
                                 '            record = {"window": window, "segments": rows, '
                                 '"language": args.language}\n'
                                 '            path = cache / f"asr-{window[\'index\']:04d}.json"\n'
                                 '            path.write_text(json.dumps(record, indent=2)+"\\n")\n'
                                 '            asr_records[window["index"]] = record\n'
                                 '            print(f"Transcribed review window '
                                 '{count}/{len(missing)}", flush=True)\n'
                                 '        del model\n'
                                 '        gc.collect()\n'
                                 '        if device == "cuda": torch.cuda.empty_cache()\n'
                                 '\n'
                                 '    import whisperx\n'
                                 '    aligned_records = {}\n'
                                 '    missing = []\n'
                                 '    for window in windows:\n'
                                 '        path = cache / '
                                 'f"alignment-{window[\'index\']:04d}.json"\n'
                                 '        if path.exists():\n'
                                 '            aligned_records[window["index"]] = '
                                 'json.loads(path.read_text())\n'
                                 '        else:\n'
                                 '            missing.append(window)\n'
                                 '    if missing:\n'
                                 '        aligner, metadata = '
                                 'whisperx.load_align_model(language_code=args.language, '
                                 'device=device)\n'
                                 '        for count, window in enumerate(missing, 1):\n'
                                 '            record = asr_records[window["index"]]\n'
                                 '            left, right = window["start"], window["end"]\n'
                                 '            if record["segments"]:\n'
                                 '                aligned = whisperx.align(record["segments"], '
                                 'aligner, metadata,\n'
                                 '                    audio[round(left*16000):round(right*16000)], '
                                 'device,\n'
                                 '                    return_char_alignments=False)\n'
                                 '            else:\n'
                                 '                aligned = {"segments": [], "word_segments": []}\n'
                                 '            output = {"window": window, **aligned}\n'
                                 '            path = cache / '
                                 'f"alignment-{window[\'index\']:04d}.json"\n'
                                 '            path.write_text(json.dumps(output, indent=2)+"\\n")\n'
                                 '            aligned_records[window["index"]] = output\n'
                                 '            print(f"Aligned review window '
                                 '{count}/{len(missing)}", flush=True)\n'
                                 '        del aligner\n'
                                 '        gc.collect()\n'
                                 '        if device == "cuda": torch.cuda.empty_cache()\n'
                                 '\n'
                                 '    from speechbrain.inference.speaker import '
                                 'SpeakerRecognition\n'
                                 '    from review_audio_window import (baseline_evidence, '
                                 'decoder_quality_flags,\n'
                                 '        decoder_sentence_bounds, resolve_timing_evidence)\n'
                                 '    options = {"source": "speechbrain/spkrec-ecapa-voxceleb", '
                                 '"run_opts": {"device": device}}\n'
                                 '    if args.speechbrain_cache:\n'
                                 '        options["savedir"] = str(args.speechbrain_cache)\n'
                                 '    encoder = SpeakerRecognition.from_hparams(**options)\n'
                                 '\n'
                                 '    def unit(value):\n'
                                 '        value = np.asarray(value, dtype=np.float32).reshape(-1)\n'
                                 '        norm = np.linalg.norm(value)\n'
                                 '        if not np.isfinite(value).all() or norm < 1e-8:\n'
                                 '            raise ValueError("Invalid voice embedding")\n'
                                 '        return value / norm\n'
                                 '\n'
                                 '    profiles = {}\n'
                                 '    for name, path in references.items():\n'
                                 '        values = np.load(path, allow_pickle=False)\n'
                                 '        if values.ndim == 1:\n'
                                 '            values = values[None, :]\n'
                                 '        if values.ndim != 2 or not len(values):\n'
                                 '            raise ValueError("Reference must contain voice '
                                 'embeddings")\n'
                                 '        profiles[name] = unit(np.mean([unit(value) for value in '
                                 'values], axis=0))\n'
                                 '    rows = []\n'
                                 '    for window in windows:\n'
                                 '        left = window["start"]\n'
                                 '        decoded = asr_records[window["index"]]\n'
                                 '        aligned = aligned_records[window["index"]]\n'
                                 '        decoder_bounds = '
                                 'decoder_sentence_bounds(decoded["segments"], '
                                 'aligned["segments"])\n'
                                 '        for local_index, (segment, bounds) in '
                                 'enumerate(zip(aligned["segments"], decoder_bounds)):\n'
                                 '            alignment_bounds = (float(segment["start"]), '
                                 'float(segment["end"]))\n'
                                 '            primary = bounds if bounds is not None else '
                                 'alignment_bounds\n'
                                 '            variants = []\n'
                                 '            for source, crop in (("decoder", bounds), '
                                 '("alignment", alignment_bounds)):\n'
                                 '                if crop is None or crop[1]-crop[0] < 0.4:\n'
                                 '                    continue\n'
                                 '                if variants and '
                                 'all(abs(crop[i]-variants[0][f"local_{\'start\' if i == 0 else '
                                 '\'end\'}"]) < 1/16000 for i in (0, 1)):\n'
                                 '                    continue\n'
                                 '                waveform = torch.from_numpy(audio[\n'
                                 '                    '
                                 'round((left+crop[0])*16000):round((left+crop[1])*16000)]).unsqueeze(0).to(device)\n'
                                 '                with torch.no_grad():\n'
                                 '                    voice = '
                                 'unit(encoder.encode_batch(waveform).detach().cpu().numpy())\n'
                                 '                scores = {}\n'
                                 '                for name, profile in profiles.items():\n'
                                 '                    if voice.shape != profile.shape:\n'
                                 '                        raise ValueError("Voice reference '
                                 'dimension/model mismatch")\n'
                                 '                    scores[name] = float(voice @ profile)\n'
                                 '                variants.append({"source": source, '
                                 '"local_start": crop[0], "local_end": crop[1],\n'
                                 '                                 "scores": scores})\n'
                                 '            hypothesis = resolve_timing_evidence([item["scores"] '
                                 'for item in variants],\n'
                                 '                                                  '
                                 'args.min_similarity, args.min_margin)\n'
                                 '            quality_flags, quality_signals = '
                                 'decoder_quality_flags(\n'
                                 '                decoded["segments"], *(bounds if bounds is not '
                                 'None else alignment_bounds))\n'
                                 '            absolute_start, absolute_end = left+primary[0], '
                                 'left+primary[1]\n'
                                 '            rows.append({"window_index": window["index"], '
                                 '"local_segment_index": local_index,\n'
                                 '                "start": absolute_start, "end": absolute_end, '
                                 '"text": segment["text"],\n'
                                 '                "review_speaker_hypothesis": hypothesis, '
                                 '"review_required": True,\n'
                                 '                "near_window_boundary": primary[0] <= .25 or '
                                 'primary[1] >= window["end"]-left-.25,\n'
                                 '                "evidence": [\n'
                                 '                    {"source": "review_selection", "reasons": '
                                 'window["reasons"]},\n'
                                 '                    {"source": "baseline_overlap", '
                                 '**baseline_evidence(\n'
                                 '                        baseline["segments"], absolute_start, '
                                 'absolute_end)},\n'
                                 '                    {"source": "decoder_support", "flags": '
                                 'quality_flags,\n'
                                 '                     "signals": quality_signals, '
                                 '"word_probability_is_accuracy": False},\n'
                                 '                    {"source": "local_voice", "timing_variants": '
                                 'variants,\n'
                                 '                     "variants_are_independent_votes": False,\n'
                                 '                     "identity_probability_calibrated": False,\n'
                                 '                     "min_similarity": args.min_similarity, '
                                 '"min_margin": args.min_margin}],\n'
                                 '                "alignment_words": [{**word,\n'
                                 '                    **({"start": left+word["start"]} if "start" '
                                 'in word else {}),\n'
                                 '                    **({"end": left+word["end"]} if "end" in '
                                 'word else {})}\n'
                                 '                    for word in segment.get("words", [])]})\n'
                                 '    assert '
                                 'hashlib.sha256(args.baseline.read_bytes()).hexdigest() == '
                                 'baseline_hash\n'
                                 '    result = {"baseline_modified": False, "baseline_sha256": '
                                 'baseline_hash,\n'
                                 '              "selection_fingerprint": fingerprint, "segments": '
                                 'rows}\n'
                                 '    (args.output_dir / '
                                 '"review_hypotheses.json").write_text(json.dumps(result, '
                                 'indent=2)+"\\n")\n'
                                 '    (args.output_dir / '
                                 '"review_transcript.txt").write_text("\\n".join(\n'
                                 '        f"[{row[\'start\']:.2f}-{row[\'end\']:.2f}] '
                                 '{row[\'review_speaker_hypothesis\']}: {row[\'text\']}"\n'
                                 '        for row in rows)+"\\n")\n'
                                 '    counts = {}\n'
                                 '    for row in rows:\n'
                                 '        counts[row["review_speaker_hypothesis"]] = '
                                 'counts.get(row["review_speaker_hypothesis"], 0)+1\n'
                                 '    summary = {"baseline_segment_count": '
                                 'len(baseline["segments"]),\n'
                                 '        "baseline_modified": False, "review_window_count": '
                                 'len(windows),\n'
                                 '        "decoded_review_minutes": sum(w["end"]-w["start"] for w '
                                 'in windows)/60,\n'
                                 '        "review_hypothesis_count": len(rows), '
                                 '"review_label_counts": counts,\n'
                                 '        "duplicate_overlap_hypotheses_retained": True,\n'
                                 '        "note": "Review hypotheses are supplemental and require '
                                 'comparison; no automatic replacement."}\n'
                                 '    (args.output_dir / '
                                 '"comparison_summary.json").write_text(json.dumps(summary, '
                                 'indent=2)+"\\n")\n'
                                 '    print(json.dumps(summary, indent=2))\n'
                                 '\n'
                                 '\n'
                                 'if __name__ == "__main__":\n'
                                 '    main()\n',
 'test_cloud_runtime.py': 'import tempfile\n'
                          'import unittest\n'
                          'from types import SimpleNamespace\n'
                          'from cloud_runtime import StageCache, create_face_analyzer, '
                          'full_audio_chunks, create_full_audio_vad\n'
                          '\n'
                          '\n'
                          'class CloudRuntimeTests(unittest.TestCase):\n'
                          '    def '
                          'test_cache_reuses_completed_stage_and_isolates_changed_inputs(self):\n'
                          '        with tempfile.TemporaryDirectory() as directory:\n'
                          '            cache = StageCache(directory, {"video": "one", "code": '
                          '"one"})\n'
                          '            self.assertEqual(cache.get("stage", lambda: {"speaker": '
                          '"SPEAKER_02"}), {"speaker": "SPEAKER_02"})\n'
                          '            repeated = StageCache(directory, {"code": "one", "video": '
                          '"one"})\n'
                          '            self.assertEqual(repeated.get("stage", lambda: '
                          'self.fail("stage reran")), {"speaker": "SPEAKER_02"})\n'
                          '            self.assertIsNone(StageCache(directory, {"video": "two", '
                          '"code": "one"}).read("stage"))\n'
                          '            self.assertIsNone(StageCache(directory, {"video": "one", '
                          '"code": "two"}).read("stage"))\n'
                          '            self.assertFalse(list(cache.root.glob("*.tmp")))\n'
                          '\n'
                          '    def test_disabled_cache_does_not_save(self):\n'
                          '        cache = StageCache(None, {})\n'
                          '        cache.write("stage", {"ok": True})\n'
                          '        self.assertIsNone(cache.read("stage"))\n'
                          '\n'
                          '    def test_face_provider_selection_and_actual_fallback(self):\n'
                          '        for device, available, expected, context in (\n'
                          '            ("cpu", ["CUDAExecutionProvider", "CPUExecutionProvider"], '
                          '["CPUExecutionProvider"], -1),\n'
                          '            ("cuda", ["CPUExecutionProvider"], '
                          '["CPUExecutionProvider"], -1),\n'
                          '            ("cuda", ["CUDAExecutionProvider", "CPUExecutionProvider"], '
                          '["CUDAExecutionProvider", "CPUExecutionProvider"], 0)):\n'
                          '            calls = []\n'
                          '            def factory(**kwargs):\n'
                          '                calls.append(kwargs)\n'
                          '                return SimpleNamespace(prepare=lambda **options: '
                          'calls.append(options),\n'
                          '                    models={"recognition": '
                          'SimpleNamespace(session=SimpleNamespace(get_providers=lambda: '
                          '["CPUExecutionProvider"]))})\n'
                          '            ort = SimpleNamespace(get_available_providers=lambda: '
                          'available, preload_dlls=lambda: None)\n'
                          '            _, actual = create_face_analyzer(device, ort, factory)\n'
                          '            self.assertEqual(calls[0]["providers"], expected)\n'
                          '            self.assertEqual(calls[1]["ctx_id"], context)\n'
                          '            self.assertEqual(actual["recognition"], '
                          '["CPUExecutionProvider"])\n'
                          '\n'
                          '\n'
                          '    def test_full_coverage_has_no_gaps_and_bounds_final_window(self):\n'
                          '        chunks = full_audio_chunks(65 * 16000, 16000)\n'
                          '        self.assertEqual(chunks, [{"start": 0, "end": 30}, {"start": '
                          '30, "end": 60}, {"start": 60, "end": 65}])\n'
                          '        for left, right in zip(chunks, chunks[1:]):\n'
                          '            self.assertEqual(left["end"], right["start"])\n'
                          '        self.assertAlmostEqual(full_audio_chunks(16001, '
                          '16000)[-1]["end"], 1.0000625)\n'
                          '        with self.assertRaises(ValueError):\n'
                          '            full_audio_chunks(0, 16000)\n'
                          '\n'
                          '    def test_whisperx_adapter_uses_requested_chunk_size(self):\n'
                          '        import numpy as np\n'
                          '        adapter = create_full_audio_vad()\n'
                          '        audio = np.zeros(25 * 16000)\n'
                          '        self.assertIs(adapter.preprocess_audio(audio), audio)\n'
                          '        detected = adapter({"waveform": audio, "sample_rate": 16000})\n'
                          '        self.assertEqual(adapter.merge_chunks(detected, 10, .5, .36),\n'
                          '            [{"start": 0, "end": 10}, {"start": 10, "end": 20}, '
                          '{"start": 20, "end": 25}])\n'
                          '\n'
                          '    def test_pipeline_reuses_stages_voice_and_visual_evidence(self):\n'
                          '        import contextlib\n'
                          '        import io\n'
                          '        import json\n'
                          '        from pathlib import Path\n'
                          '        from unittest.mock import patch, Mock\n'
                          '        import numpy as np\n'
                          '        import torch\n'
                          '        import chainofrules as pipeline\n'
                          '        with tempfile.TemporaryDirectory() as directory:\n'
                          '            root = Path(directory)\n'
                          '            (root/"video.mp4").write_bytes(b"fake media")\n'
                          '            np.save(root/"voice.npy", np.ones((2, 192), '
                          'dtype=np.float32))\n'
                          '            np.save(root/"face.npy", np.ones((2, 512), '
                          'dtype=np.float32))\n'
                          '            records = [{"start": 0., "end": 1., "speaker": '
                          '"SPEAKER_00"},\n'
                          '                       {"start": 1., "end": 2., "speaker": '
                          '"SPEAKER_01"}]\n'
                          '            assigned = {"segments": [{"start": 0., "end": 1., "text": '
                          '"Hello.", "speaker": "SPEAKER_00"}]}\n'
                          '            whisper = Mock(); whisper.transcribe.return_value = '
                          '{"language": "en", "segments": []}\n'
                          '            voice = Mock(); voice.encode_batch.return_value = '
                          'torch.ones((1, 1, 192))\n'
                          '            detector = '
                          'Mock(return_value=pipeline.pd.DataFrame(records))\n'
                          '            cap = Mock(); cap.get.return_value = 30\n'
                          '            argv = ["chainofrules.py", str(root/"video.mp4"), '
                          '"--voice-priors", str(root/"voice.npy"),\n'
                          '                    "--face-priors", str(root/"face.npy"), "--output", '
                          'str(root/"result.json"),\n'
                          '                    "--cache-dir", str(root/"cache")]\n'
                          '            with patch("sys.argv", argv), patch.dict("os.environ", '
                          '{"HF_TOKEN": "test-placeholder"}), \\\n'
                          '                 patch.object(pipeline.torch.cuda, "is_available", '
                          'return_value=False), \\\n'
                          '                 patch.object(pipeline.whisperx, "load_audio", '
                          'return_value=np.zeros(32000)), \\\n'
                          '                 patch.object(pipeline.torchaudio, "load", '
                          'return_value=(torch.zeros(1, 32000), 16000)), \\\n'
                          '                 patch.object(pipeline.cv2, "VideoCapture", '
                          'return_value=cap), \\\n'
                          '                 patch.object(pipeline.whisperx, "load_model", '
                          'return_value=whisper) as load, \\\n'
                          '                 patch.object(pipeline.whisperx, "load_align_model", '
                          'return_value=(Mock(), {})) as align_load, \\\n'
                          '                 patch.object(pipeline.whisperx, "align", '
                          'return_value=assigned), \\\n'
                          '                 patch.object(pipeline.whisperx, '
                          '"assign_word_speakers", return_value=assigned), \\\n'
                          '                 patch.object(pipeline, "DiarizationPipeline", '
                          'return_value=detector) as diarize_load, \\\n'
                          '                 patch.object(pipeline.SpeakerRecognition, '
                          '"from_hparams", return_value=voice) as voice_load, \\\n'
                          '                 patch.object(pipeline, "create_face_analyzer", '
                          'return_value=(Mock(), {})), \\\n'
                          '                 patch.object(pipeline, "collect_visual_evidence") as '
                          'visual, \\\n'
                          '                 contextlib.redirect_stdout(io.StringIO()):\n'
                          '                pipeline.main()\n'
                          '                first = json.loads((root/"result.json").read_text())\n'
                          '                pipeline.main()\n'
                          '                second = json.loads((root/"result.json").read_text())\n'
                          '                self.assertEqual(first, second)\n'
                          '                self.assertEqual(load.call_count, 1)\n'
                          '                self.assertEqual(align_load.call_count, 1)\n'
                          '                self.assertEqual(diarize_load.call_count, 1)\n'
                          '                self.assertEqual(voice.encode_batch.call_count, 2)\n'
                          '                self.assertEqual(visual.call_count, 1)\n'
                          '                '
                          'self.assertEqual(voice_load.call_args.kwargs["run_opts"], {"device": '
                          '"cpu"})\n'
                          '                self.assertEqual(cap.release.call_count, 2)\n'
                          '                # Coverage changes only ASR/alignment caches, not track '
                          'identity.\n'
                          '                with patch("sys.argv", argv + '
                          '["--transcription-coverage", "full"]):\n'
                          '                    pipeline.main()\n'
                          '                self.assertEqual(load.call_count, 2)\n'
                          '                self.assertIn("vad_model", load.call_args.kwargs)\n'
                          '                self.assertEqual(align_load.call_count, 2)\n'
                          '                self.assertEqual(diarize_load.call_count, 1)\n'
                          '                self.assertEqual(voice.encode_batch.call_count, 2)\n'
                          '                self.assertEqual(visual.call_count, 1)\n'
                          '\n'
                          '\n'
                          'if __name__ == "__main__":\n'
                          '    unittest.main()\n',
 'test_confident_transcript.py': 'import json\n'
                                 'import tempfile\n'
                                 'import unittest\n'
                                 'from pathlib import Path\n'
                                 '\n'
                                 'from export_confident_transcript import classify, '
                                 'export_transcript, timestamp\n'
                                 '\n'
                                 '\n'
                                 'def row(speaker, confidence, text="Words", start=1.0, end=2.0):\n'
                                 '    return {"start": start, "end": end, "final_speaker": '
                                 'speaker,\n'
                                 '            "final_confidence": confidence, "text": text}\n'
                                 '\n'
                                 '\n'
                                 'class ConfidentTranscriptTests(unittest.TestCase):\n'
                                 '    def test_uses_separate_target_and_other_thresholds(self):\n'
                                 '        self.assertTrue(classify(row("Target_Speaker", '
                                 '.35))[0])\n'
                                 '        self.assertFalse(classify(row("Target_Speaker", '
                                 '.349))[0])\n'
                                 '        self.assertTrue(classify(row("SPEAKER_03", .65))[0])\n'
                                 '        self.assertFalse(classify(row("SPEAKER_03", .649))[0])\n'
                                 '\n'
                                 '    def '
                                 'test_uncertain_overlap_and_empty_text_are_reviewed(self):\n'
                                 '        cases = [\n'
                                 '            (row("Uncertain", 1), "uncertain_identity"),\n'
                                 '            (row("Overlapping_Speakers", 1), '
                                 '"overlapping_speakers"),\n'
                                 '            (row("Target_Speaker", 1, " "), "empty_text"),\n'
                                 '        ]\n'
                                 '        for value, reason in cases:\n'
                                 '            with self.subTest(reason=reason):\n'
                                 '                self.assertEqual(classify(value), (False, '
                                 'reason))\n'
                                 '\n'
                                 '    def test_target_like_secondary_track_is_quarantined(self):\n'
                                 '        self.assertEqual(\n'
                                 '            classify(row("SPEAKER_03", 1.0), '
                                 'ambiguous_target_tracks={"SPEAKER_03"}),\n'
                                 '            (False, "ambiguous_target_like_track"),\n'
                                 '        )\n'
                                 '\n'
                                 '    def test_export_preserves_every_row_in_one_output(self):\n'
                                 '        payload = {"target_candidate": "SPEAKER_04",\n'
                                 '                   "cluster_voice_means": {"SPEAKER_04": .42, '
                                 '"SPEAKER_02": .08},\n'
                                 '                   "segments": [\n'
                                 '            row("Target_Speaker", .5, "Target line"),\n'
                                 '            row("SPEAKER_02", .8, "Other line", 2, 3),\n'
                                 '            row("Uncertain", .1, "Review me", 3, 4),\n'
                                 '            row("Overlapping_Speakers", 0, "Two people", 4, 5),\n'
                                 '        ]}\n'
                                 '        with tempfile.TemporaryDirectory() as directory:\n'
                                 '            summary = export_transcript(payload, directory)\n'
                                 '            confident = Path(directory, '
                                 '"confident_transcript.txt").read_text()\n'
                                 '            review = json.loads(Path(directory, '
                                 '"review_segments.json").read_text())\n'
                                 '        self.assertEqual(summary["included_segments"], 2)\n'
                                 '        self.assertEqual(summary["review_segments"], 2)\n'
                                 '        self.assertIn("Target line", confident)\n'
                                 '        self.assertIn("Other line", confident)\n'
                                 '        self.assertEqual([item["baseline_index"] for item in '
                                 'review], [2, 3])\n'
                                 '        self.assertFalse(summary["baseline_modified"])\n'
                                 '\n'
                                 '    def '
                                 'test_export_detects_globally_ambiguous_target_track(self):\n'
                                 '        payload = {\n'
                                 '            "target_candidate": "SPEAKER_04",\n'
                                 '            "cluster_voice_means": {\n'
                                 '                "SPEAKER_04": .415, "SPEAKER_03": .261, '
                                 '"SPEAKER_02": .056,\n'
                                 '            },\n'
                                 '            "segments": [\n'
                                 '                row("SPEAKER_03", 1.0, "Could be target"),\n'
                                 '                row("SPEAKER_02", 1.0, "Clearly other", 2, 3),\n'
                                 '            ],\n'
                                 '        }\n'
                                 '        with tempfile.TemporaryDirectory() as directory:\n'
                                 '            summary = export_transcript(payload, directory)\n'
                                 '            review = json.loads(Path(directory, '
                                 '"review_segments.json").read_text())\n'
                                 '        self.assertEqual(summary["ambiguous_target_tracks"], '
                                 '["SPEAKER_03"])\n'
                                 '        self.assertEqual(summary["included_segments"], 1)\n'
                                 '        self.assertEqual(review[0]["disposition"], '
                                 '"ambiguous_target_like_track")\n'
                                 '\n'
                                 '    def test_timestamp_supports_long_videos(self):\n'
                                 '        self.assertEqual(timestamp(3661.25), "01:01:01.250")\n'
                                 '\n'
                                 '    def '
                                 'test_included_dominant_speaker_retains_overlap_warning(self):\n'
                                 '        value = row("Target_Speaker", .7, "Dominant target '
                                 'words")\n'
                                 '        value["evidence"] = [{\n'
                                 '            "source": "overlapping_speakers",\n'
                                 '            "target_score": 0.0,\n'
                                 '            "confidence": 0.0,\n'
                                 '            "details": {"overlap_seconds": .2, '
                                 '"overlap_fraction": .2,\n'
                                 '                        "intervals": [{"start": 1.4, "end": '
                                 '1.6}]},\n'
                                 '        }]\n'
                                 '        with tempfile.TemporaryDirectory() as directory:\n'
                                 '            export_transcript({"segments": [value]}, directory)\n'
                                 '            confident = Path(directory, '
                                 '"confident_transcript.txt").read_text()\n'
                                 '        self.assertIn("unresolved overlap: 0.20s, 20% of '
                                 'segment", confident)\n'
                                 '        self.assertIn("Dominant target words", confident)\n'
                                 '\n'
                                 '\n'
                                 'if __name__ == "__main__":\n'
                                 '    unittest.main()\n',
 'test_overlap_extraction_review.py': 'import unittest\n'
                                      '\n'
                                      'import numpy as np\n'
                                      '\n'
                                      'from review_overlap_extraction import (\n'
                                      '    classify_extraction, corroborated_novel_words, '
                                      'select_overlap_segments,\n'
                                      '    stereo_metrics,\n'
                                      ')\n'
                                      '\n'
                                      '\n'
                                      'class OverlapExtractionReviewTests(unittest.TestCase):\n'
                                      '    def test_selects_only_target_non_target_overlap(self):\n'
                                      '        baseline = {"segments": [\n'
                                      '            {"text": "yes", "evidence": [{"source": '
                                      '"overlapping_speakers", "details": '
                                      '{"target_and_non_target": True}}]},\n'
                                      '            {"text": "no", "evidence": [{"source": '
                                      '"overlapping_speakers", "details": '
                                      '{"target_and_non_target": False}}]},\n'
                                      '            {"text": "plain", "evidence": []},\n'
                                      '        ]}\n'
                                      '        selected = select_overlap_segments(baseline)\n'
                                      '        self.assertEqual([row[0] for row in selected], '
                                      '[0])\n'
                                      '\n'
                                      '    def '
                                      'test_low_energy_is_suppressed_even_if_asr_hallucinates(self):\n'
                                      '        self.assertEqual(\n'
                                      '            classify_extraction(0.01, 0.40, 0.075, "My name '
                                      'is Jack."),\n'
                                      '            "likely_suppressed_residual",\n'
                                      '        )\n'
                                      '\n'
                                      '    def test_stronger_retained_target_is_candidate(self):\n'
                                      '        self.assertEqual(\n'
                                      '            classify_extraction(0.18, 0.43, 0.108, "I\'m '
                                      'not answering questions."),\n'
                                      '            "candidate_target_speech",\n'
                                      '        )\n'
                                      '\n'
                                      '    def test_borderline_result_abstains(self):\n'
                                      '        self.assertEqual(\n'
                                      '            classify_extraction(0.25, 0.31, 0.15, '
                                      '"Maybe."),\n'
                                      '            "unresolved",\n'
                                      '        )\n'
                                      '\n'
                                      '    def '
                                      'test_duplicated_mono_is_not_analyzed_as_stereo(self):\n'
                                      '        mono = np.linspace(-1, 1, 1600, dtype=np.float32)\n'
                                      '        result = stereo_metrics(np.column_stack([mono, '
                                      'mono]))\n'
                                      '        self.assertTrue(result["available"])\n'
                                      '        self.assertFalse(result["distinct"])\n'
                                      '\n'
                                      '    def '
                                      'test_meaningfully_different_channels_are_detected(self):\n'
                                      '        time = np.arange(1600, dtype=np.float32) / 16000\n'
                                      '        left = np.sin(2 * np.pi * 220 * time)\n'
                                      '        right = np.sin(2 * np.pi * 370 * time)\n'
                                      '        result = stereo_metrics(np.column_stack([left, '
                                      'right]))\n'
                                      '        self.assertTrue(result["distinct"])\n'
                                      '        self.assertLess(result["correlation"], .98)\n'
                                      '\n'
                                      '    def test_novel_word_requires_two_channel_views(self):\n'
                                      '        transcripts = {\n'
                                      '            "left": "No, not on that property. Well.",\n'
                                      '            "right": "No, not on that property.",\n'
                                      '            "middle": "No, not on that property.",\n'
                                      '            "difference": "No, not on that property. Well. '
                                      'Yes.",\n'
                                      '        }\n'
                                      '        result = corroborated_novel_words(\n'
                                      '            transcripts, "No, not on that property."\n'
                                      '        )\n'
                                      '        self.assertEqual(result, [{\n'
                                      '            "word": "well", "support": 2,\n'
                                      '            "views": ["difference", "left"],\n'
                                      '        }])\n'
                                      '\n'
                                      '    def '
                                      'test_repeated_substitution_is_not_recovered_speech(self):\n'
                                      '        transcripts = {\n'
                                      '            "left": "I am going to see you.",\n'
                                      '            "right": "I am going to see you.",\n'
                                      '            "middle": "I am going to see you.",\n'
                                      '        }\n'
                                      '        self.assertEqual(\n'
                                      '            corroborated_novel_words(\n'
                                      '                transcripts, "I am going to sue you."\n'
                                      '            ),\n'
                                      '            [],\n'
                                      '        )\n'
                                      '\n'
                                      '\n'
                                      'if __name__ == "__main__":\n'
                                      '    unittest.main()\n',
 'test_overlap_resolution.py': 'import unittest\n'
                               '\n'
                               'from chainofrules import Baseline, Evidence, TimelineSegment, '
                               'add_overlap_evidence, resolve_segment\n'
                               '\n'
                               '\n'
                               'class OverlapResolutionTests(unittest.TestCase):\n'
                               '    def segment(self):\n'
                               '        value = TimelineSegment(7.21, 9.01, "My name is Jeff.", '
                               'Baseline("SPEAKER_01", "SPEAKER_01"))\n'
                               '        value.evidence.append(Evidence("local_voice", -1.0, 1.0, '
                               '{}))\n'
                               '        return value\n'
                               '\n'
                               '    def test_target_non_target_overlap_stays_unassigned(self):\n'
                               '        segment = self.segment()\n'
                               '        add_overlap_evidence(segment, [\n'
                               '            {"start": 7.18, "end": 8.01, "speaker": '
                               '"SPEAKER_00"},\n'
                               '            {"start": 7.44, "end": 9.14, "speaker": '
                               '"SPEAKER_01"},\n'
                               '        ], "SPEAKER_00")\n'
                               '        resolve_segment(segment, "SPEAKER_00", 1.0)\n'
                               '        self.assertEqual(segment.final_speaker, '
                               '"Overlapping_Speakers")\n'
                               '        self.assertEqual(segment.final_confidence, 0.0)\n'
                               '\n'
                               '    def test_adjacent_tracks_are_not_overlap(self):\n'
                               '        segment = self.segment()\n'
                               '        add_overlap_evidence(segment, [\n'
                               '            {"start": 7.2, "end": 8.0, "speaker": "SPEAKER_00"},\n'
                               '            {"start": 8.0, "end": 9.1, "speaker": "SPEAKER_01"},\n'
                               '        ], "SPEAKER_00")\n'
                               '        self.assertFalse(any(e.source == "overlapping_speakers" '
                               'for e in segment.evidence))\n'
                               '\n'
                               '    def test_tiny_boundary_overlap_is_ignored(self):\n'
                               '        segment = self.segment()\n'
                               '        add_overlap_evidence(segment, [\n'
                               '            {"start": 7.2, "end": 8.05, "speaker": "SPEAKER_00"},\n'
                               '            {"start": 8.0, "end": 9.1, "speaker": "SPEAKER_01"},\n'
                               '        ], "SPEAKER_00")\n'
                               '        self.assertFalse(any(e.source == "overlapping_speakers" '
                               'for e in segment.evidence))\n'
                               '\n'
                               '    def '
                               'test_localized_overlap_preserves_strong_dominant_target(self):\n'
                               '        segment = TimelineSegment(\n'
                               '            10.0, 13.0, "What are you doing?",\n'
                               '            Baseline("SPEAKER_00", "Target_Speaker"),\n'
                               '        )\n'
                               '        segment.evidence.append(Evidence("local_voice", 0.70, '
                               '0.80, {\n'
                               '            "similarity": 0.72,\n'
                               '            "track_similarities": {"SPEAKER_00": 0.71, '
                               '"SPEAKER_01": 0.04},\n'
                               '            "best_track": "SPEAKER_00",\n'
                               '            "track_margin": 0.67,\n'
                               '        }))\n'
                               '        add_overlap_evidence(segment, [\n'
                               '            {"start": 10.0, "end": 13.0, "speaker": '
                               '"SPEAKER_00"},\n'
                               '            {"start": 11.1, "end": 11.7, "speaker": '
                               '"SPEAKER_01"},\n'
                               '        ], "SPEAKER_00")\n'
                               '        resolve_segment(segment, "SPEAKER_00", 1.0)\n'
                               '        self.assertEqual(segment.final_speaker, "Target_Speaker")\n'
                               '        self.assertGreaterEqual(segment.final_confidence, 0.35)\n'
                               '        overlap = next(e for e in segment.evidence if e.source == '
                               '"overlapping_speakers")\n'
                               '        '
                               'self.assertAlmostEqual(overlap.details["overlap_fraction"], 0.2)\n'
                               '        self.assertIn("remains unresolved", segment.reasons[-1])\n'
                               '\n'
                               '    def '
                               'test_broad_overlap_remains_unassigned_despite_target_voice(self):\n'
                               '        segment = TimelineSegment(\n'
                               '            10.0, 13.0, "Two people talking.",\n'
                               '            Baseline("SPEAKER_00", "Target_Speaker"),\n'
                               '        )\n'
                               '        segment.evidence.append(Evidence("local_voice", 0.70, '
                               '0.80, {\n'
                               '            "similarity": 0.72,\n'
                               '            "track_similarities": {"SPEAKER_00": 0.71, '
                               '"SPEAKER_01": 0.04},\n'
                               '            "best_track": "SPEAKER_00",\n'
                               '            "track_margin": 0.67,\n'
                               '        }))\n'
                               '        add_overlap_evidence(segment, [\n'
                               '            {"start": 10.0, "end": 13.0, "speaker": '
                               '"SPEAKER_00"},\n'
                               '            {"start": 11.0, "end": 12.5, "speaker": '
                               '"SPEAKER_01"},\n'
                               '        ], "SPEAKER_00")\n'
                               '        resolve_segment(segment, "SPEAKER_00", 1.0)\n'
                               '        self.assertEqual(segment.final_speaker, '
                               '"Overlapping_Speakers")\n'
                               '        self.assertEqual(segment.final_confidence, 0.0)\n'
                               '\n'
                               '\n'
                               'if __name__ == "__main__":\n'
                               '    unittest.main()\n',
 'test_reference_promotion.py': 'import unittest\n'
                                '\n'
                                'import numpy as np\n'
                                '\n'
                                'from reference_promotion import (evaluate_reference_update,\n'
                                '                                 promoted_source_keys, '
                                'select_candidates,\n'
                                '                                 source_key)\n'
                                '\n'
                                '\n'
                                'def segment(index=0, *, target=True, confidence=1.0, '
                                'duration=2.0,\n'
                                '            similarity=.7, voice_strength=1.0, overlap=False, '
                                'raw="SPEAKER_04"):\n'
                                '    evidence = [{"source": "local_voice", "confidence": '
                                'voice_strength,\n'
                                '                 "target_score": .5, "details": {"similarity": '
                                'similarity}}]\n'
                                '    if overlap:\n'
                                '        evidence.append({"source": "overlapping_speakers", '
                                '"confidence": 0,\n'
                                '                         "target_score": 0, "details": {}})\n'
                                '    return {"start": float(index * 3), "end": float(index * 3 + '
                                'duration),\n'
                                '            "text": f"sample {index}",\n'
                                '            "baseline": {"raw_speaker_track": raw, "speaker": '
                                'raw},\n'
                                '            "final_speaker": "Target_Speaker" if target else '
                                'raw,\n'
                                '            "final_confidence": confidence, "evidence": '
                                'evidence}\n'
                                '\n'
                                '\n'
                                'class ReferencePromotionTests(unittest.TestCase):\n'
                                '    def '
                                'test_selects_only_clean_strong_original_target_segments(self):\n'
                                '        payload = {"target_candidate": "SPEAKER_04", "segments": '
                                '[\n'
                                '            segment(0), segment(1, confidence=.79), segment(2, '
                                'duration=1.0),\n'
                                '            segment(3, similarity=.49), segment(4, '
                                'overlap=True),\n'
                                '            segment(5, raw="SPEAKER_03"), segment(6, '
                                'target=False),\n'
                                '        ]}\n'
                                '        self.assertEqual(\n'
                                '            [row["baseline_index"] for row in '
                                'select_candidates(payload)], [0]\n'
                                '        )\n'
                                '\n'
                                '    def test_safe_consistent_update_passes(self):\n'
                                '        parent = np.array([[1, 0], [.99, .1], [.99, -.1]], '
                                'dtype=np.float32)\n'
                                '        candidates = np.array([[.98, .05]], dtype=np.float32)\n'
                                '        result = evaluate_reference_update(parent, candidates)\n'
                                '        self.assertTrue(result["passed"])\n'
                                '\n'
                                '    def test_inconsistent_candidate_fails(self):\n'
                                '        parent = np.array([[1, 0], [.99, .1], [.99, -.1]], '
                                'dtype=np.float32)\n'
                                '        candidates = np.array([[-1, 0]], dtype=np.float32)\n'
                                '        result = evaluate_reference_update(parent, candidates)\n'
                                '        self.assertFalse(result["passed"])\n'
                                '        '
                                'self.assertFalse(result["checks"]["all_candidates_match_parent"])\n'
                                '\n'
                                '    def test_excludes_already_promoted_source_interval(self):\n'
                                '        metadata = {"promotion_review": {"promoted_candidates": '
                                '[{\n'
                                '            "source": {"video": "https://example/video", "start": '
                                '0, "end": 2}\n'
                                '        }]}}\n'
                                '        excluded = promoted_source_keys(metadata)\n'
                                '        payload = {"target_candidate": "SPEAKER_04", "segments": '
                                '[\n'
                                '            segment(0), segment(1),\n'
                                '        ]}\n'
                                '        selected = select_candidates(\n'
                                '            payload, source_video="https://example/video",\n'
                                '            excluded_sources=excluded,\n'
                                '        )\n'
                                '        self.assertEqual([row["baseline_index"] for row in '
                                'selected], [1])\n'
                                '        self.assertIn(source_key("https://example/video", 0, 2), '
                                'excluded)\n'
                                '\n'
                                '\n'
                                'if __name__ == "__main__":\n'
                                '    unittest.main()\n',
 'test_repeat_evidence.py': 'import unittest\n'
                            '\n'
                            'from chainofrules import Baseline, Evidence, TimelineSegment\n'
                            'from repeat_evidence import (find_repeat_groups, '
                            'build_repeat_proposals,\n'
                            '                             repeat_target_corroboration,\n'
                            '                             resolve_repeat_target_corroboration)\n'
                            '\n'
                            '\n'
                            'def segment(start, text, speaker="Uncertain", confidence=0.0):\n'
                            '    value = TimelineSegment(start, start + 1.0, text, Baseline("S0", '
                            '"S0"))\n'
                            '    value.final_speaker = speaker\n'
                            '    value.final_confidence = confidence\n'
                            '    return value\n'
                            '\n'
                            '\n'
                            'class RepeatEvidenceTests(unittest.TestCase):\n'
                            '    def test_detects_ordered_repeated_presentation(self):\n'
                            '        first = [\n'
                            '            segment(0, "They can ask you to get off their '
                            'property."),\n'
                            '            segment(10, "Are you going to leave the property?"),\n'
                            '            segment(20, "We will arrest you on the property if you do '
                            'not leave."),\n'
                            '            segment(30, "Where does the property begin?"),\n'
                            '        ]\n'
                            '        second = [\n'
                            '            segment(100, "They can ask you to get off their '
                            'property."),\n'
                            '            segment(110, "Are you going to leave the property?"),\n'
                            '            segment(120, "We are going to arrest you on the property '
                            'if you do not leave."),\n'
                            '            segment(130, "Where does the property begin here?"),\n'
                            '        ]\n'
                            '        groups = find_repeat_groups(first + second)\n'
                            '        self.assertEqual(len(groups), 1)\n'
                            '        self.assertEqual(len(groups[0]["anchors"]), 4)\n'
                            '        self.assertAlmostEqual(groups[0]["offset_seconds"], 100.0)\n'
                            '\n'
                            '    def test_bracketed_corruption_receives_donor_candidate(self):\n'
                            '        first = [\n'
                            '            segment(0, "They can ask you to get off their '
                            'property."),\n'
                            '            segment(10, "I\'m a garbled person."),\n'
                            '            segment(20, "We will arrest you on the property if you do '
                            'not leave."),\n'
                            '            segment(30, "Where does the property begin?"),\n'
                            '            segment(40, "My name is Matthew Cox."),\n'
                            '        ]\n'
                            '        second = [\n'
                            '            segment(100, "They can ask you to get off their '
                            'property."),\n'
                            '            segment(110, "I am engaged constitutionally.", '
                            '"Target_Speaker", 0.9),\n'
                            '            segment(120, "We are going to arrest you on the property '
                            'if you do not leave."),\n'
                            '            segment(130, "Where does the property begin here?"),\n'
                            '            segment(140, "My name is Matthew Cox."),\n'
                            '        ]\n'
                            '        timeline = first + second\n'
                            '        groups = find_repeat_groups(timeline)\n'
                            '        proposals = build_repeat_proposals(timeline, groups)\n'
                            '        self.assertEqual(proposals[1][0]["donor_text"], "I am engaged '
                            'constitutionally.")\n'
                            '        self.assertEqual(proposals[1][0]["donor_final_speaker"], '
                            '"Target_Speaker")\n'
                            '        self.assertEqual(first[1].text, "I\'m a garbled person.")\n'
                            '\n'
                            '    def test_isolated_repeated_phrase_is_not_a_presentation(self):\n'
                            '        timeline = [\n'
                            '            segment(0, "Have a nice day."),\n'
                            '            segment(60, "Have a nice day."),\n'
                            '            segment(120, "Something unrelated happened here."),\n'
                            '        ]\n'
                            '        self.assertEqual(find_repeat_groups(timeline), [])\n'
                            '\n'
                            '    def test_exact_repeat_can_corroborate_strong_target_donor(self):\n'
                            '        recipient = segment(10, "I mean, I understand.", '
                            '"SPEAKER_03", 0.77)\n'
                            '        recipient.evidence.append(Evidence(\n'
                            '            "local_voice", -1.0, 0.65, {"similarity": 0.095}\n'
                            '        ))\n'
                            '        proposal = {\n'
                            '            "group_id": "repeat_01", "donor_start": 110, "donor_end": '
                            '111,\n'
                            '            "donor_text": "I mean, I understand.",\n'
                            '            "donor_final_speaker": "Target_Speaker",\n'
                            '            "donor_final_confidence": 0.90,\n'
                            '            "alignment_confidence": 0.75,\n'
                            '            "timing_error_seconds": 0.1,\n'
                            '        }\n'
                            '        corroboration = repeat_target_corroboration(recipient, '
                            '[proposal])\n'
                            '        self.assertIsNotNone(corroboration)\n'
                            '        '
                            'self.assertTrue(resolve_repeat_target_corroboration(recipient, '
                            'corroboration))\n'
                            '        self.assertEqual(recipient.final_speaker, "Target_Speaker")\n'
                            '        self.assertEqual(recipient.final_confidence, 0.55)\n'
                            '        self.assertEqual(recipient.text, "I mean, I understand.")\n'
                            '\n'
                            '    def '
                            'test_repeat_corroboration_rejects_weak_or_partial_evidence(self):\n'
                            '        base = {\n'
                            '            "group_id": "repeat_01", "donor_start": 110, "donor_end": '
                            '111,\n'
                            '            "donor_text": "I mean, I understand.",\n'
                            '            "donor_final_speaker": "Target_Speaker",\n'
                            '            "donor_final_confidence": 0.90,\n'
                            '            "alignment_confidence": 0.75,\n'
                            '            "timing_error_seconds": 0.1,\n'
                            '        }\n'
                            '        cases = [\n'
                            '            ({**base, "donor_final_confidence": 0.74}, 0.10,\n'
                            '             "I mean, I understand."),\n'
                            '            ({**base, "alignment_confidence": 0.71}, 0.10,\n'
                            '             "I mean, I understand."),\n'
                            '            (base, 0.049, "I mean, I understand."),\n'
                            '            (base, 0.10, "I understand a different request '
                            'entirely."),\n'
                            '            ({**base, "donor_final_speaker": "SPEAKER_02"}, 0.10,\n'
                            '             "I mean, I understand."),\n'
                            '        ]\n'
                            '        for proposal, local_similarity, text in cases:\n'
                            '            with self.subTest(proposal=proposal, '
                            'local_similarity=local_similarity,\n'
                            '                              text=text):\n'
                            '                recipient = segment(10, text, "SPEAKER_03", 0.8)\n'
                            '                recipient.evidence.append(Evidence(\n'
                            '                    "local_voice", -1.0, 0.8,\n'
                            '                    {"similarity": local_similarity}\n'
                            '                ))\n'
                            '                self.assertIsNone(\n'
                            '                    repeat_target_corroboration(recipient, '
                            '[proposal])\n'
                            '                )\n'
                            '\n'
                            '    def test_overlap_is_never_reassigned_by_repeat(self):\n'
                            '        recipient = segment(10, "I mean, I understand.",\n'
                            '                            "Overlapping_Speakers", 0.0)\n'
                            '        recipient.evidence.append(Evidence(\n'
                            '            "local_voice", 1.0, 1.0, {"similarity": 0.8}\n'
                            '        ))\n'
                            '        proposal = {\n'
                            '            "donor_text": recipient.text,\n'
                            '            "donor_final_speaker": "Target_Speaker",\n'
                            '            "donor_final_confidence": 1.0,\n'
                            '            "alignment_confidence": 1.0,\n'
                            '        }\n'
                            '        self.assertIsNone(repeat_target_corroboration(recipient, '
                            '[proposal]))\n'
                            '\n'
                            '\n'
                            'if __name__ == "__main__":\n'
                            '    unittest.main()\n',
 'test_review_regions.py': 'import unittest\n'
                           'from review_transcript_regions import select_review_regions\n'
                           '\n'
                           '\n'
                           'class RegionTests(unittest.TestCase):\n'
                           '    def test_good_long_segment_is_not_selected(self):\n'
                           '        rows = [{"start": 1, "end": 4, "final_speaker": '
                           '"Target_Speaker",\n'
                           '                 "final_confidence": .9}]\n'
                           '        self.assertEqual(select_review_regions(rows, 5), [])\n'
                           '\n'
                           '    def '
                           'test_uncertain_short_and_gap_are_selected_without_changing_rows(self):\n'
                           '        rows = [{"start": 5, "end": 5.5, "final_speaker": '
                           '"Uncertain",\n'
                           '                 "final_confidence": .1},\n'
                           '                {"start": 20, "end": 24, "final_speaker": '
                           '"SPEAKER_04",\n'
                           '                 "final_confidence": .8}]\n'
                           '        snapshot = [dict(row) for row in rows]\n'
                           '        result = select_review_regions(rows, 30, context=1, '
                           'minimum_gap=5)\n'
                           '        self.assertEqual(rows, snapshot)\n'
                           '        self.assertTrue(any("uncertain_speaker" in row["reasons"] for '
                           'row in result))\n'
                           '        self.assertTrue(any("transcript_gap" in row["reasons"] for row '
                           'in result))\n'
                           '\n'
                           '    def '
                           'test_windows_are_bounded_and_external_controls_are_data(self):\n'
                           '        rows = [{"start": 1, "end": 99, "final_speaker": "Uncertain",\n'
                           '                 "final_confidence": 0}]\n'
                           '        result = select_review_regions(rows, 100, context=0, '
                           'minimum_gap=200,\n'
                           '                                       maximum_window=30, overlap=4,\n'
                           '                                       extra_regions=[(40, 50)])\n'
                           '        self.assertTrue(all(0 < row["end"]-row["start"] <= 30 for row '
                           'in result))\n'
                           '        self.assertTrue(any("external_review_control" in '
                           'row["reasons"] for row in result))\n'
                           '\n'
                           '    def test_invalid_region_is_rejected(self):\n'
                           '        with self.assertRaises(ValueError):\n'
                           '            select_review_regions([], 10, extra_regions=[(9, 11)])\n'
                           '\n'
                           '\n'
                           'if __name__ == "__main__":\n'
                           '    unittest.main()\n',
 'test_short_answers.py': '"""Behavior checks for conversational attribution, independent of model '
                          'downloads."""\n'
                          'import unittest\n'
                          'import numpy as np\n'
                          'from chainofrules import (Baseline, Evidence, TimelineSegment, '
                          'short_voice_crop,\n'
                          '                          add_question_response_evidence, '
                          'add_echo_question_evidence, add_brief_exchange_evidence, '
                          'resolve_segment, collect_visual_evidence)\n'
                          '\n'
                          '\n'
                          'class ShortAnswerTests(unittest.TestCase):\n'
                          '    def question(self, text="Do you have any weapons?", strength=0.9, '
                          'speaker="SPEAKER_00"):\n'
                          '        segment = TimelineSegment(10.0, 12.83, text, Baseline(speaker, '
                          'speaker))\n'
                          '        segment.final_speaker, segment.final_confidence = speaker, '
                          'strength\n'
                          '        return segment\n'
                          '\n'
                          '    def reply(self, start=12.89, end=12.99, text="No.", '
                          'raw="SPEAKER_00"):\n'
                          '        segment = TimelineSegment(start, end, text, Baseline(raw, '
                          'raw))\n'
                          '        segment.evidence.append(Evidence("local_voice", 0.0, 0.0))\n'
                          '        return segment\n'
                          '\n'
                          '    def infer(self, reply, question, tracks=("SPEAKER_00", '
                          '"SPEAKER_01"), mapping=1.0):\n'
                          '        add_question_response_evidence(reply, question, set(tracks), '
                          '"SPEAKER_01", mapping)\n'
                          '        resolve_segment(reply, "SPEAKER_01", mapping)\n'
                          '        return reply\n'
                          '\n'
                          '    def test_brief_answer_has_weak_alternative_identity(self):\n'
                          '        reply = self.infer(self.reply(), self.question())\n'
                          '        self.assertEqual(reply.final_speaker, "Target_Speaker")\n'
                          '        self.assertLessEqual(reply.final_confidence, 0.25)\n'
                          '        self.assertTrue(any("not voice-verified" in reason for reason '
                          'in reply.reasons))\n'
                          '\n'
                          '    def test_target_question_does_not_force_target_answer(self):\n'
                          '        reply = self.infer(self.reply(raw="SPEAKER_01"), '
                          'self.question(speaker="Target_Speaker"))\n'
                          '        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n'
                          '\n'
                          '    def test_three_speakers_leave_answer_unresolved(self):\n'
                          '        reply = self.infer(self.reply(), self.question(), '
                          '("SPEAKER_00", "SPEAKER_01", "SPEAKER_02"))\n'
                          '        self.assertEqual(reply.final_speaker, "Uncertain")\n'
                          '\n'
                          '    def '
                          'test_no_question_or_weak_question_leaves_answer_unresolved(self):\n'
                          '        for question in (None, self.question("You are on private '
                          'property."),\n'
                          '                         self.question("Why are you here?"), '
                          'self.question(strength=0.35)):\n'
                          '            with self.subTest(question=question):\n'
                          '                self.assertEqual(self.infer(self.reply(), '
                          'question).final_speaker, "Uncertain")\n'
                          '\n'
                          '    def test_gap_overlap_and_long_answer_are_not_inferred(self):\n'
                          '        for reply in (self.reply(13.5, 13.6), self.reply(12.8, 12.9),\n'
                          '                      self.reply(12.89, 14.5), self.reply(text="No one '
                          'should be doing that.")):\n'
                          '            with self.subTest(reply=reply):\n'
                          '                result = self.infer(reply, self.question())\n'
                          '                self.assertFalse(any(item.source == "question_response" '
                          'for item in result.evidence))\n'
                          '                if reply.end - reply.start < 0.4:\n'
                          '                    self.assertEqual(result.final_speaker, '
                          '"Uncertain")\n'
                          '\n'
                          '    def test_weak_target_mapping_does_not_infer(self):\n'
                          '        self.assertEqual(self.infer(self.reply(), self.question(), '
                          'mapping=0.4).final_speaker, "Uncertain")\n'
                          '\n'
                          '    def test_available_voice_is_not_overridden_by_conversation(self):\n'
                          '        reply = self.reply()\n'
                          '        reply.evidence = [Evidence("local_voice", -1.0, 0.5)]\n'
                          '        self.assertEqual(self.infer(reply, '
                          'self.question()).final_speaker, "SPEAKER_00")\n'
                          '\n'
                          '    def test_padded_short_audio_does_not_claim_high_confidence(self):\n'
                          '        reply = self.reply(raw="SPEAKER_01")\n'
                          '        reply.evidence = [Evidence("local_voice", -0.64, 0.28)]\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1.0)\n'
                          '        self.assertLessEqual(reply.final_confidence, 0.30)\n'
                          '\n'
                          '    def test_independent_voice_corrects_short_baseline_conflict(self):\n'
                          '        reply = self.reply(start=0.25, end=0.92, raw="SPEAKER_01", '
                          'text="Question")\n'
                          '        reply.evidence = [Evidence("local_voice", -0.57, 0.55,\n'
                          '            {"best_track": "SPEAKER_00", "track_margin": 0.15,\n'
                          '             "track_similarities": {"SPEAKER_00": 0.31, "SPEAKER_01": '
                          '0.16}})]\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1.0)\n'
                          '        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n'
                          '\n'
                          '    def test_poor_profile_match_cannot_correct_identity(self):\n'
                          '        reply = self.reply(start=0.25, end=0.92, raw="SPEAKER_01", '
                          'text="Question")\n'
                          '        reply.evidence = [Evidence("local_voice", -0.8, 0.55,\n'
                          '            {"best_track": "SPEAKER_00", "track_margin": 0.15,\n'
                          '             "track_similarities": {"SPEAKER_00": 0.20, "SPEAKER_01": '
                          '0.05}})]\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1.0)\n'
                          '        self.assertEqual(reply.final_speaker, "Uncertain")\n'
                          '\n'
                          '    def test_face_presence_alone_does_not_override_voice(self):\n'
                          '        reply = self.reply(start=28.0, end=29.0)\n'
                          '        reply.evidence = [Evidence("local_voice", -0.8, 0.7,\n'
                          '            {"track_margin": 0.03, "track_similarities": {"SPEAKER_00": '
                          '0.23, "SPEAKER_01": 0.20}}),\n'
                          '            Evidence("target_face_visible", 1.0, 1.0)]\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1.0)\n'
                          '        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n'
                          '\n'
                          '    def '
                          'test_mouth_hint_is_tentative_only_with_ambiguous_profiles(self):\n'
                          '        for margin, expected in ((0.03, "Target_Speaker"), (0.20, '
                          '"SPEAKER_00")):\n'
                          '            reply = self.reply(start=28.0, end=29.0)\n'
                          '            reply.evidence = [Evidence("local_voice", -0.8, 0.7,\n'
                          '                {"track_margin": margin, "track_similarities": '
                          '{"SPEAKER_00": 0.23, "SPEAKER_01": 0.20}}),\n'
                          '                Evidence("target_mouth_motion", 1.0, 0.2)]\n'
                          '            resolve_segment(reply, "SPEAKER_01", 1.0)\n'
                          '            self.assertEqual(reply.final_speaker, expected)\n'
                          '            if margin < 0.05:\n'
                          '                self.assertLessEqual(reply.final_confidence, 0.25)\n'
                          '\n'
                          '    def '
                          'test_strong_voice_and_target_mouth_recover_secondary_cluster(self):\n'
                          '        for similarity in (0.374, 0.474, 0.504, 0.528, 0.567):\n'
                          '            with self.subTest(similarity=similarity):\n'
                          '                reply = self.reply(start=215.0, end=217.7, '
                          'raw="SPEAKER_03",\n'
                          '                                   text="Known target passage")\n'
                          '                reply.evidence = [\n'
                          '                    Evidence("local_voice", 0.2, 1.0,\n'
                          '                        {"similarity": similarity, "best_track": '
                          '"SPEAKER_03",\n'
                          '                         "track_margin": 0.1,\n'
                          '                         "track_similarities": {"SPEAKER_03": 0.5, '
                          '"SPEAKER_04": 0.4}}),\n'
                          '                    Evidence("target_face_visible", 0, 0,\n'
                          '                             {"target_visible_hint": True}),\n'
                          '                    Evidence("target_mouth_motion", 1.0, 0.2),\n'
                          '                ]\n'
                          '                resolve_segment(reply, "SPEAKER_04", 1.0)\n'
                          '                self.assertEqual(reply.final_speaker, '
                          '"Target_Speaker")\n'
                          '                self.assertAlmostEqual(reply.final_confidence, '
                          'similarity, places=3)\n'
                          '                self.assertTrue(any("local target voice" in reason for '
                          'reason in reply.reasons))\n'
                          '\n'
                          '    def '
                          'test_secondary_cluster_recovery_requires_all_three_signals(self):\n'
                          '        def evidence(similarity=0.5, visible=True, motion=True, '
                          'voice_strength=1.0):\n'
                          '            values = [\n'
                          '                Evidence("local_voice", 0.2, voice_strength,\n'
                          '                         {"similarity": similarity, '
                          '"track_similarities": {"SPEAKER_03": 0.5}}),\n'
                          '                Evidence("target_face_visible", 0, 0,\n'
                          '                         {"target_visible_hint": visible}),\n'
                          '            ]\n'
                          '            if motion:\n'
                          '                values.append(Evidence("target_mouth_motion", 1.0, '
                          '0.2))\n'
                          '            return values\n'
                          '        cases = [\n'
                          '            evidence(similarity=0.349),\n'
                          '            evidence(visible=False),\n'
                          '            evidence(motion=False),\n'
                          '            evidence(voice_strength=0.49),\n'
                          '        ]\n'
                          '        for values in cases:\n'
                          '            with self.subTest(values=values):\n'
                          '                reply = self.reply(start=215.0, end=217.7, '
                          'raw="SPEAKER_03")\n'
                          '                reply.evidence = values\n'
                          '                resolve_segment(reply, "SPEAKER_04", 1.0)\n'
                          '                self.assertNotEqual(reply.final_speaker, '
                          '"Target_Speaker")\n'
                          '\n'
                          '    def '
                          'test_direct_voice_recovers_only_flagged_target_like_secondary_segment(self):\n'
                          '        def resolve(similarity=0.375, score=0.24, strength=1.0,\n'
                          '                    duration=3.0, target_like=("SPEAKER_03",)):\n'
                          '            reply = self.reply(start=10.0, end=10.0 + duration,\n'
                          '                               raw="SPEAKER_03", text="Known target '
                          'passage")\n'
                          '            reply.evidence = [Evidence(\n'
                          '                "local_voice", score, strength,\n'
                          '                {"similarity": similarity, "best_track": "SPEAKER_03",\n'
                          '                 "track_margin": 0.11,\n'
                          '                 "track_similarities": {"SPEAKER_03": 0.5,\n'
                          '                                        "SPEAKER_04": 0.39}},\n'
                          '            )]\n'
                          '            resolve_segment(reply, "SPEAKER_04", 1.0, target_like)\n'
                          '            return reply\n'
                          '\n'
                          '        recovered = resolve()\n'
                          '        self.assertEqual(recovered.final_speaker, "Target_Speaker")\n'
                          '        self.assertAlmostEqual(recovered.final_confidence, 0.375)\n'
                          '        self.assertTrue(any("target-like secondary track" in reason\n'
                          '                            for reason in recovered.reasons))\n'
                          '\n'
                          '        # Promotion raised the run-level target mean in a real '
                          'full-video run,\n'
                          '        # lowering the normalized target_score even though direct '
                          'similarity\n'
                          '        # improved. The target-to-competitor margin remains stable and '
                          'should\n'
                          '        # preserve this conservative secondary-track recovery.\n'
                          '        stable = self.reply(start=20.0, end=23.0, raw="SPEAKER_03",\n'
                          '                            text="Repeated target passage")\n'
                          '        stable.evidence = [Evidence(\n'
                          '            "local_voice", 0.12, 1.0,\n'
                          '            {"similarity": 0.379, "competitor_mean": 0.276,\n'
                          '             "reference_margin": 0.103, "best_track": "SPEAKER_03",\n'
                          '             "track_margin": 0.09,\n'
                          '             "track_similarities": {"SPEAKER_03": 0.51,\n'
                          '                                      "SPEAKER_04": 0.42}},\n'
                          '        )]\n'
                          '        resolve_segment(stable, "SPEAKER_04", 1.0, ("SPEAKER_03",))\n'
                          '        self.assertEqual(stable.final_speaker, "Target_Speaker")\n'
                          '\n'
                          '        for kwargs in (\n'
                          '            {"similarity": 0.349}, {"score": 0.149}, {"strength": '
                          '0.49},\n'
                          '            {"duration": 0.59}, {"target_like": ()},\n'
                          '        ):\n'
                          '            with self.subTest(kwargs=kwargs):\n'
                          '                self.assertNotEqual(resolve(**kwargs).final_speaker,\n'
                          '                                    "Target_Speaker")\n'
                          '\n'
                          '    def '
                          'test_face_identity_continues_through_head_turn_but_not_bbox_jump(self):\n'
                          '        class Capture:\n'
                          '            index = 0\n'
                          '            def get(self, prop): return 20\n'
                          '            def set(self, prop, value): self.index = int(value)\n'
                          '            def read(self): return True, np.zeros((200, 200, 3), '
                          'dtype=np.uint8)\n'
                          '        class Face:\n'
                          '            pass\n'
                          '        for jump in (False, True):\n'
                          '            capture = Capture()\n'
                          '            class Analyzer:\n'
                          '                def get(self, frame):\n'
                          '                    face = Face()\n'
                          '                    face.embedding = np.array([0.7, np.sqrt(1-0.7**2)]) '
                          'if capture.index < 2 else np.array([0.3, np.sqrt(1-0.3**2)])\n'
                          '                    face.bbox = np.array([10,10,80,100]) if not jump or '
                          'capture.index < 2 else np.array([120,120,190,200])\n'
                          '                    points = np.zeros((68,3))\n'
                          '                    points[64,0] = 10\n'
                          '                    points[66,1] = 0.4 if capture.index % 2 else 1.2\n'
                          '                    face.landmark_3d_68 = points\n'
                          '                    return [face]\n'
                          '            reply = self.reply(start=0.5, end=1.4)\n'
                          '            collect_visual_evidence(reply, capture, 8.0, Analyzer(), '
                          'np.array([1.0, 0.0]))\n'
                          '            motion = next(item for item in reply.evidence if '
                          'item.source == "target_mouth_motion")\n'
                          '            self.assertEqual(motion.confidence > 0, not jump)\n'
                          '\n'
                          '    def test_crop_respects_both_neighbors(self):\n'
                          '        following = TimelineSegment(13.01, 15.6, "Next", '
                          'Baseline("SPEAKER_00", "SPEAKER_00"))\n'
                          '        start, end = short_voice_crop(self.reply(), self.question(), '
                          'following, 30.0)\n'
                          '        self.assertAlmostEqual(start, 12.83)\n'
                          '        self.assertAlmostEqual(end, 13.01)\n'
                          '        self.assertLess(end-start, 0.4)\n'
                          '\n'
                          '    def test_overlapping_timing_does_not_trim_reply(self):\n'
                          '        preceding = self.question()\n'
                          '        preceding.end = 12.92\n'
                          '        reply = self.reply()\n'
                          '        self.assertEqual(short_voice_crop(reply, preceding, None, '
                          '30.0), (reply.start, reply.end))\n'
                          '\n'
                          '\n'
                          '    def '
                          'test_echo_requires_weak_supporting_voice_and_visible_target(self):\n'
                          '        previous = self.question("You are being arrested for criminal '
                          'loitering.")\n'
                          '        reply = self.reply(start=13.3, end=13.88, text="Criminal '
                          'loitering?")\n'
                          '        reply.evidence = [Evidence("local_voice", -.8, .48,\n'
                          '            {"best_track": "SPEAKER_01", "track_margin": .07,\n'
                          '             "track_similarities": {"SPEAKER_00": .05, "SPEAKER_01": '
                          '.12}}),\n'
                          '            Evidence("target_face_visible", 0, 0, '
                          '{"target_visible_hint": True})]\n'
                          '        add_echo_question_evidence(reply, previous, {"SPEAKER_00", '
                          '"SPEAKER_01"}, "SPEAKER_01", 1)\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1)\n'
                          '        self.assertEqual(reply.final_speaker, "Target_Speaker")\n'
                          '        self.assertEqual(reply.final_confidence, .20)\n'
                          '        reply.evidence = [e for e in reply.evidence if e.source != '
                          '"target_face_visible"]\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1)\n'
                          '        self.assertNotEqual(reply.final_speaker, "Target_Speaker")\n'
                          '\n'
                          '    def '
                          'test_echo_does_not_override_clear_voice_or_choose_among_three_people(self):\n'
                          '        previous = self.question("You are being arrested for criminal '
                          'loitering.")\n'
                          '        reply = self.reply(start=13.3, end=13.88, text="Criminal '
                          'loitering?")\n'
                          '        reply.evidence = [Evidence("local_voice", -1, .6,\n'
                          '            {"best_track": "SPEAKER_00", "track_margin": .4,\n'
                          '             "track_similarities": {"SPEAKER_00": .5, "SPEAKER_01": '
                          '.1}}),\n'
                          '            Evidence("target_face_visible", 0, 0, '
                          '{"target_visible_hint": True})]\n'
                          '        add_echo_question_evidence(reply, previous, {"SPEAKER_00", '
                          '"SPEAKER_01"}, "SPEAKER_01", 1)\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1)\n'
                          '        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n'
                          '        reply.evidence = []\n'
                          '        add_echo_question_evidence(reply, previous, {"SPEAKER_00", '
                          '"SPEAKER_01", "SPEAKER_02"}, "SPEAKER_01", 1)\n'
                          '        self.assertEqual(reply.evidence, [])\n'
                          '\n'
                          '    def test_padded_but_weak_voice_allows_tentative_answer(self):\n'
                          '        reply = self.reply()\n'
                          '        reply.evidence = [Evidence("local_voice", -1, .28,\n'
                          '            {"best_track": "SPEAKER_00", "track_similarities": '
                          '{"SPEAKER_00": .23, "SPEAKER_01": .08}})]\n'
                          '        self.infer(reply, self.question())\n'
                          '        self.assertEqual(reply.final_speaker, "Target_Speaker")\n'
                          '        self.assertEqual(reply.final_confidence, .20)\n'
                          '\n'
                          '\n'
                          '    def test_acknowledgement_requires_independent_agreement(self):\n'
                          '        previous = self.question("Your request has been accepted.")\n'
                          '        reply = self.reply(text="Oh, okay.")\n'
                          '        reply.evidence = [Evidence("local_voice", -.7, .28,\n'
                          '            {"best_track": "SPEAKER_01", "track_similarities": '
                          '{"SPEAKER_00": 0, "SPEAKER_01": .1}})]\n'
                          '        add_brief_exchange_evidence(reply, previous, {"SPEAKER_00", '
                          '"SPEAKER_01"}, "SPEAKER_01", 1)\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1)\n'
                          '        self.assertEqual(reply.final_speaker, "Target_Speaker")\n'
                          '        self.assertEqual(reply.final_confidence, .2)\n'
                          '        for strength, best, tracks in ((.6, "SPEAKER_01", '
                          '{"SPEAKER_00", "SPEAKER_01"}),\n'
                          '                                      (.28, "SPEAKER_00", '
                          '{"SPEAKER_00", "SPEAKER_01"}),\n'
                          '                                      (.28, "SPEAKER_01", '
                          '{"SPEAKER_00", "SPEAKER_01", "SPEAKER_02"})):\n'
                          '            reply.evidence = [Evidence("local_voice", -.7, strength,\n'
                          '                {"best_track": best, "track_similarities": '
                          '{"SPEAKER_00": 0, "SPEAKER_01": .1}})]\n'
                          '            add_brief_exchange_evidence(reply, previous, tracks, '
                          '"SPEAKER_01", 1)\n'
                          '            self.assertEqual(len(reply.evidence), 1)\n'
                          '\n'
                          '    def '
                          'test_confirmation_requires_face_baseline_and_no_inference_chain(self):\n'
                          '        previous = self.question("Really?", .30, "SPEAKER_01")\n'
                          '        previous.final_speaker = "Target_Speaker"\n'
                          '        previous.evidence = [Evidence("target_face_visible", 0, 0, '
                          '{"target_visible_hint": True})]\n'
                          '        reply = self.reply(text="Yeah.", raw="SPEAKER_01")\n'
                          '        voice = Evidence("local_voice", -1, .15,\n'
                          '            {"best_track": "SPEAKER_00", "track_similarities": '
                          '{"SPEAKER_00": .06, "SPEAKER_01": .02}})\n'
                          '        reply.evidence = [voice]\n'
                          '        add_brief_exchange_evidence(reply, previous, {"SPEAKER_00", '
                          '"SPEAKER_01"}, "SPEAKER_01", 1)\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1)\n'
                          '        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n'
                          '        self.assertEqual(reply.final_confidence, .20)\n'
                          '        for face, reasons in (([], []), (previous.evidence, ["weak '
                          'question/answer inference"])):\n'
                          '            previous.evidence, previous.reasons = face, reasons\n'
                          '            reply.evidence = [voice]\n'
                          '            add_brief_exchange_evidence(reply, previous, {"SPEAKER_00", '
                          '"SPEAKER_01"}, "SPEAKER_01", 1)\n'
                          '            self.assertEqual(len(reply.evidence), 1)\n'
                          '\n'
                          '\n'
                          'if __name__ == "__main__":\n'
                          '    unittest.main()\n',
 'test_single_speaker_mapping.py': 'import unittest\n'
                                   '\n'
                                   'from chainofrules import voice_mapping_confidence\n'
                                   '\n'
                                   '\n'
                                   'class SingleSpeakerMappingTests(unittest.TestCase):\n'
                                   '    def test_strong_well_sampled_single_track_can_map(self):\n'
                                   '        self.assertAlmostEqual(\n'
                                   '            voice_mapping_confidence(["S0"], {"S0": 0.304}, '
                                   '{"S0": 13}),\n'
                                   '            (0.304 - 0.18) / 0.15,\n'
                                   '        )\n'
                                   '\n'
                                   '    def '
                                   'test_weak_or_under_sampled_single_track_stays_uncertain(self):\n'
                                   '        self.assertEqual(voice_mapping_confidence(["S0"], '
                                   '{"S0": 0.18}, {"S0": 13}), 0.0)\n'
                                   '        self.assertEqual(voice_mapping_confidence(["S0"], '
                                   '{"S0": 0.40}, {"S0": 2}), 0.0)\n'
                                   '\n'
                                   '    def '
                                   'test_multi_track_behavior_still_uses_separation(self):\n'
                                   '        self.assertAlmostEqual(\n'
                                   '            voice_mapping_confidence(\n'
                                   '                ["target", "other"],\n'
                                   '                {"target": 0.319, "other": 0.055},\n'
                                   '                {"target": 3, "other": 2},\n'
                                   '            ),\n'
                                   '            1.0,\n'
                                   '        )\n'
                                   '\n'
                                   '\n'
                                   'if __name__ == "__main__":\n'
                                   '    unittest.main()\n',
 'test_transcript_gaps.py': 'import unittest\n'
                            'from recover_transcript_gaps import '
                            'uncovered_intervals,recovery_windows,collect_candidates,preserve_baseline\n'
                            'class GapTests(unittest.TestCase):\n'
                            ' def test_union_handles_overlaps_and_edges(self):\n'
                            '  '
                            "self.assertEqual(uncovered_intervals([{'start':2,'end':5},{'start':4,'end':8},{'start':12,'end':14}],17),[(0.,2.),(8.,12.),(14.,17)])\n"
                            ' def test_windows_cover_gap_without_tiny_tail(self):\n'
                            '  windows=recovery_windows((10,30.00001),40,size=8,overlap=4)\n'
                            '  self.assertTrue(all(0<r-l<=8.000001 for l,r in '
                            'windows));self.assertEqual(windows[0][0],9.5);self.assertEqual(windows[-1][1],30.50001)\n'
                            '  self.assertTrue(all(b[0]<=a[1] for a,b in '
                            'zip(windows,windows[1:])))\n'
                            ' def test_context_is_bounded_and_invalid_context_rejected(self):\n'
                            '  windows=recovery_windows((1,39),40,size=20,overlap=10,context=2)\n'
                            '  '
                            'self.assertEqual(windows[0][0],0);self.assertEqual(windows[-1][1],40)\n'
                            "  for context in (-1,float('nan'),float('inf')):\n"
                            '   with '
                            'self.assertRaises(ValueError):recovery_windows((10,30),40,context=context)\n'
                            ' def test_repetition_needs_distinct_windows_and_stays_in_gap(self):\n'
                            '  '
                            "words=[{'start':4,'end':4.5,'word':'Height?','probability':.9,'window_index':0},{'start':4.1,'end':4.6,'word':'height','probability':.8,'window_index':1},{'start':1,'end':2,'word':'existing','probability':1,'window_index':1}]\n"
                            '  '
                            "candidates=collect_candidates(words,(3,6));self.assertEqual(len(candidates),1);self.assertTrue(candidates[0]['repeated_in_overlapping_windows']);self.assertTrue(candidates[0]['review_required'])\n"
                            ' def test_candidate_does_not_splice_disagreeing_windows(self):\n'
                            '  '
                            "words=[{'start':4,'end':4.2,'word':'one','probability':.9,'window_index':0},{'start':4.3,'end':4.5,'word':'answer','probability':.6,'window_index':0},{'start':4,'end':4.2,'word':'another','probability':.6,'window_index':1},{'start':4.3,'end':4.5,'word':'answer','probability':.9,'window_index':1}]\n"
                            '  '
                            "candidates=collect_candidates(words,(3,6));self.assertEqual(len(candidates),1);self.assertEqual(len({w['window_index'] "
                            "for w in candidates[0]['words']}),1)\n"
                            ' def '
                            'test_zero_duration_word_kept_with_phrase_not_invented_timing(self):\n'
                            "  words=[{'start':4,'end':4.5,'word':' "
                            "How','probability':.9,'window_index':0},{'start':4.5,'end':4.5,'word':' "
                            "tall?','probability':.9,'window_index':0}]\n"
                            '  '
                            "result=collect_candidates(words,(3,6));self.assertEqual(result[0]['text'],'How "
                            "tall?');self.assertEqual(result[0]['words'][1]['start'],result[0]['words'][1]['end'])\n"
                            '  self.assertEqual(collect_candidates(words[1:],(3,6)),[])\n'
                            ' def test_keeps_baseline_nested_fields_and_identity_unchanged(self):\n'
                            '  '
                            "original={'segments':[{'start':0,'end':1,'text':'works','final_speaker':'Target_Speaker','words':[{'word':'works'}],'reasons':['original']}]}\n"
                            '  '
                            "result=preserve_baseline(original,[{'text':'new','speaker':'Uncertain'}]);self.assertEqual(result['segments'],original['segments']);result['segments'][0]['words'][0]['word']='mutated';self.assertEqual(original['segments'][0]['words'][0]['word'],'works')\n"
                            "if __name__=='__main__':unittest.main()\n",
 'test_window_review.py': 'import unittest\n'
                          'from review_audio_window import '
                          'baseline_evidence,resolve_voice,decoder_sentence_bounds,resolve_timing_evidence,decoder_quality_flags\n'
                          'class WindowTests(unittest.TestCase):\n'
                          '    def test_confident_beep_words_do_not_prove_speech(self):\n'
                          '        '
                          "s=[{'start':0,'end':4.2,'no_speech_prob':.808,'avg_logprob':-.584,'words':[{'probability':.94}]}]\n"
                          '        flags,_=decoder_quality_flags(s,1,2)\n'
                          '        self.assertTrue(flags)\n'
                          '    def test_unrelated_decoder_warning_not_applied(self):\n'
                          '        '
                          "flags,_=decoder_quality_flags([{'start':0,'end':1,'no_speech_prob':.9}],3,4)\n"
                          '        self.assertEqual(flags,[])\n'
                          '    def test_consistent_crops_with_one_qualified_match(self):\n'
                          '        '
                          "self.assertEqual(resolve_timing_evidence([{'Target':.34,'Other':.18},{'Target':.27,'Other':.21}]),'Target')\n"
                          '    def test_conflicting_crops_do_not_choose_the_stronger_match(self):\n'
                          '        '
                          "self.assertEqual(resolve_timing_evidence([{'Target':.7,'Other':.1},{'Target':.1,'Other':.3}]),'Uncertain')\n"
                          '    def test_two_weak_crops_do_not_accumulate_confidence(self):\n'
                          '        '
                          "self.assertEqual(resolve_timing_evidence([{'Target':.2,'Other':.1},{'Target':.21,'Other':.1}]),'Uncertain')\n"
                          '    def test_decoder_words_follow_repeated_sentences_in_order(self):\n'
                          "        d=[{'text':'I agree. I "
                          "agree.','words':[{'word':'I','start':0,'end':.1},{'word':'agree.','start':.1,'end':1},{'word':'I','start':2,'end':2.1},{'word':'agree.','start':2.1,'end':3}]}]\n"
                          "        a=[{'text':'I agree.'},{'text':'I agree.'}]\n"
                          '        self.assertEqual(decoder_sentence_bounds(d,a),[(0,1),(2,3)])\n'
                          '    def test_missing_word_links_do_not_invent_timings(self):\n'
                          '        '
                          "self.assertEqual(decoder_sentence_bounds([{'text':'Hi.'}],[{'text':'Hi.'}]),[None])\n"
                          '    def test_raw_ids_preserved_and_offsets_applied(self):\n'
                          '        '
                          "a=[{'start':0,'end':2,'raw_speaker_track':'SPEAKER_03','final_speaker':'Target_Speaker'},{'start':2,'end':4,'raw_speaker_track':'SPEAKER_07','final_speaker':'SPEAKER_07'}]\n"
                          '        result=baseline_evidence(a,101,103,offset=100)\n'
                          '        '
                          "self.assertEqual(result['raw_track_overlap_seconds'],{'SPEAKER_03':1,'SPEAKER_07':1})\n"
                          "        self.assertEqual(a[0]['raw_speaker_track'],'SPEAKER_03')\n"
                          '    def test_current_pipeline_nested_baseline_schema(self):\n'
                          '        '
                          "s=[{'start':0,'end':2,'baseline':{'raw_speaker_track':'SPEAKER_08','speaker':'SPEAKER_08'},'final_speaker':'SPEAKER_08'}]\n"
                          '        '
                          "self.assertEqual(baseline_evidence(s,0,1)['raw_track_overlap_seconds'],{'SPEAKER_08':1})\n"
                          '    def test_no_observation_is_not_negative_identity_evidence(self):\n'
                          "        self.assertEqual(baseline_evidence([],0,2)['segments'],[])\n"
                          "        self.assertEqual(resolve_voice({}),'Uncertain')\n"
                          '    def test_short_response_cannot_borrow_questioners_identity(self):\n'
                          '        '
                          "self.assertEqual(resolve_voice({'Target_Speaker':.22,'Officer':.21}),'Uncertain')\n"
                          '    def test_close_profiles_abstain_even_if_similarity_is_high(self):\n'
                          '        '
                          "self.assertEqual(resolve_voice({'Target_Speaker':.6,'Other':.57}),'Uncertain')\n"
                          '    def '
                          'test_strong_separated_profile_produces_review_hypothesis(self):\n'
                          '        '
                          "self.assertEqual(resolve_voice({'Target_Speaker':.5,'Other':.1}),'Target_Speaker')\n"
                          "if __name__=='__main__':unittest.main()\n"}
for name, source in EMBEDDED_FILES.items():
    (WORK/name).write_text(source)

ENV = os.environ.copy()
ENV['PYTHONUNBUFFERED'] = '1'
ENV['MPLCONFIGDIR'] = str(BASE/'matplotlib-cache')
ENV['MPLBACKEND'] = 'Agg'
ENV['NUMBA_CACHE_DIR'] = str(BASE/'numba-cache')
def checked(command, **kwargs):
    return subprocess.run(command, env=ENV, check=True, **kwargs)

print('1/4: Creating/checking the isolated environment', flush=True)
ready = False
if Path(PYTHON).exists():
    ready = subprocess.run([PYTHON, '-m', 'pip', '--version'], env=ENV,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not ready:
    bootstrap = BASE/'bootstrap-tools'
    checked([sys.executable, '-m', 'pip', 'install', '--target', str(bootstrap),
             'virtualenv>=20.26,<21', 'wrapt'])
    bootstrap_env = ENV.copy(); bootstrap_env['PYTHONPATH'] = str(bootstrap)
    subprocess.run([sys.executable, '-m', 'virtualenv', '--no-download', str(VENV)],
                   env=bootstrap_env, check=True)

print('2/4: Installing the pipeline and extraction model', flush=True)
requirements = [
    'whisperx==3.8.6', 'speechbrain==1.1.1', 'insightface==2.0',
    'torch==2.8.0', 'torchaudio==2.8.0', 'numpy==2.5.3',
    'opencv-python==5.0.0.93', 'onnxruntime-gpu==1.23.2', 'wrapt',
    'yt-dlp', 'soundfile',
]
checked([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'])
checked([PYTHON, '-m', 'pip', 'install', *requirements])
checked([PYTHON, '-m', 'pip', 'install',
         'git+https://github.com/wenet-e2e/wespeaker.git'])
# WeSep's current package metadata omits its namespace-style wesep/utils
# directory. Keep the checkout and put it first on PYTHONPATH so the complete
# source tree is used, while pip still installs all declared dependencies.
WESEP_SOURCE = WORK/'vendor'/'wesep'
if not (WESEP_SOURCE/'wesep'/'utils'/'utils.py').is_file():
    WESEP_SOURCE.parent.mkdir(parents=True, exist_ok=True)
    checked(['git', 'clone', '--depth', '1',
             'https://github.com/wenet-e2e/wesep.git', str(WESEP_SOURCE)])
checked([PYTHON, '-m', 'pip', 'install', str(WESEP_SOURCE)])
# The upstream wheel omits wesep/utils because that directory has no
# __init__.py. Overlay the complete checkout onto site-packages so imports do
# not depend on PYTHONPATH or notebook process state.
site_packages = Path(subprocess.check_output(
    [PYTHON, '-c', 'import site; print(site.getsitepackages()[0])'],
    env=ENV, text=True).strip())
installed_wesep = site_packages/'wesep'
shutil.copytree(WESEP_SOURCE/'wesep', installed_wesep, dirs_exist_ok=True)
missing_utility = installed_wesep/'utils'/'utils.py'
if not missing_utility.is_file():
    raise RuntimeError(f'WeSep repair failed; missing {missing_utility}')
# These upstream source directories contain Python modules but omit package
# markers, which is also why they disappear from the built wheel.
for directory in [installed_wesep/'utils', installed_wesep/'dataset', installed_wesep/'bin']:
    if directory.is_dir():
        (directory/'__init__.py').touch()
ENV['PYTHONPATH'] = str(WORK) + os.pathsep + ENV.get('PYTHONPATH', '')

print('3/4: Selecting the CUDA ONNX runtime', flush=True)
checked([PYTHON, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'])
checked([PYTHON, '-m', 'pip', 'install', '--no-deps', '--force-reinstall',
         'onnxruntime-gpu==1.23.2'])
if shutil.which('ffmpeg') is None:
    raise RuntimeError('ffmpeg is required. Kaggle normally includes it.')
library_dirs = subprocess.check_output([PYTHON, '-c',
    "import site,pathlib; print(':'.join(str(p) for d in site.getsitepackages() for p in pathlib.Path(d).glob('nvidia/*/lib')))"], env=ENV, text=True).strip()
ENV['LD_LIBRARY_PATH'] = library_dirs + ':' + ENV.get('LD_LIBRARY_PATH', '')

print('4/4: Verifying GPU imports and focused behavior tests', flush=True)
verification = """import torch,onnxruntime as ort,wrapt,wesep
import chainofrules,repeat_evidence
print('Torch:', torch.__version__, 'CUDA build:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (local CPU)')
print('ONNX providers:', ort.get_available_providers())
print('WeSep repaired package:', wesep.__file__)
"""
if ON_KAGGLE:
    verification += "assert torch.cuda.is_available(), 'Kaggle GPU is unavailable'\nassert 'CUDAExecutionProvider' in ort.get_available_providers(), 'GPU ONNX runtime is unavailable'\n"
checked([PYTHON, '-c', verification], cwd=WORK)
checked([PYTHON, '-m', 'unittest', 'test_cloud_runtime', 'test_short_answers',
         'test_repeat_evidence', 'test_overlap_resolution',
         'test_overlap_extraction_review', 'test_confident_transcript',
         'test_reference_promotion'], cwd=WORK)

import hashlib
REFERENCE_FILES = {'auditor_enrollment.wav': '...',
 'face_embeddings.npy': '...',
 'opening_officer_reference.npy': 'k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE5MiwpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIApDR569eD+ePSVTQjzXL8Q8mFepvQvxDz3M/vy9YNoYvbW3or2vrIm9B/1hvZUsuD3xL569DlK0vZXXV76CoP49pTbJvXm8+D0rOMO7dQlDPgHbrb32ZTq9CKD5vQ9U37tiJZm9awdSPA8JgL2s9lw9o0yAPS0WeT0kG5e9Qyscvnywsr0gUcM9MYe9PWgwhT2T9/+8dN+VPE0xhL0qjMe9kvxuPYCZDL4TrUS9GluQPZgFOD1fWoc9hl6NPSuuzj07pmY9G28RvUyrVrx3J4o9nemUvf1lCb2RV1O9samYOx9Kwz3HZcS83eeqPRlqHrx3YXi9tTCWvIBTfT3P+nq97hzyu3W4qT1uji49wXiDPatGqT3xJ6Y9+7ytvRKQeD0oG5W9AEVSO0Rk2z0s18S8yngZvqcF+TwuJk07nuqgvFu6i7zfPdk8XlA7vIzIU72kDnQ7LuPhPflzkTy7ZaU9SQprvOXkfz2Rcaa971UaveED+r1wB3E9oWuXvfqJ6b04ZL+8glvUvf9vVzvUXMy9PJmRvGJhQz04Ou49Me+yvV+DsL3XHok97akwOyr1q7x9JoE8t7utvXZgrT1+4sS97rJQvWCahrzGHao9BmubPWbicbydw8o9UoElPaMZxT2Kjim8lFDIPAeF873jfsk9KHSFPQCjizwX4sw8R/MEPahYvr1tTkC9i+NzPR4WEz0djJY8bPG+PL661j2XY0S8ZmstPcr++r0yOwc8hiOaPAC8ar3XBI68ty0JvAQC2LyIdd69/njeu/eipT3X4pE9lrucvL/j3D0f7uA9prS+PRFYW71r4v+7YbtYPRYMaD09C4u8QvxFPX7KIb1GAbE99asPvp3whr3hlR46/mkAPvSJrTyyiaW8tCXrPKVQu7xzO7+96JEzPevqjjydtB6+tr9tPdb7Kr2T0hs9InaLPXhmhb1qw0Y9pWg+vdvlEr2TkHA9JC7gvdhyML0Dtlc95fy6PVnmz71jg+O9MjXBPethZz0e+hg9OvCCPXPA3ro=',
 'reference.json': 'ewogICJzY2hlbWFfdmVyc2lvbiI6IDEsCiAgIm1ldGhvZCI6ICJyZXZpZXdlZCBwb3N0LXJ1biBwcm9tb3Rpb247IHZlcnNpb25lZCBhbmQgcmV2ZXJzaWJsZSIsCiAgInBhcmVudF9yZWZlcmVuY2UiOiB7CiAgICAicGF0aCI6ICJyZWZlcmVuY2VzL2F1ZGl0b3Ita2FnZ2xlLXYxMy1wYXJlbnQiLAogICAgInZvaWNlX2VtYmVkZGluZ3Nfc2hhMjU2IjogImRiNTcwMTE4ODAwNjBhN2Q4NmY0YjU5ZmU4MjVkYzM5ZTFjZTE3YjZmN2NkNWJmMmI1ZDczNGI4ODFhYmE2ZTAiLAogICAgInJlZmVyZW5jZV9qc29uX3NoYTI1NiI6ICI1YTI2NjNlMjZhNjEyNWMwMTEwMDYzZTk3YzY4NGFkY2QyMjNkMzA3ODVmNDUzZDRhYTQ0MGJjMGM2NjhmZWFhIiwKICAgICJtZXRhZGF0YSI6IHsKICAgICAgIm1ldGhvZCI6ICJtYW51YWwgaWRlbnRpdHkgYXBwcm92YWwgZm9sbG93ZWQgYnkgY29uc2lzdGVuY3kgc2NyZWVuaW5nIiwKICAgICAgInNjcmVlbmluZyI6IHsKICAgICAgICAidm9pY2UiOiB7CiAgICAgICAgICAiYW5jaG9yX3JvdyI6IDQsCiAgICAgICAgICAiYWNjZXB0ZWRfcm93cyI6IFsKICAgICAgICAgICAgMCwKICAgICAgICAgICAgMSwKICAgICAgICAgICAgMiwKICAgICAgICAgICAgMywKICAgICAgICAgICAgNCwKICAgICAgICAgICAgNSwKICAgICAgICAgICAgNiwKICAgICAgICAgICAgNywKICAgICAgICAgICAgOSwKICAgICAgICAgICAgMTAsCiAgICAgICAgICAgIDExLAogICAgICAgICAgICAxMiwKICAgICAgICAgICAgMTMsCiAgICAgICAgICAgIDE0LAogICAgICAgICAgICAxNQogICAgICAgICAgXSwKICAgICAgICAgICJleGNsdWRlZF9yb3dzIjogWwogICAgICAgICAgICA4LAogICAgICAgICAgICAxNgogICAgICAgICAgXSwKICAgICAgICAgICJzaW1pbGFyaXR5X3RvX2FuY2hvciI6IHsKICAgICAgICAgICAgIjAiOiAwLjY1MzI3Mjc0Nzk5MzQ2OTIsCiAgICAgICAgICAgICIxIjogMC42NjgwODI0MTYwNTc1ODY3LAogICAgICAgICAgICAiMiI6IDAuNjMwNjA5MDM1NDkxOTQzNCwKICAgICAgICAgICAgIjMiOiAwLjY0NjczNDM1Njg4MDE4OCwKICAgICAgICAgICAgIjQiOiAxLjAwMDAwMDIzODQxODU3OSwKICAgICAgICAgICAgIjUiOiAwLjY5NjUyNTgxMjE0OTA0NzksCiAgICAgICAgICAgICI2IjogMC42NjY2NzAzMjI0MTgyMTI5LAogICAgICAgICAgICAiNyI6IDAuNjk0NDA0MzYzNjMyMjAyMSwKICAgICAgICAgICAgIjgiOiAwLjM4NzU2NTA3NjM1MTE2NTc3LAogICAgICAgICAgICAiOSI6IDAuNTM2NDkyNDA3MzIxOTI5OSwKICAgICAgICAgICAgIjEwIjogMC41NTY0OTY4NTg1OTY4MDE4LAogICAgICAgICAgICAiMTEiOiAwLjU2NTQ1MDk2NjM1ODE4NDgsCiAgICAgICAgICAgICIxMiI6IDAuNTEyNjA2NjgwMzkzMjE5LAogICAgICAgICAgICAiMTMiOiAwLjU5NTc1NjExMzUyOTIwNTMsCiAgICAgICAgICAgICIxNCI6IDAuNDc3MTU5MzgwOTEyNzgwNzYsCiAgICAgICAgICAgICIxNSI6IDAuNTY5MTMyMzI4MDMzNDQ3MywKICAgICAgICAgICAgIjE2IjogMC40MjcxOTU3Mjc4MjUxNjQ4CiAgICAgICAgICB9LAogICAgICAgICAgIm1pbmltdW1fc2ltaWxhcml0eSI6IDAuNDUKICAgICAgICB9LAogICAgICAgICJmYWNlIjogewogICAgICAgICAgImFuY2hvcl9yb3ciOiAxOSwKICAgICAgICAgICJhY2NlcHRlZF9yb3dzIjogWwogICAgICAgICAgICAyLAogICAgICAgICAgICAzLAogICAgICAgICAgICA0LAogICAgICAgICAgICA5LAogICAgICAgICAgICAxMCwKICAgICAgICAgICAgMTEsCiAgICAgICAgICAgIDEyLAogICAgICAgICAgICAxMywKICAgICAgICAgICAgMTQsCiAgICAgICAgICAgIDE1LAogICAgICAgICAgICAxNiwKICAgICAgICAgICAgMTcsCiAgICAgICAgICAgIDE4LAogICAgICAgICAgICAxOSwKICAgICAgICAgICAgMjAKICAgICAgICAgIF0sCiAgICAgICAgICAiZXhjbHVkZWRfcm93cyI6IFsKICAgICAgICAgICAgMCwKICAgICAgICAgICAgMSwKICAgICAgICAgICAgNSwKICAgICAgICAgICAgNiwKICAgICAgICAgICAgNywKICAgICAgICAgICAgOAogICAgICAgICAgXSwKICAgICAgICAgICJzaW1pbGFyaXR5X3RvX2FuY2hvciI6IHsKICAgICAgICAgICAgIjAiOiAwLjQxNzU0ODA2MDQxNzE3NTMsCiAgICAgICAgICAgICIxIjogMC4zNTYxNzQ3MDc0MTI3MTk3LAogICAgICAgICAgICAiMiI6IDAuNjY0NjU4ODQ0NDcwOTc3OCwKICAgICAgICAgICAgIjMiOiAwLjYxMTAzNjM2MDI2MzgyNDUsCiAgICAgICAgICAgICI0IjogMC42MDYwNTc4ODIzMDg5NiwKICAgICAgICAgICAgIjUiOiAwLjQxNzEwNDYwMTg2MDA0NjQsCiAgICAgICAgICAgICI2IjogMC40MTk0MjAyNzIxMTE4OTI3LAogICAgICAgICAgICAiNyI6IDAuMzk2NDAyNjU3MDMyMDEyOTQsCiAgICAgICAgICAgICI4IjogMC40MzU0NDgxMTAxMDM2MDcyLAogICAgICAgICAgICAiOSI6IDAuNDkzMzMwMzU5NDU4OTIzMzQsCiAgICAgICAgICAgICIxMCI6IDAuNTE3MTQ2OTQ0OTk5Njk0OCwKICAgICAgICAgICAgIjExIjogMC40NjExMjI2OTE2MzEzMTcxNCwKICAgICAgICAgICAgIjEyIjogMC41MDkzODgxNDg3ODQ2Mzc1LAogICAgICAgICAgICAiMTMiOiAwLjczNjIwOTYzMDk2NjE4NjUsCiAgICAgICAgICAgICIxNCI6IDAuNjc1NTUyOTA0NjA1ODY1NSwKICAgICAgICAgICAgIjE1IjogMC42MzA3NDkzNDQ4MjU3NDQ2LAogICAgICAgICAgICAiMTYiOiAwLjY5NTE5MDE5MTI2ODkyMDksCiAgICAgICAgICAgICIxNyI6IDAuNTk5NTAwMjk4NTAwMDYxLAogICAgICAgICAgICAiMTgiOiAwLjgzMzAwNzkzMTcwOTI4OTYsCiAgICAgICAgICAgICIxOSI6IDEuMCwKICAgICAgICAgICAgIjIwIjogMC45Mjk5MDYzMDg2NTA5NzA1CiAgICAgICAgICB9LAogICAgICAgICAgIm1pbmltdW1fc2ltaWxhcml0eSI6IDAuNDUKICAgICAgICB9CiAgICAgIH0sCiAgICAgICJ2b2ljZV9zb3VyY2VzIjogWwogICAgICAgICJkOTVkYjU1NzkxZWE0YmI2OWQ5YTRjYjYxNzg2NWQ0ZCIsCiAgICAgICAgIjVjYzgwNjQ5YzQwZjRmMWFhYTU4N2U3NTQ0MmU3ZjQzIiwKICAgICAgICAiOTgwYmI5MTZlZjhiNDlmMGFlYTdkZTU4ZmY4NWM4MWQiLAogICAgICAgICIwMjc0NjE2M2IzZDI0N2UzOTBjZGRkMWMxZmFmNjIzZCIsCiAgICAgICAgImQyZDE2ZDVmNzE5MjQ4Y2ZhOGRhZDExODk3Mjc5MWRmIiwKICAgICAgICAiNTZkOTA0MDA4Mjk4NGFkZTk4Mzg3YmU0NjhiMDk1OTEiLAogICAgICAgICJiMWIyMDZiZTk4ZDc0ZmJkOTZhZWNjY2VhZWMwYWMwOCIsCiAgICAgICAgIjQ0ZTZlODA1YmVlYzQyOTQ4MmY1YzVkNWQzODliZTkzIiwKICAgICAgICAiMTFhZWNhNzg0ZDgyNDRmZThkODkzNTRkMTM2MzM1ZDgiLAogICAgICAgICJhZWI1YmU4OTFkOTU0YjY1ODZmYWY0ZWYyNzQxNTViZiIsCiAgICAgICAgImZmOTM0Nzc3MzViNDRmYzFiNzBjMDc2MTYzZjljNTkyIiwKICAgICAgICAiNjhhNGI2ZTgwM2I1NGQxM2FjNGI2YTFiMGJhOWY4OTIiLAogICAgICAgICI5YWZlZGFlZDUzMTE0NWIwOTVkMThmYjE4NmU2ZmU5OCIsCiAgICAgICAgIjcyNDJhNWUxMzFlNTQ0MjNhZDg0YWIzMTQxZmY5ZWQyIiwKICAgICAgICAiZTdiNzJlZTkxNzRkNGZmOWE5OTgxYzJhYWNkZjE0OWQiLAogICAgICAgICIzMjE0ODVjNWI2Y2E0ZWY0ODU1MjFmYTMwMjNlOTgwYyIsCiAgICAgICAgIjNiNzhjMjNmNzFiZDQ0NGY5NzcxNWUzODAwMWNkNDg1IgogICAgICBdLAogICAgICAiZmFjZV9zb3VyY2VzIjogWwogICAgICAgICJkOTgyNmZlNzZmNDM0MmMxODdiNjljZDI0YjM0NDQyYyIsCiAgICAgICAgImU2MjhjYmEzN2VjNjQzMjRhMWRiNjg0MmQzMzFkNDBmIiwKICAgICAgICAiNTY3MmQ4NWM2Njg2NGE2MzhhNDU1NGRlNWIwZWU0OGIiLAogICAgICAgICI4MzUxYTdjYzQ1Y2Y0MzM5OWZiYTk5YWYyMmFlNzliZSIsCiAgICAgICAgIjE5ZGQxZGZmY2U2NDQ0OGZhYjcyOWIwYWFlNWJmNDJhIiwKICAgICAgICAiYTlmNzI2NTc4ZDliNDFkY2JmNTQzMjFjYjU4NmY1NDUiLAogICAgICAgICJmYzA5YmFkZDQwZjA0NGJiYjQ2ZDgxZTVmNmNjZTVmMCIsCiAgICAgICAgIjEzOGMwMzhiM2I5MTQ0MjZiOWJlZWE1MGU3OTRkZDhiIiwKICAgICAgICAiOTI3NzI2ZjNiYWUzNDU3OTk4ZDFmNTE1NzZmNDNkYmYiLAogICAgICAgICI2NWQ1MzMyYTkwNDU0YzZkOWY1MzM4Y2I5OWU1NTljZiIsCiAgICAgICAgIjUwY2FiZDk3NzQ3YjRlMWI5Zjc2YWYxZWE5YTg4YjVhIiwKICAgICAgICAiNzAzNWU0MmUwMmE5NGI1Zjk3YmZjODNjMDRlYzhmOWQiLAogICAgICAgICJhZWI1YmU4OTFkOTU0YjY1ODZmYWY0ZWYyNzQxNTViZiIsCiAgICAgICAgImZmOTM0Nzc3MzViNDRmYzFiNzBjMDc2MTYzZjljNTkyIiwKICAgICAgICAiNjhhNGI2ZTgwM2I1NGQxM2FjNGI2YTFiMGJhOWY4OTIiLAogICAgICAgICI5YWZlZGFlZDUzMTE0NWIwOTVkMThmYjE4NmU2ZmU5OCIsCiAgICAgICAgIjcyNDJhNWUxMzFlNTQ0MjNhZDg0YWIzMTQxZmY5ZWQyIiwKICAgICAgICAiZTdiNzJlZTkxNzRkNGZmOWE5OTgxYzJhYWNkZjE0OWQiLAogICAgICAgICIzMjE0ODVjNWI2Y2E0ZWY0ODU1MjFmYTMwMjNlOTgwYyIsCiAgICAgICAgIjkwMDk3MjlmOTNmODQ2ZjNiMDA2M2FjZmYyZDMwN2Q5IiwKICAgICAgICAiM2I3OGMyM2Y3MWJkNDQ0Zjk3NzE1ZTM4MDAxY2Q0ODUiCiAgICAgIF0sCiAgICAgICJzaG9ydF9saWJyYXJ5X2lkcyI6IFtdLAogICAgICAibm90ZSI6ICJObyBpbmZlcmVuY2UgaXMgYW4gYXBwcm92YWwuIFNob3J0IHNhbXBsZXMgZXhjbHVkZWQgZnJvbSB0aGUgbWFpbiB2b2ljZSBjZW50cm9pZC4gSG9sZCBldmFsdWF0aW9uIHZpZGVvcyBvdXQgb2YgZW5yb2xsbWVudC4iCiAgICB9CiAgfSwKICAicHJvbW90aW9uX3JldmlldyI6IHsKICAgICJtYW5pZmVzdF9zaGEyNTYiOiAiYjQwMzkxYjE4ZWFhYWJkNjgzMWYyZjUyNzBkNmMzMWU4YWZkMGIxMTVjMmQ3OThjNmI0NjkxNTI4MGYxMTAxZiIsCiAgICAiYXBwcm92YWxzX3NoYTI1NiI6ICIxMTQ4NmU4MzBlMjNjNjEwOTJlMDJkYzUxNjM2NWFlNWU1NTJiNDVhMDJiMTc5NWI3MmRkZDdiZDk1MDNkN2RhIiwKICAgICJwcm9tb3RlZF9jYW5kaWRhdGVzIjogWwogICAgICB7CiAgICAgICAgImJhc2VsaW5lX2luZGV4IjogMTcsCiAgICAgICAgInN0YXJ0IjogNDguNjA4LAogICAgICAgICJlbmQiOiA1MC4xNywKICAgICAgICAiZHVyYXRpb24iOiAxLjU2MjAwMDAwMDAwMDAwNDcsCiAgICAgICAgInRleHQiOiAiSSdtIHN0YW5kaW5nIGhlcmUgc2F5aW5nIEdvZCBibGVzcyBob21lbGVzcyB2ZXRlcmFucy4iLAogICAgICAgICJyYXdfc3BlYWtlcl90cmFjayI6ICJTUEVBS0VSXzA0IiwKICAgICAgICAiZmluYWxfY29uZmlkZW5jZSI6IDEuMCwKICAgICAgICAicmVmZXJlbmNlX3NpbWlsYXJpdHkiOiAwLjU4Mzk5NDI2OTM3MTAzMjcsCiAgICAgICAgImxvY2FsX3ZvaWNlX3N0cmVuZ3RoIjogMS4wLAogICAgICAgICJyZXZpZXdfcmVxdWlyZWQiOiB0cnVlLAogICAgICAgICJhdWRpbyI6ICJhdWRpby9jYW5kaWRhdGUtMDAxNy00OC42MDgtNTAuMTcwLndhdiIsCiAgICAgICAgImF1ZGlvX3NoYTI1NiI6ICJlZmJhN2E3MjkzMjhhMWIzN2ZmZmE2Yjc1ZmQyOGJlMzEyZmUxNjlhOThkZGM1NTVmNzE4ZjNmNTM1MThkNzIxIiwKICAgICAgICAic291cmNlIjogewogICAgICAgICAgInZpZGVvIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9dUF0aUV2aVV6R0EiLAogICAgICAgICAgInN0YXJ0IjogNDguNjA4LAogICAgICAgICAgImVuZCI6IDUwLjE3CiAgICAgICAgfQogICAgICB9LAogICAgICB7CiAgICAgICAgImJhc2VsaW5lX2luZGV4IjogMTkyLAogICAgICAgICJzdGFydCI6IDU0NS40MjksCiAgICAgICAgImVuZCI6IDU0Ny40MzEsCiAgICAgICAgImR1cmF0aW9uIjogMi4wMDIwMDAwMDAwMDAwNjY0LAogICAgICAgICJ0ZXh0IjogIllvdXIgcXVhbGlmaWVkIGltbXVuaXR5IGlzIG5vdCBnb2luZyB0byBzdXJ2aXZlIHRoaXMuIiwKICAgICAgICAicmF3X3NwZWFrZXJfdHJhY2siOiAiU1BFQUtFUl8wNCIsCiAgICAgICAgImZpbmFsX2NvbmZpZGVuY2UiOiAxLjAsCiAgICAgICAgInJlZmVyZW5jZV9zaW1pbGFyaXR5IjogMC41MzM2OTgzNzk5OTM0Mzg3LAogICAgICAgICJsb2NhbF92b2ljZV9zdHJlbmd0aCI6IDEuMCwKICAgICAgICAicmV2aWV3X3JlcXVpcmVkIjogdHJ1ZSwKICAgICAgICAiYXVkaW8iOiAiYXVkaW8vY2FuZGlkYXRlLTAxOTItNTQ1LjQyOS01NDcuNDMxLndhdiIsCiAgICAgICAgImF1ZGlvX3NoYTI1NiI6ICIxMWZjMDA0NjM0MWRjNzFiNzkwOGQ1ZTM4NTY3MWU5YjQ5YzAwNzExM2QxNGIxN2JmM2I1ODk2OTQzMjFjNDI0IiwKICAgICAgICAic291cmNlIjogewogICAgICAgICAgInZpZGVvIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9dUF0aUV2aVV6R0EiLAogICAgICAgICAgInN0YXJ0IjogNTQ1LjQyOSwKICAgICAgICAgICJlbmQiOiA1NDcuNDMxCiAgICAgICAgfQogICAgICB9CiAgICBdCiAgfSwKICAidmFsaWRhdGlvbiI6IHsKICAgICJwYXNzZWQiOiB0cnVlLAogICAgImNoZWNrcyI6IHsKICAgICAgImFsbF9jYW5kaWRhdGVzX21hdGNoX3BhcmVudCI6IHRydWUsCiAgICAgICJjZW50cm9pZF9zaGlmdF9pc19ib3VuZGVkIjogdHJ1ZSwKICAgICAgImV4aXN0aW5nX3JlZmVyZW5jZV9hZmZpbml0eV9pc19wcmVzZXJ2ZWQiOiB0cnVlCiAgICB9LAogICAgInRocmVzaG9sZHMiOiB7CiAgICAgICJtaW5pbXVtX3BhcmVudF9zaW1pbGFyaXR5IjogMC41LAogICAgICAibWluaW11bV9jZW50cm9pZF9zaW1pbGFyaXR5IjogMC45OTUsCiAgICAgICJtYXhpbXVtX2V4aXN0aW5nX21lZGlhbl9kcm9wIjogMC4wMQogICAgfSwKICAgICJjYW5kaWRhdGVfc2ltaWxhcml0eV90b19wYXJlbnRfY2VudHJvaWQiOiBbCiAgICAgIDAuNTgxMTYxMDIyMTg2Mjc5MywKICAgICAgMC41MzE4Mjk5NTMxOTM2NjQ2CiAgICBdLAogICAgIm9sZF90b19uZXdfY2VudHJvaWRfc2ltaWxhcml0eSI6IDAuOTk1NzA0OTQ4OTAyMTMwMSwKICAgICJleGlzdGluZ19yZWZlcmVuY2VfbWVkaWFuX2FmZmluaXR5X2Ryb3AiOiAtMC4wMDIxMzc3MjA1ODQ4NjkzODQ4CiAgfSwKICAidm9pY2UiOiB7CiAgICAicGFyZW50X3Jvd3MiOiAxNSwKICAgICJwcm9tb3RlZF9yb3dzIjogMiwKICAgICJ0b3RhbF9yb3dzIjogMTcsCiAgICAiZW5yb2xsbWVudF9zaGEyNTYiOiAiNDY2ZTg2MmRjMTU5MTVhMzFlZjk5MzJjZWZjM2VlNjZlYTY1MGY2Y2E2YmEwOGEzZDkwYmU3OGQxMjFjMTlkNyIKICB9Cn0K',
 'voice_embeddings.npy': 'k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE3LCAxOTIpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAryYYA9E3/UPKH0CD7MhyU9xdnRPW10mT0daXQ5ahlPPYpuob2c04O9luHlPP9rbz0a46y9mgDPPZknEz4Beni9hyhlPfRCj7vrA4y9itULvslpAT3tH489TvS/u+Hd5jzRNBO82H4ivTIZjT2+seC8MHu7PGww4L0p1yK9fpBDPaf23byFqDA8YlwqPkJlo709Ssq9GN2uu4+ynb0Y7jE9+04PvuVCxD1XbCI9+0VxPbu5CryWeby9nGhEveePpL2thSK892Squz16Zb342KC9a41Vupy3e7ysHRg+7RnDvC90or0/kBa9AkjIPZ70Br6Tx2+94YKzvcES3L3IAsY8gNXIPT0GJL3rrMc4YlCWvT7HKz2LBZ48uD6+vCi50z0UKky8ZnkpvQQj/T3sOo29Mc1lvXhh+r1uyXM8Q/NzvcWvPj0m+cO8TrLZPdUlwT0dK8U9OVK1vYpkRr4/pVQ9icuiPWuRXj3Eq0O8Q35CO2dyKj3SDng9z419vZC4JD2dfJy9bMx2vbXUJj5VW/k9BMs3vRv7IL1nJw497TaXvNuos7zBfGC+nI/HvfX0cb0F+RU9IDYrPk3foD3wpra8Rr3fvcpDgT0Sd548KO1TPexPTT3EpT887TLZPD2PQLyW8MW8JR6LvdEztj02oqY8LSyCvUG8Vb7mere8HoOxvAsriTxRA1a8M+2cvJHnnj3JRL+93+1BPYRVmjwAmQK8OZCHvZu2pT3b9La8MUacOo6hkz3e8wY9jPmsvZuOyL1phi49VDESPUjaFj02F0A9uIjevHLjA70tptE8pgjQvOI8lT2PQW49FMBYPQgBWj0gOee91YDXPQOUgL0zNZs9A3G5PeWCaDxe/uk8nnhLvfYIZr0IGD293YSyPYX5Aj5DguI8tuQZPBEfhD28CnU9NtdlvLncEr6aXsu8a5miPe+xrT0p+LQ9AjT/vCKTKz3ZUFs9U4cXPMgDuz0aF+g8WmQMvZJFgb1xtAW9Vu2+PNG21j3czgG93t8WPQmI+TzOs8c8mPqpuy9L0T3Yrz09Hs4EvZbblT3UEmS9OfLoPWwsLr2Wcti91u5OPRT0Nz1f9De9AMwjPTLBHD6k9RG944kIPkGjhz0iYpa82gPaveyVgDvmCgM93AaAvFgeTb3J4sK8jDYovTDMVj27gvQ8YJ66PfuypL2KBFi89PDHvLUfzTyD8qw8daZFPoO8IL7HlzO9DMeXvFnEz72p7Qm9kbbyvY7jyD3hQg09gglqPSGZqTxYoGy9RJc5PBinkr0TjLM8KD4rvZ1LBb5RW6i9uLayPGWTDr2mqvA8nJ+ivYYZKL3ZDUu95KauPaH4p72mSkg98LahvaL4x71mPpA9euNFPaAgnDyHiza9S4/OvNa2O72IT4s8K7H8vE4miz2HMQM95iXsvd0hoD1pzqS9OLl3vbEc3L1dsoc9pFvMvD950T3m1HG9V8P8PTSYlj2+cMU9qbWovUqfCb5iEqo9m3mRPULHsb1y8P+8g7IlPSlOxTxuIrw8sIeZvbf1KL2r4vS8ziSUvTlsdT4JuxQ+PBIRPQ0xs72nG/M9nMwDvGoUE71utwm+XtASvi+rtb1mzYs9QhEDPh+qCT7kpSW9QPE8vbWewjzudd098keHu/swBL2XTpo9uNqlvHorZL2WgQW9YnkNvvAHiD3VYpY8Nh+lvYbdAr5Tc708ibMFvVfi5T04zhI9czzEvTl+DT009Nu9PBSTPe12YL2mKH69b2oAvbBVcT09ylO9A9AwvMhHSb1/RI89HROvvVdiVb3rd5M8V5d0PeauWL2nge27d6+MOzg2OT0lNDU93+Q0PBRL9z0WCtO85gEoPUw5SbwrNK+9GcOmPbMmw711G589NRtlPWSrfrybA7k8SKXJvGWUBr0YJIG9nG+ZPXzDtD36fpk9xrZjPD0Bwz1U+YS8MvUEvkSmgr1WArS9dBiOPel8CT0nMC08zymovPzwsz3DTo89VCi7uiVzJDx04k09vG8qPLkBe70HpUe9Fq06PRZvoD1djpi9EvBaPLLuJD1TJnQ9dfImvDx2Rj0bGZc8XDsmvIXusz186729+MUVPiGDAr202bC9TeQCPt5sUDyhhdW9HqNjPSZXmD3H9Hq9c/DIPZBzFL3/fH68L522vRwRpzxIPUE9RXvTPPQ9Bzyx7+Q8UC5EvbfWGz6qWT894laiPRJOHr5vWgu7kdp/vPZArbzaq583csKCPgVz1b2SSoi9OUGBvQyZpL2Lo9K92xoovlVyWzz5B6e8ffvyuVszID15xju+Sh51vfuuuL1I5Vs9fL95vf+6ir3EyEW9ZhQdPKTktrwi2xQ+vdG0vcjGM71BbgS9w+rlOvsyXb2O8UQ8QxWyvAcltjy6gOg8m2lWPbr8e72A3DY9w7cMvQ8Ibr0XipE85L4avHBQTz2LeYs94LWrvbQbrT01f+W9PTkwPSBjmr1Ljym9fdrovDjsXT3agNe83W9WPc28sz0U4gY+QQ9BvWuNOr2N2AM9UaZUPQb7mr02SC09SCQYPahTaj3by/Y9g2kPvnPl/7wad5M9xzQ3vdd+Bj4f6XQ9rs6NvbePPzyKrOs8cU32vW1/grw1xSe+HScGviBeML3Fbtu8YUupPUMrAT7Q84496mYTvhs2/zyKjfc8DxH8vMxsez26PBc8Jb8ePd8zv7wk86U7ArRPvNiNj7z9xxO9QJTYvZ0D8r1XS1e9Nr9EPKPKAb1iTYg8c005vYL4TD3AMcO8Ogi5PK1XijyDrHy90J+EvQIxPj03Xe69jHbFu9UfCz0+LnI91eabvbh+ub0gG4A9CqCOPbU0mz1FNSm902ndPIjypr1ymwA928lXPa/Q7j2BB+o83EibPZ+eBj7auiO+Ud2CPfnDjL0pzrg9pZvKujGVwj3sPH26BqP3u4pyFr2eKqK9elJROx1YkT0j5z49EklLvbvVGT6haU49Wn2wvfDEfb0BPGU7gsJuPH12Oj3k+Gc9feLYPP6dHT5KZz89+iK1PDLwkD0W0iq9oaD8PDHVuroO8ae9nWfGPJJ9cz23edK8pyLDvHUv7jxIQA49BwEmOlXBkj2gVo09chmmPBhf9z3UVZO968VTPYBJGr6fE9y9CfLzPX4eDj4vkpy9DF3xPVZvgj09aLa9OmnaPfb4L70MggY7Ge4uvvVlET3CM4c9C+A1vL5RVL1zXcM8cy3oPEz3iz06VIk80w41PfyZJr3xgJI9xE7DvJ2jCz347Lk8RPodPt0sjb2m/pS9Q+govH4tub0F9IG9OfEOvixRMTwjBRy97k49PZB00DwFWuG9L9bcvCDrh73ii+E9TjxevdsvS7ySfzG9wv5vO9rGpL2PRrQ9ChKvvQJQprxz1qS8WS3BPUYAsL09Uma7E03Ovd1u3713Ux89vk6NPNKWmbzbC2k8Ly4yvcgGpD1uIje9XgWJu84noTt67H48J7AAvrMRnz3U9eO8jLCjvEvcQzuvckA9ci4lPJXUeD3/jI67P6LjPZTuTj3UYTA+035pve4TIL6Zo/c9DbSbPactl70/EKw7yFm8PQEAhr0gFn08Ts0vPPjrPr1sLJU84BNXvbQwTD5GoEQ+HhcOvAxsbbyNzxY8ykiNvbjXi7xWU6K9YphnvH8AXL3QGTq6JxktPpzkhT0ve2o878R9vbqTTD0Nq948drCrvWPaWTy+iaC8MhWRPelfvzvoPx29yqmCPO8WULzpmnw8+eswvYVORr6nBac9Si98vA38Oj0VqBU8xYQSvisddj04r/29XuCGPZrm7Tyzemm96BQtvdA8oT0IlMc8kfpAvG4Hqj3xSNU9tq6lvRk/a71RG7c9Ue4GuzJZZjyliSO9OO3yvIFFIj1AQ3494/gAPVww7zzxX4s6vBBzPdvBxD0/kzC+v+ffPazcCL6Po1A9u4aHPS/9JDxLw1W9s7wIvC0Pl7yc07G9NYmkPSNcxj200Xs9DbIqPBV9gz2x5K095oDQvepU+73SI1q9yQibPEf/wjwFXQk9mhpLO8l68j3gasA8+PG7PIhQWLxJb9k8wAFKvTGEjL18Ghq96lYYPQXl2Dw+I1E7qeaYOgKMH7xnscm80llsPWWrhz2Doj4962znPI2A2jzxjyY8azQwPW4yQjzhs1q89OnCPDIsID6twyS8m7AHPlhibT0B/4+9COWbPYhlBrojv1S9WTrVvdFmaT1H6wG8DIWOvVLu4TylOyO9n2HFPb2jQTwAzRy9zfDXu0Em4L0qAqc8c+pVvStNBTtFoJE7+S8wPnpfBL5ZCjI8s9R6PcGySL2fCTe94j1DvsnRxj2opoq9FgKSvNs1vD1iHhm+jedcPE+IPL0uiq+9Mif5vUJjrr133UK9raotO+Cupzy21v085hwevrrAjr3k4y+8o/KZPcXa8b1NDiY9FDIJPbKdY71yc8c9S7m/PdKjRb2hijI9fIqMvQrkBL0eAVy9fg6iPZjfwT0BxAI7GaUwvk5XFz7cGDu+OKnAvTKi3bxiEK68gQXJu/8ZjDy7zO+80f7EPcGMtryERJ89Ufe5vdEk073keHk9v4ufPXCMz71j5yC5PvKyvE7LTD2Wqry61bJZve3tdb3Brki9vVfEvQO7nz1BlbE97kGLvHbCr7zkYFO8uSj8vOGLpjxZpBu+Qu88vZCynL2FMgC9hOjoPaoH9T1tO4e64X4Hvq7WFz1Wc288OQqQvGyjyjraIzW6WUeQPRgHiTwFfT29xxOevWcIfz1zJ4M9NlDYvVY0fb67Dg69R0FhPeEzQzz11+U8UqG6vbto6j0ljLa97COavDuuKj0r2Uq9Opx0vUBYJr0GoTC9IIuDO9PXGTq14bY8jiPjvdePrb1lc2q6A/v/PZI9bL2b9XU8gu6gvWs0hLxebmE86YQzPfsCqDzBBJS9mVF/PP8FaT0XUpu94cpvu26z6b1umjg6fClYvR+W/bw+gCK9nHtAvRZ11rzkEwa+XHnaPemZvD36UNQ8GODDOx8rBj1yYlM9PjfCvYaWLr10Qlq9IldpPFycGT0adNQ8eST0vM6IrT1/Vk899hHvvN3kB72oe6q95mdUPUhGZL0FaDm+FUaMPBMIj7sZarK8h9yUPE92IDzyGN89xUOJPVExeTxeiBQ9n4lTPVqd/j3gA5y9uc+QPf1v2Lzsb7a8LkEcPsHoKz75L1O9MlLMPXlxuT0tDiK+ELcPPmGZSzuCfui99I65vaIQmj1ejKY918KIu3z31DyaOei8Lwztu5o+Gj5H9JQ9fYttPU7zGr6dvRa851XTOmK6Bj3qaAq8Aqo8PndYyb2/lIO9OOJmvBCwhb0cDyG9NgkCvmhN4D3pGWG9OuIiPXCoKT20U6G9x9EpvRxQNL2RuaE8SwqAvaqIAT1XjS69PHIaPJm5or2nRb89udurvcCBqbw63p882W3CO1ukqb1cyie9RaYNvDtRsbybHgQ8bAukPLUwWrwHrHM9C/pSO5sPyjufq4Y9pGVKvJVCkz3VTz89oQHWveEF2T1tMci9f91vveR0E75zjMI8/2eUPA7SubukPR68J4yfPZx0E7sS2Bc+tuc/vUd18L2cNRo+PEPgPOmkOjxBsDw9g97ovFEnNj1JcCw9WrG9vaOqATuKR349dhGvvb39Dj4uwCY9y/FWvanQaDzhz8Y9KHbWvYhX3TwZjx++a3ELvkkWAL1U+5C806ESPoBCCz66lAM9vvbLvYN/JL3RCBA9pBDBvRvfUjyr67o8pVOePFkLHr0V2z+9Wj7cujx7xT1yTQI9B6M0vWqTI77l0pu8vuc7Pf+Ukjtz/fI8k7VMvebjzz1UMBe+lRSyPCO5ej1NU7e9XZYuvUbDbT2541y9Q4/FPDMM0rk0o/26V9KovUqI1ryoBA49Af/kPXOeOj2DqMI82ZJgPDTrfb0ty7M9AcrjPF8h2D3CYsc8ekEPPDHYwz0Kf7+9E8SROgWfWr14K6E9IPjQu73bAr1G2Je9GeiZu+cHL73NkAu+dCNVPS8Mzz2CBGk9OVMEvC7obT1iwZW8BSmnvOnljb0Yr4W92etYPN2Fdz0g+QU+fjdBvQRejz35m589XxmkPObrBr3/WLG882nmPePJ8bt74628gZ0RPZOrKD2dZ2O9J74GPRCfubvJsGa9gw0XPKbuaj2PbGK9PrJ0PYXrAT43mxi93oUePflmEL1rara9DhjaPYKf7T0L0iu9OeXsPXVn6T2xAIm9KcyAvJLZFr3Avi68dQILvoLlgr3ZbOQ6yeNZvfaHlj3BBYi9oQRxvZjH2T0Uv809aMpIPel69r2V9Sw7NgO7vCRmrjygY3i87PgUPml9h72O74u9cleBvBnmoLxjEIq8/eQWvoK1g7v1U5S8u4CfvGjxwz0yy9C9oh9JO8gv5L0cZ0g9w3+nveAGjr0CJ8S9LMG5vCgShb0v6LU9ZUPPvTfVNr1vYuI8Td/ivCfSpb3lkHc91f6YvS8mtTwXSAU9QHRUPagKGrw+Jzk9vagDPRa/krwUDrw856psvJmGBj6PXLA67qASvkgayz09I+K8ERstvA7hPb49pyq78rVOvQzKqLuHZZO9Et1gPb/oOz3+89o95uTWvTK8YL44shs+bVnVO6aQnr2tVZO8oXbAvCM81LwtDD099TQOvX59bzkhSeC8ESOkvUZ2Lj4DX849TAQBvYWkn73ZmJg8JtfmPBqEtjvm4ku+e1hIvUO93L1mAy47mOYdPhsJDz6hsnO8uzgGvrcGGL1OejE8X7jsvFTrRj1G90E7ap+WPb3DJD3mcdG9wLBxvHrf4zrJ0oS91d2hvYRw1b0dnzS6BjFfPIuD3rohLBQ9N9cFvRBgDT0TR+i9BXDPOxfbhb1LBxC9lCalvfDQ1rulFXu9fP8ePYBaAL0VveW7Kh+wPFEfq71aqEM9BKnFPcQFmL0BnbO8sxaNPMJNw7uNcaQ8QEHgOzXHrT0v9jk9yd+RPADXtT0GCti9jm33PBs2AL4xh/Q9A1cwPTlSN7y7dOK9G3C5vGMcLzyt7du9r2LLPfppP7td7iW8GgIkvLo6sjsY/7s8J0ADvnwgm70nhvu9KckYPWt0JT3vo0E9UsFavSlx6j1rTPY9eo0dvb85JzoGBYE9tjQ0Pdw8CbwRfEK9V19zPOHarz3zyMs8CQLmPGWHgbwnAn0928YpPWPb0D3mqrS8PY31PLiIjT0a4vq9ka6sPae9Fz2p/Ni9mp27PS2g0TyWpnO96YEZPjhcDz6wqmC9znIHPs52STzZkei99gqdvSoJ/rwFR6s9Q1K1vOs4gTyQmU28H4byvN+hYT2AD9I9trTqPc8SFL5Gtl48YVTrPCK34Dwtxx29RYwZPoVO7739jwm9m9OSPOe0570YlsO8DMM8vvAYHD2hdbM6YfeYPG7L2j35Nsu9zOoEvucFvr0j3bm9uXaQvKoeYb27qqC9lP0ovM27E73Ryo097RSmvSZ9h71JdAq8KUetPBqkgL1o8pI8+SeMvNlgeLyVyHE9qlgXvL8ldr1Twls8nh3uvXdcq70WIPu8BiKBO9aSnj2xy7w9n+sFvg7/IT7Qjpy8JNZ1vTnKq71LMbC8LM72vI3xez17G5y8HqbkPXMDkT0cQ4I9G6ynvafgRr05u4c9IJjhPRhKSb2P4Rc9ugY3PWbETT2To4g9iaStvZMnI734GzO9MVEWvdtfQz4j/hE+mwF+vTvueDySD6o9yzk6vTp0jTz78Ai+UeF7vScMBb4v2b88EsQcPkq37j1ok9u7c7ePvYmqYLupuhM9cVonvQVtozu+y208rOCFOyHjGb14iGC9qBCPvNkuUT2Mev+9xH+UvS5RBr5XrXu9vaPzO/ayozyDe/U8KYGMvSq8CD0OOzG+jpx0vXNGLbxiHZO9PPhfvTPQbj19z369Ik9bvDIQqjxA75w8ReYgvBvOcL2t75Y9iJwBPkgg7DoTPH291BqvvDPzsrtG3Hc9xW5qPf8toztwpP87kuTbPVy00zzMum295LyDPSyK4L21HQg8w1m3uyBFIj316zy9KN1+vQlNiL0sC9u9EYmEPUV5BT5os/08Y5aJvf6X5z220A89nzcCvv6hFrxcuZ29m/S4O2eKOTwg5xC8mJeCu2Xx5j2dsxs9oRl7vapRBD1kI7u8bGakPL5GvLzj5Nm6EPV6vIpD6ztNzXG9zFH5PO5Srbx8bQ4+0QApPaIbwDxgWko95tlcPXMSDj1kr5W8+pAJPiwZHD0qwsy9KxVBPZuEpz1EDAK8POzRvHEmUD2vg769R8ObPKYzLTyotow8FNDIvRFxij2agD299uREPf9/u7wjZ0W9CBBYPRAf7j0EP9c8vmtXPTICLr1slXW8iA27PIB2Az7ryiq8C54GPrpRG74lBE49iZ6fvOhf+zwxWRm9COkUvnfy/D1DQUI9gKuiPBEFTz070W28iNwMvlBX6j3+xYa9kz3QvaS89zyIdpS8wkDuPNmrijvdPsm8kQgIvsJevbs2tya9QAjPPc1Yb7s7wAY8pyyaPeTwwD0AbBM9x62dO4PFeL19kW89v7SYvDfZqbyS6sq9OC/dvLoE5DzrELK7TeCsvHoilz2bn968nqFAvFgUR70ALxA+iPVxPGPjKr3liGi9Te5MPPabrrqNBng9SZhmvV79xbxI4yk+fD7GPT69gr3mDVq8enMGPammHDs2tPY8oxlbvhHDvr2FTsw7xUuyvTuJ2z1kC6I8jQTJvXnKoDxST8U9IRluPdObtD0Imrg8ACwbvsCmR70nuey808nuPb3ICj6aoJu93IIbvRXh3b3eT4k9XMOLPeoZojxvdwa9rcXAvF34srxzWSc9XnWjvdxOAz5tWog6ec0LvmLg1723Fr28NM/fO7JqgT2mOY89K1ywvKkVOz4RoVI8Bq/CPUEyIj3DP8a8pSaBvBxpdDwxSQy9JazIPebWhr2emZo8BH8UvrQEcr3E8xk9PyS2PX0Wlb0rXl6995IUPW7tGz33RjI8+bFqPfgQ0z1ycsC9O6GVPRthfz0Jxao8WgYZPC9fLr7Q4Yk85trivTYv2zspG7M95q2hPZ/uAjzRSKU7biTuPfddhzwUWtE8lnJXPFmBaTzplgE+tkAJvhJ/ZL1OzMW80m1GPLSMpDzOpw8+qkLcvUeNqj3OGpw8HTYbPWUknTzB8iS9fnb2PRMhxb0ZHMy8whTbvK4nRzwpWQk6NYYYuzUnMr0h5SE+Dd8+PUyJWD1KpSc8/7s3vC8fCT6JA7q9SWlRPDCmHj3TBpm9JKtVPfqjwjx3LGk8DqloPQbx4Dx0N0W9JaH/O22yxryLBLS9rpWDvb94AD1trhQ9XSgUvcKb87wKqPU6JRTjPEJSDT74FCk+UWngPKPBoL1PFBw9j0AfvETGkT3bDOc9iVcOPuZwNL4cqt08kqFou6iQRb3xZK+9boPmvb0llT0XWuu82xmJvRvJ8TyCYuK9TRbKvfHhAT11fkC9Xo22vYfRgbymypO9oXxvPf1e0L1zBNC9Y99mve2zCz1d87q9bpuyPXdUED5ZOf07qMOsu8T9az1/fzk8GhJ1PHLWmbv6w1U8a7jHvY0ugL3+m/C9hP5/PA84Vj3LfNE9yfa2vQZDkzy8Z6e9BPYTPXueUb2k6+g96GRhPRH6tr2v5uK8T7WAPcHWIr31nEA8v757vWy6e71gJrE9Z7XsPexP7L0glDU9PxiavLbXyLz1fV689A35vZPh8713ASS9rsUSvnyhvj1Fbxs9CULoukrJzbxJ5j89iqMHPZzatD1rTJG9d08LvkTVkL17ixu9NaClPdWWAj7h9/Y6OS67vcwx4LsnrrO84fUPvdJDVz1xo6I74gNmvWiPtrygkTE9ZP0CvkN71z3mZPM7J3fdvbVCGb5P8mu9m40CvQiADz0PjrK8TcnZuouUMT5aIZO9yiNePMYxir0BuRW6mrDCPK8EozzW8vS9fMECPUWqHr0wvZO9Ovfjvah7Prxf9ZA9JNf8PQeZTL1G/Om8TNduva+tcz1hFw69VMWkPaVdJj4b2x88Mht7PdJ2ID0rlp87e+fNvHiYqr0DzFk9r8JOvbK9tj1NIdo9bRn3PH8j97xf2aK9hxnjPYMH6bqKIQk9N8MhPWZSCT34nYo9myscvubBsb3Zsgq+1NDjuk4+VD3LVaw9bl0pvdwvGT0RVLG9bCPbPF0oeD1Bchq8VHBmPcl7AzzM8I69mGqJvXb0Izx//L28FqNcvf3UFD04RAk+mMECvAaLED3PrJe6ERMsPLGaJz0agQO9UBhtPRepKj1NctO9e/tvPIR2CD2w7Pg7lw0ivbrc0jyp72a9hxGUOwSLvDu/Epa97dfFvaT1qD337xi9YCrhPMtEDz2o+JI8GeeVPVuk1j17jh891F2OvN0Kcb09Vbk9vTajvVkVCT5trT+8Jf4/PtbOG76+cs08WmhYPXksTr0Xg6u9czMUvid1hj0+ZIi9SI+VvP1LpD1uWGq93kbDvewMtj3x9PG9x80Dvs6HJrw9fCG96lvdPf45w7xPlx486M75vTQL3jx8QTK9gWkFPj0WLL1Ul8K8Ii4RuvyX4T2v8G08XjiTPat4+bxiGQE9XyfNvXh/Jb3hy4e9DsCVvOSJvLw5b6u9G2VUvZA7AT44Ht+9MZ+2vHzNML3mye09zMlfvftOAL3cMLS8b53APF/kiTzOzlQ9kZyZvBDa2jwVoug9FieKPSyoLr7xpMk8FqNpvfSsCDyg5v09VHcnvkWB3719xwi9rhD7vD4dLT6gQU68xQzCvW/N0LwOWJM9Hfr3POXJPT3oqMy92pXZvZaNrb0ujKy8BbjZPWenID6W+/U89AzcvJIEfb2xpvg8XQ9BPUlMqzwnJPO8PGHevLBhi7375I49ONgAvkSymT120E09uyuGvQ9iwr1DYUe8ofuRPCvluTuLcV89/G7HPMUNFT6R2TS9cAtYPR+9qzvTHuC88kf/PBRGhDy8HY29udwAPo5qPjuTsvS8qlBkO8sjDb1NA0g9QMMAPti9Db132xS9ExYQPSrnYzzbAF+9udPUOsKx0j1RTSU9uqmPPZGrBzzg4Nm8aODQvLaXRb716kg900aNvei1pLwRGnM9c9ySPWIZWD0uOxI9WnnyPIogoj1v+uU8itIiPT24Zj0H9Dg+eMCjvYbefjwR+gi+IHb0u6dQNjxEyvE94RSwvSBquTyinum8qVOgPPpHND1d7Im9cwSgPRBgN71FK4S9r0kVvlzEFTyDvGc95bQivLDVHjxE1BM+aAgIui1WRD3AOnO979CPPUHjtD0j+gm98B9HvQ10IrxTL228d6vTPQ0Y/jzkU5e9cI6KvTGTMb12sYy9DV4LvE4WZD15QCe+CYjDvUPbuj37Ssw7Irk5vD/YAz1WSNY7OhHZPW0pmj3mzFk9FvFuPetF/rzdgWU9fQ5Vve36oz1ui007WBQaPlun972LQIE9Sz6CPPU7mbwfydS9QYMDvmHEgj28Mta9tzZjO1ZSoDuaF2e6jWr4vavo/D2FUsi9W+SCvfBcOL15s868N2wcvNkYtLzZbEk8dnjYvWi0ortlF286luXSPeoC0TtXvzS8M14FvcBlBT5WGM865IxvPQlhRDzkUW49xSaFvaC2i7yYoYy9qBDXPNdokjwRvfO8uICru1LkpD0Zw0m+RUTEPOTN/LuFkRE+L3MUvX03Er3i5YW9JhiPPdiEyD3KpBo9V/jrOn9QTL2L7DU+uEtlPYN7G74TXQA9bxGrPNHtiDw2rQU+WWAhvvd0wL0Q9l88NLmJvRYZKj3d/nE9qFAGvsoAgDxE48U8kCsHPUMOVz1x/IC9gdgZvnkYgb00wJo80lDUPZIe9jwv+OC7wMycvU3pBr2Bjoc8GdtSPcGOKj1tCMi8Q2divVNbAr0LhW89ogQ5vtEWLD1AxUw8GXPDvSkFW77rUJI8Y19jPWzlDr2ZjBY60HESPRyLkz07w/G9ybUEPh8tYj3Iyp26hGAIvYrTmj2A2ya9MaMBPnItkr0gN6A7oy1WvUKNpbttAPA8O13XPe5kSzzhpTi9pWjbPAFRjT2gZMM8Jjc8PQMi/z23CN08+EeQPVii6rw77TQ994jbvAeYBb5WjMA9E4OXvfCvITz1OgM9UPupPYr3pbwnfHe9Cwn5vFZEPD3yGGk9EWaGPNgs2jwS33o9JE4ZvvRJwbyXrg+9AOiEPdzNhDvbv+o9oRbAvbTyRLzmOJK9UvwtPdETBz1O7xu8gPstPXV11Dz9HgG+pJ3gvTFsHD2/1TW88anyu6yjtbuyfv49bnxBPXiyNTwk3qU8W23CPNKYNz1vyzq91WsxPeNl9ry/WX28H7SHukBgMz00FWS97DjdPaYE9TzFYvO8iH3wuuCB2rsSV6u9NPBjvSkOoT2IGts9i8ikPIxMoDzs9Sk5WbrjuR+ETz3Qgjk9OIRBPTslf73NMny8YhWPvT68hj1T6Qs9vbXyPWDiWr7sdtU6nHRIPO+7mr1qPlq9NprCvUo5kz3PZ5W9NW3COz0KuDw4MKK8w9oFvmN7RT1HJaq9+MNsvQCJmDvH/R+94lutPcFPijzqYkS9V4LKvYo8x7xvjk+9WxW/PSG5Jj0IuKk8qCa+PeK04j0U1BM9CLWbPez+Cb1gfK08MfbbvDW3TL0qF6S9kQzxPBRnkz0kqhO94MKnvdV/oz3h2hC+5DURvI9bZ725NAg+Ql2KvSv0n73PmqW9OhQVPUPfIz3FYJA9wSVQvC43KL1/ZP097FzIPWTA3b0Syss8QHVUugb3yzywf7s9wXcjvv0SDr676RG8uwnzu9t8Az4/oJo92aewvVhsvjxLU6C8a+RbO28DHz0w6Vy9IKUWvs+Ds72KQPG8UgDbPeBeCD4nbAG9PIzbvZHH0LzzDZ28w2C1PFQy1LtXhVm9meHjvKtzpb0GGac9MXH0vVS+WD0QH149v6CNvc9tMr5+DL+8rjN8O7G2GT2OGiE8T4lKPSOUHT5Cd1W9S5nFPQNq5jy9SqS9VlgEvSXUAz5mq9u97/gZPTLhnjrTB8e7ZGTfvdL23rzWoIo9ceImPgaXDLwSo2e9qCbDPOYMrT31sQe9spJ9Ou1NoD3bOUE8m9rcPUl4uTz1aX86G2vJvSUZU74V5yE9F+K9vVGwZbw/NDQ9wKsmPVaKeDxwkD88H8uoPGtspz1qBhM9uD8lPbUmoT2KcuQ9zX0Zvj1nib3AeRG9+mdIPSOsY73jPPw9qZucvTz0jD1A9g08GAUZPKOFiT1IZkC8II+FPYv7fLxR9pm9DSksvv3/UD07/gI7aCqzvNhVEbxczms+qdEDvFUetj1lYDs944LAuzlL0D3uTOm9SyMVPSG7lD1o/ms9w3PBvBtNKbzGV7M80aNAvFEoKT2x4gK9ieQtvagsZT0txgK+whE5vfv4hT0EQ/a8f4GAvHvsCz2S/+u8EyqUOy46Az5xxKw9DjYCvR3b6LyPT0I81l56vVcXLT5HZNU99smyPetKHb5NMdc8dNp5PGGFa73UIa48S9qFvRxNoD2Z6Q68t8mNvSdqK7wDNu+8tzzHvV9Xk7zDp7C9kxaIvOtp6rsP5rq9aTtxPO7kar1Y5Cu97PbovSOPDT2uqQC9qWDuPcutvDzJ4YA8GRzLPKi/FD6kIO08fXyNPfOcyrpaHd07n0UkvrGdcLypY7m8nGFSPcy8YzyeEHK9uX7YvQceEbx/6AS+Y+NGPdXyQ73nPvs9/OhjPDGz/bwuD6m96yLKuhypBz2+dp098ni6vL+Onb3MQbc9FJwIPlv68r3Kg1Y9S7qmvP0QmD1G5vW8P6qYvUjtZ739nz+6AbY/vQD9FD7gIYI9Z9YmvXyaarysC3Q9/MO4PXE6wzwFlwC9TxkRvjXlpb0+Gyc9bUcRPb0MNj4KdD68jKmnvWBXIL2Z4Pu8qlWiPfdfPD14pFc9oQlzvVg6ob2LQWU9uNKpvREHmD07rYA9bJCcOQ6VHL5roJ+9TaQQPQT53DyldFi8LLZEO5FiQT60+Lu9Q2fQPA66srzMnqe9hXcEPUfXMrxWK9S9PNG+PL3OezwhrQo9/BQMveGPzrwXoOE88QtfPY/ljL3QYWe99kyvPIEAyj382oS8yBqRvZW33j3JiW686NW0PZXRSz1mDUa9n6cFvuWjDb7yqKg9M2e2vJqG9zvpfW87RUrvO8hi8DxuY1W9KTScvMeIybrmxrE9kjXlPDePBz7wYao9WzcevoJtZr3Sita9nUxjvEQxwjx1vvo9bKxqvXx+Kr3cPJi9hewFPO/LBLwtvYq9HVf4PPa2oD2Djam99LUvvq18pLx6DQe9UKMIvQUgoLxu9b49AkcrPT/ICL2l3Q08ICRXPSqK6T3yahu+2gCuPR2CojyKV7e855qIPe64Yj2fo4O9BLBUPDooXD2jfVq9OALGPPl7jjym4Iu9lMuMvQEvSj31Y309kDLjvIX0Dr0x/z66cwqhOzNCgD0Jiv88c58fPdhom712Gpi9muY+veIejT3c97A8FP0PPnJT7r2r0gI9nFxKvHeO2L1QuLa9BYSIvUMWuz3mXIm95EM+PLbRIjzcRcm8hQEKvnHEOzwnYmi9phZwvYFAhDzBcmQ9G5ybPfff/bsowNY8b6nhvZL6KryRkZu8csDQPFbTgj29N449lfyGPUwqzz26R8s9ELyLPTzNrL35I4Q9l2jxvA/a/r1DuBq9K2Yjvcb32DxIkiK6H0TZvdytpD1VGR++EMsIvLrDA72ug/g9KHgqvTCqXrwhSxS9prWBPf98iz2ODWA9OstQvb07Cj0jbA0+6IOCPTf/Hb5JppQ7s2LYPKvm5jxwX8Y9z9hBvrDy0jsg26q6KbfpvUhTiD3FvBc+wv+BPfzUhT02QLI8v/YxPTZRkjwbw9e9uNYjvevIrr3KbTm9uj7gPAjGaT1L1448gIaDvQ1GPj2bil49KJegvPaf5DyCqji89estvNh8Uz3/OCS9VLAIvgWStjxSUCe9R3JnvddvWr6WYf697WCaPaxibL1gKKo984c9PcBtoz25VCi9ZMq4Pb8XkzxVZqE8+3usvWkmJjzIPtW9R4j1PdqOyrzeiPw8pe9uO6fxnbw8/vo8S+3aPfCAer3mH8q9+qCuPYslAT0tIs47QunevDBFKj4AYM89KFdXPWgauj3alDa9pJeGvAWgAL7pprs9NvNGvTICjrxK9cA82K+EvaaWory5zqi9feqcvMoBHT1e9yY9bNhgvM6uJT19iiU+6RMkvnI1g7wbJMu9KKcsvdNuW7yeJbQ9avEnvbx51jwdjbI8g+1wPYMADj2mLGK6y1kbPuwH8Tw8tx2+CFndvfATGr2XhZu9zeeBvP40PrvsfIE9g43TO+zC2bzV7QQ7YjoRvdIkqz1JsY28hhgTvG9P9Tz+nz69A5KKPQ9ubj1r5ow8s/c0PRUPq71tD6G8akXOPFnOBr4q9sC9iagjvlDLjzyaDMQ9xqoTvFp07z004iA9UN3wPcBX2j2iHXk8mb//O7T1pL123je8TnvxOpZZOj0aZAQ9cQIjPqleQ72GdMI8jsNLPTuT+LyKOja8mLjKvVDGmD1T2Um9aTOhPcGvJT2QtHq92KWpvYxmDbsaIMG8KJZnvUDEyjyDh0m9AX0TPRo3xL2xAdG78lmNvFBzNjwSRqa8Wx0pPqx6tz0sAoG8e3z8PVVqBz4z3oM9oOm6vbngA73tquY8YAeyvbWtnr1ApZK9OiHkPATyGbxfKWQ9QVUcus67qL0ptTS9rpS0vRjTnb0ZEJc9SeRvPVBp+r0utwC82qSRu49lSj1HRZU9h2b/vAOMxb30+eo80f70u9cxh701RQq+uyKYvcJfBr2GKIQ916f1vXl7n7049By8USIXvgJKXT6rWSu94augvQFjPLvQPCG97j6qPb4vWTvzLQ29yszfvZrrBL53w4u9Dc4jPsHTUDxeb6Q9BSvAvWirDz5Ipda9WSeavUc4qjuFmrG9UD/hvSxvET3vQy49O+OYvdCqlT06n6Y9iBeevWjjQb2Wgki9XCrmvFLqqb0ueIi9ez+GvMStDj7RjT+9YT5YPIq+Db2xncq8+iX3vVh+MzzCZAi+jEKsPcVGsbzZvCq86+B/vN4lGb3qd6W7jHUmPtQCfz3SlEi94GYKvD1rID3dWaE8djiLPXk/tz0ADvg8Gj0zPdMK2j2hxKo8XlsgvW3wS73lzwG8Jkt4vQz7tj2kIgy98ferPI61SrzfSDg9EHPqPSzLfz1tU1c9MiCKPO7ljD0XHg09+Vf5vZsV+L3enWq9wUHUPOOrwr0OTgY+jkBnPTSHtD1FkGK99X0FvMrHqD2NMHS8oFGcvSd8lb2/KMu8SOIgvSNenD3HArS8Jw2zvSIQRr1dINs9VTEJPiYWmLxzJxa+q1ImPTscLzyVy807ZnJzO+Dx27yujYu9rCGfPQ2X9DyVGia9yRDLvTVC7T3+LYu9WWX3vOCyrz0Aa7q9tc7KO8EsID5vZko9vP2pPPEgRTuRgNo8VQa+PZjKMT7rezS9G0NUPbhFJ74A46s9C3lrvdBkBLxXOLi85zfnuv/ZCL46q+e7qSYsvLb3lz3GvFi7sFZIvha+P7zQHYM7sALvvdOJhD3/rf29b3mBPQP6kT0AEIG9eDeyvbn3tzq1B2W8N2ByPZsn1L2zicq8FsP+vX4Na73ie+29zEHyPV5vtL0tYN+9C3UfPSMUTTxqTJm9afJMtzuHCr3IDHQ9DUa1vF5bmLsyM5e8ir0NvVQwXT3y9Dq9vSkOvQ1upj2quPm92aoAvNLE3bwCGJg9KfH7PE3sFL51HX29JP4GPSgRYT0i9cC89yBZvX1Frr3t3gM+XMuTPFbPaL1oZK+9Dh2aOxq5hj2NVuI9IXGzvahsQrzzkkY9TtRGPKUAcj1gnq29KBS4vUtWb70KEzU9zn1UPFJugL0nyt68EVALvdQUML1hWeo8FIj2PU1NJz56rA29duY9PAjJxTy8Eic9r160PRGILb2I0pS8gxWfPM8W4ryW6M29hY8HvnpxxL3eHLc965++PAKnCLw0bwO9jYSEO12JXT1I4Zi8tIqTPacxwT1n9ay8kXpSPe0yVr3moM28IxQAvrmrMr0eJg69B9+8PXgzob1KFyo8OddovcZCQj3i7q09Bya6PR2OAD3MXgO+NVaDPA/SGry5edM9jjVwvB/SaryUsiO90w4BPh3CGj39CaU9LNtIvB1mX76M6N883tjnvTqZ7zxURHe8ekwxO3AijjwweZm9INa9PZ6KlL291wA8Ia2bPOVLnT38j1w92bPVvQ74oL3PDim9lsrPPP7n9j2GRJw9IoCovcaHxT2TaD+8utTfO3EF3rzpqJe90VpOPW1E6Tx9idW7RLqTvYnwsr0nJPI6bRYpPfBDzbw='}
reference_hashes = {}
for name, value in REFERENCE_FILES.items():
    content = base64.b64decode(value)
    (REFERENCE/name).write_bytes(content)
    reference_hashes[name] = hashlib.sha256(content).hexdigest()
(RESULTS/'reference-hashes.json').write_text(json.dumps(reference_hashes, indent=2))
print('Reference bundle ready:', reference_hashes)


## Credentials, checkpoint restore, and attached video

Create a private Kaggle secret named `HF_TOKEN`. The token is read from the environment and is never embedded or printed. The current video is copied from the attached dataset because YouTube blocks Kaggle's shared addresses.


In [ ]:
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    ENV['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
else:
    import getpass
    ENV['HF_TOKEN'] = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN') or getpass.getpass('Hugging Face token: ')
if not ENV['HF_TOKEN']:
    raise RuntimeError('A Hugging Face token with diarization-model access is required.')
if ON_KAGGLE:
    for archive in Path('/kaggle/input').rglob('stage-checkpoints.zip'):
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                target = (BASE/member.filename).resolve()
                if not target.is_relative_to(CACHE.resolve()):
                    raise RuntimeError('Unexpected checkpoint archive path')
            zipped.extractall(BASE)
if not VIDEO.exists():
    if ON_KAGGLE:
        matches = list(Path('/kaggle/input').rglob('uAtiEviUzGA.mp4'))
        if len(matches) != 1:
            raise RuntimeError(
                'Attach a Kaggle dataset containing exactly one file named '
                'uAtiEviUzGA.mp4. YouTube blocks downloads from Kaggle.')
        source_video = matches[0]
    else:
        local_video = Path.cwd()/'uAtiEviUzGA.mp4'
        if not local_video.is_file():
            raise RuntimeError(f'Missing local video: {{local_video}}')
        source_video = local_video
    # The supplied YouTube video uses AV1, which Kaggle's OpenCV build cannot
    # decode. Normalize it once so the full visual pass actually reads frames.
    checked(['ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error', '-y',
             '-i', str(source_video), '-map', '0:v:0', '-map', '0:a:0',
             '-c:v', 'libx264', '-preset', 'fast', '-crf', '22',
             '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '160k', str(VIDEO)])
video_codec = subprocess.check_output(
    ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
     '-show_entries', 'stream=codec_name', '-of', 'default=nw=1:nk=1', str(VIDEO)],
    env=ENV, text=True).strip()
if video_codec != 'h264':
    raise RuntimeError(f'Expected normalized H.264 video, found {{video_codec!r}}')
print('Normalized video codec:', video_codec)
print('Credentials configured; token not displayed.')
print('Video ready:', VIDEO, VIDEO.stat().st_size, 'bytes')


In [ ]:
def export_checkpoints():
    if CACHE.exists():
        shutil.make_archive(str(BASE/'stage-checkpoints'), 'zip', CACHE.parent, CACHE.name)

def stream(command, log_name, failure):
    with (RESULTS/log_name).open('w') as log:
        process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            line = line.replace(ENV['HF_TOKEN'], '[REDACTED]')
            log.write(line); log.flush()
            print(line if len(line) < 1000 else line[:1000]+' ... [full line saved]\n', end='')
        if process.wait() != 0:
            raise RuntimeError(failure)

def run_test(video, stem, batch_size=4):
    output = RESULTS/(stem+'_evidence.json')
    command = [PYTHON, str(WORK/'chainofrules.py'), str(video),
        '--voice-priors', str(REFERENCE/'voice_embeddings.npy'),
        '--face-priors', str(REFERENCE/'face_embeddings.npy'),
        '--output', str(output), '--cache-dir', str(CACHE), '--batch-size', str(batch_size)]
    started = time.monotonic()
    try:
        stream(command, stem+'.log', 'Pipeline failed; see the saved log. Retry with BATCH_SIZE=1 for CUDA memory errors.')
    finally:
        export_checkpoints()
    result = json.loads(output.read_text())
    transcript = '\n'.join(f"[{s['start']:.2f}-{s['end']:.2f}] {s['final_speaker']} (strength={s['final_confidence']:.2f}): {s['text']}" for s in result['segments'])
    (RESULTS/(stem+'_transcript.txt')).write_text(transcript+'\n')
    repeat_rows = []
    for segment in result['segments']:
        for evidence in segment.get('evidence', []):
            if evidence.get('source') == 'repeated_presentation':
                d = evidence.get('details', {})
                repeat_rows.append(f"[{segment['start']:.2f}] {segment['text']}  <=>  [{d.get('donor_start', 0):.2f}] {d.get('donor_text', '')}")
    (RESULTS/(stem+'_repeat_candidates.txt')).write_text('\n'.join(repeat_rows)+'\n')
    print(f'Elapsed: {(time.monotonic()-started)/60:.1f} minutes')
    print('Repeated-presentation groups:', len(result.get('repeated_presentations', [])))
    return result

def extract_clip(start, duration, name):
    clip = WORK/name
    checked(['ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error', '-y',
             '-ss', str(start), '-i', str(VIDEO), '-t', str(duration),
             '-c:v', 'libx264', '-preset', 'fast', '-crf', '20', '-c:a', 'aac', str(clip)])
    return clip

def run_targeted_review():
    output_dir = RESULTS/'targeted-review'
    command = [PYTHON, str(WORK/'review_transcript_regions.py'), '--video', str(VIDEO),
        '--baseline', str(RESULTS/'full_video_evidence.json'),
        '--target-reference', str(REFERENCE/'voice_embeddings.npy'),
        '--other-reference', 'Opening_officer='+str(REFERENCE/'opening_officer_reference.npy'),
        '--output-dir', str(output_dir), '--cache-dir', str(CACHE/'targeted-review'),
        '--weak-confidence', str(REVIEW_WEAK_CONFIDENCE), '--short-seconds', str(REVIEW_SHORT_SECONDS),
        '--minimum-gap', '5', '--context-seconds', '3', '--maximum-window-seconds', '30',
        '--window-overlap-seconds', '4', '--device', 'cuda' if ON_KAGGLE else 'auto']
    before = (RESULTS/'full_video_evidence.json').read_bytes()
    try:
        stream(command, 'targeted-review.log', 'Targeted review failed; see the saved log.')
    finally:
        export_checkpoints()
    assert (RESULTS/'full_video_evidence.json').read_bytes() == before
    return json.loads((output_dir/'comparison_summary.json').read_text())

def run_overlap_extraction():
    output_dir = RESULTS/'overlap-extraction'
    command = [PYTHON, str(WORK/'review_overlap_extraction.py'), '--video', str(VIDEO),
        '--baseline', str(RESULTS/'full_video_evidence.json'),
        '--enrollment', str(REFERENCE/'auditor_enrollment.wav'),
        '--voice-priors', str(REFERENCE/'voice_embeddings.npy'),
        '--output-dir', str(output_dir), '--device', 'cuda' if ON_KAGGLE else 'cpu',
        '--whisper-model', 'large-v2']
    before = (RESULTS/'full_video_evidence.json').read_bytes()
    try:
        stream(command, 'overlap-extraction.log', 'Overlap extraction failed; baseline results remain valid.')
    finally:
        export_checkpoints()
    assert (RESULTS/'full_video_evidence.json').read_bytes() == before
    return json.loads((output_dir/'report.json').read_text())

def export_reference_promotion_review():
    output_dir = RESULTS/'reference-promotion-review'
    command = [PYTHON, str(WORK/'reference_promotion.py'), 'export',
        '--video', str(VIDEO), '--evidence', str(RESULTS/'full_video_evidence.json'),
        '--output-dir', str(output_dir), '--source-url', VIDEO_URL,
        '--reference-metadata', str(REFERENCE/'reference.json')]
    checked(command, cwd=WORK)
    return json.loads((output_dir/'manifest.json').read_text())


## Run the opening check and whole video

The opening check confirms that face analysis is actually using CUDA. Then the full baseline, targeted review, repeat evidence, and overlap extraction run.


In [ ]:
opening_clip = extract_clip(0, 30, 'opening_30s.mp4')
opening = run_test(opening_clip, 'opening', BATCH_SIZE)
providers = opening.get('runtime', {}).get('face_providers', {})
if ON_KAGGLE and (not providers or any('CUDAExecutionProvider' not in p for p in providers.values())):
    raise RuntimeError('Face models are still on CPU. Review opening.log before starting the full video.')
print((RESULTS/'opening_transcript.txt').read_text())

if RUN_FULL_VIDEO:
    full_video = run_test(VIDEO, 'full_video', BATCH_SIZE)
    decoded_visual_segments = sum(
        1 for segment in full_video['segments']
        for evidence in segment.get('evidence', [])
        if evidence.get('source') == 'visual_context'
        and evidence.get('details', {}).get('frames_read', 0) > 0)
    if decoded_visual_segments == 0:
        raise RuntimeError('No full-video frames were decoded; supplemental reviews were not started.')
    print('Full-video segments with decoded visual frames:', decoded_visual_segments)
    confident_dir = RESULTS/'confidence-filtered-transcript'
    checked([PYTHON, str(WORK/'export_confident_transcript.py'),
             str(RESULTS/'full_video_evidence.json'),
             '--output-dir', str(confident_dir),
             '--target-minimum', '0.35', '--other-minimum', '0.65'], cwd=WORK)
    print('Confidence-filtered transcript:', confident_dir/'confident_transcript.txt')
    if RUN_TARGETED_REVIEW:
        targeted_review = run_targeted_review()
        print('Targeted review:', json.dumps(targeted_review, indent=2))
    if RUN_OVERLAP_EXTRACTION:
        overlap_review = run_overlap_extraction()
        print('Overlap extraction:', json.dumps(overlap_review['summary'], indent=2))
    promotion_review = export_reference_promotion_review()
    print('Reference promotion candidates:', len(promotion_review['candidates']))
else:
    print('Full video disabled. Review the opening result, then set RUN_FULL_VIDEO=True.')


## Download results

`diarization-results.zip` contains the baseline transcript and evidence, repeat candidates, targeted review, overlap extraction audio and report, logs, and package versions. `stage-checkpoints.zip` can restart expensive baseline stages.


In [ ]:
from IPython.display import FileLink, display
export_checkpoints()
with (RESULTS/'runtime-packages.txt').open('w') as packages:
    checked([PYTHON, '-m', 'pip', 'freeze'], stdout=packages)
shutil.make_archive(str(BASE/'diarization-results'), 'zip', RESULTS)
display(FileLink(str(BASE/'diarization-results.zip')))
display(FileLink(str(BASE/'stage-checkpoints.zip')))
print('Saved in:', BASE)
